In [0]:
def _cli_runner_get_venv_path_from_databricks_yml(cwd):
  import yaml

  databricks_yml_path = f"{cwd}/databricks.yml"
  with open(databricks_yml_path, "r") as f:
    data = yaml.safe_load(f)
    return data.get("python", {}).get("venv_path", ".venv")


def _cli_runner_get_custom_profiles(workspace_url, workspace_id, custom_url_host):
  import re

  if not re.fullmatch(r"[0-9]+", workspace_id):
    return None
  profiles = "[__settings__]\ndefault_profile = workspace_classic_id\n"
  profiles += f"\n[workspace_classic]\nhost = {workspace_url}\n"
  profiles += f"\n[workspace_classic_id]\nhost = {workspace_url}\nworkspace_id = {workspace_id}\n"
  custom_url_host = custom_url_host.lower()
  if re.fullmatch(r"[a-z0-9-]+(\.[a-z0-9-]+)+", custom_url_host):
    profiles += f"\n[workspace_custom_host]\nhost = https://{custom_url_host}\nworkspace_id = {workspace_id}\n"
  return profiles


def _cli_runner_run_cli_command(cli_command, cwd, tmp_file_path, include_stderr, env, *, cli_upstream, capture_stdout=False):
  import os
  import subprocess

  stderr_arg = subprocess.PIPE if include_stderr else None
  if tmp_file_path or capture_stdout:
    if tmp_file_path:
      os.makedirs(os.path.dirname(tmp_file_path), exist_ok=True)
    result = subprocess.run(cli_command, cwd=cwd, stdout=subprocess.PIPE, stderr=stderr_arg, text=True, env=env)
    if tmp_file_path:
      with open(tmp_file_path, 'w') as output_file:
        output_file.write(result.stdout)
    print(result.stdout)
  else:
    result = subprocess.run(cli_command, cwd=cwd, stderr=stderr_arg, text=True, env=env)
  if result.returncode != 0:
    error_msg = f"Command {result.args[:3]} failed with exit code {result.returncode}"
    if include_stderr and result.stderr:
      error_msg += f":\n{result.stderr}"
    raise RuntimeError(error_msg)

  if cli_upstream == "genie-code":
    print(result.stderr)

    try:
      with open("/Workspace/.proc/self/status", "r") as status_file:
        status_file.read()
    except Exception as e:
      print(f"Warning: Failed to read /Workspace/.proc/self/status: {e}")


def _cli_runner_run_cli_command_with_token(cli_command, cwd, tmp_file_path, include_stderr, token, env, *, cli_upstream, capture_stdout=False):
  _cli_runner_run_cli_command(
    cli_command,
    cwd,
    tmp_file_path,
    include_stderr,
    {**env, "DATABRICKS_TOKEN": token},
    cli_upstream=cli_upstream,
    capture_stdout=capture_stdout,
  )


def _cli_runner_get_download_url(workspace_url, token, file_api_path, runner_download_error_marker):
  import requests
  import time

  api_create_download_url = f"{workspace_url}/api/2.0/fs/create-download-url"
  headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
  data = {"path": file_api_path}

  max_attempts = 3
  last_error = None
  for attempt in range(max_attempts):
    try:
      response = requests.post(api_create_download_url, headers=headers, json=data)
      if response.status_code == 200:
        return response.json()["url"]
      last_error = f"HTTP {response.status_code} {response.text}"
    except requests.RequestException as e:
      last_error = str(e)
    if attempt < max_attempts - 1:
      time.sleep(2 ** attempt)

  print(f"{runner_download_error_marker}: {file_api_path} {last_error}")
  raise RuntimeError(f"Failed to download {file_api_path}: {last_error}")


def _databricks_cli_runner():
  import hashlib
  import os
  import platform
  import requests
  import shlex
  import shutil
  import subprocess
  import tempfile
  import time
  import zipfile
  from dbruntime.databricks_repl_context import get_context
  from urllib.parse import urlparse

  cli_version = "1.24.0"
  cwd = "/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1"
  args = "bundle summary -o json --include-locations --target dev"
  tmp_file = "/Workspace/Users/deepaksinghal@cyntexa.com/.bundle/claims_lakehouse/.output/async-summary-output-2etmb4j0nti.json"
  capture_stdout = "true"
  dev_cli_snippet = ""
  artifact_download_error_prefix = "DABsRunnerError: Failed to download CLI dependency"
  runner_download_error_marker = "DabsRunnerErrorFallback: Downloading error, using fallback URL"
  allow_python_execution = "false"
  enable_yaml_sync = "true"
  enable_deployment_history = "true"
  include_stderr_in_error = "false"
  cli_upstream = "dabs-in-workspace"
  use_custom_profiles = "false"

  if os.environ.get("IS_SERVERLESS", "").strip().lower() != "true":
    raise RuntimeError("Databricks CLI runner must run on serverless compute")

  setup_runner_environment_start = time.monotonic()
  ctx = get_context()
  token = ctx.apiToken
  workspace_url = f"https://{urlparse(ctx.apiUrl).netloc}"
  cpu_arch = "linux_arm64" if platform.machine() == "aarch64" else "linux_amd64"

  env = os.environ.copy()

  def emit_event(phase, start_time):
    print(f"\n__RUNNER_PHASE__:{phase}:{(time.monotonic() - start_time) * 1000:.0f}", flush=True)

  def get_download_url(file_api_path):
    return _cli_runner_get_download_url(
      workspace_url, token, file_api_path, runner_download_error_marker
    )

  def download_and_extract(file_url, dest_dir):
    response = requests.get(file_url)
    if response.status_code != 200:
      raise RuntimeError(f"HTTP {response.status_code} while downloading artifact: {response.text}")
    tmp_file = f"{dest_dir}.zip"

    with open(tmp_file, "wb") as zip_file:
        zip_file.write(response.content)
    with zipfile.ZipFile(tmp_file, 'r') as zip_ref:
        zip_ref.extractall(dest_dir)

  def download_artifact_if_not_exists(input_file_name, binary_name, dest_path, tmp_dir):
    if os.path.exists(dest_path):
      return

    try:
      os.makedirs(os.path.dirname(dest_path), exist_ok=True)

      path_prefix = f"/DatabricksInternal/Jobs/ArtifactRepository/public/databricks-cli-binaries/{cli_version}"
      download_url = get_download_url(f"{path_prefix}/{input_file_name}.zip")

      tmp_archive_dir = f"{tmp_dir}/{input_file_name}"
      download_and_extract(download_url, tmp_archive_dir)

      src_path = f"{tmp_archive_dir}/{binary_name}"
      if not os.path.exists(src_path):
        available = ", ".join(os.listdir(tmp_archive_dir)) if os.path.isdir(tmp_archive_dir) else "(missing archive directory)"
        raise RuntimeError(f"Expected binary '{binary_name}' not found in archive contents: {available}")

      shutil.move(src_path, dest_path)
      os.chmod(dest_path, 0o755)
    except Exception as e:
      error_msg = f"{artifact_download_error_prefix} {input_file_name}/{binary_name} CLI v{cli_version}: {str(e)}"
      raise RuntimeError(f"{error_msg}")

  tmp_base = os.path.expanduser("~/tmp")
  if not os.path.exists(tmp_base):
    os.makedirs(tmp_base)
  os.chmod(tmp_base, 0o700)

  with tempfile.TemporaryDirectory(dir=tmp_base) as tmpdirname:
      emit_event("setup_runner_environment", setup_runner_environment_start)

      download_artifacts_start = time.monotonic()
      cli_exec_path = f"{tmp_base}/databricks_{cli_version}_{cpu_arch}"
      download_artifact_if_not_exists(
        f"databricks_cli_{cpu_arch}",
        "databricks",
        cli_exec_path,
        tmpdirname
      )
      emit_event("download_artifacts", download_artifacts_start)

      setup_cli_environment_start = time.monotonic()
      cwd = os.path.normpath(f"/Workspace{cwd}")

      if dev_cli_snippet:
        cli_path = cli_exec_path
        exec(dev_cli_snippet, globals(), locals())

      env["DATABRICKS_BUNDLE_TMP"] = f"{tmp_base}/bundle-2-{hashlib.sha256(cwd.encode()).hexdigest()}"
      env["DATABRICKS_CLI_UPSTREAM"] = cli_upstream

      if enable_yaml_sync == "true":
        env["DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC"] = "1"
      else:
        env.pop("DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC", None)

      if enable_deployment_history == "true":
        env["DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY"] = "true"
      else:
        env.pop("DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY", None)

      env["DATABRICKS_BUNDLE_ENGINE"] = "direct"

      emit_event("setup_cli_environment", setup_cli_environment_start)

      if allow_python_execution == "true":
        install_dependencies_start = time.monotonic()
        venv_path = _cli_runner_get_venv_path_from_databricks_yml(cwd)
        venv_folder_path = os.path.normpath(f"{cwd}/{venv_path}")
        venv_bin_path = f"{venv_folder_path}/bin"

        os.makedirs(venv_bin_path, exist_ok=True)

        python3_wrapper_path = f"{venv_bin_path}/python3"
        if not os.path.exists(python3_wrapper_path):
          with open(python3_wrapper_path, "w") as f:
            f.write('#!/usr/bin/env bash\nexec python3 "$@"\n')
          os.chmod(python3_wrapper_path, 0o755)

        pyvenv_cfg_path = f"{venv_folder_path}/pyvenv.cfg"
        if not os.path.exists(pyvenv_cfg_path):
          with open(pyvenv_cfg_path, "w") as f:
            pass

        pyproject_path = f"{cwd}/pyproject.toml"
        uv_pip_install_args = ["uv", "pip", "install", "--quiet", "--python", python3_wrapper_path, "--link-mode=copy"]

        override_file = None
        if os.path.exists(pyproject_path):
          uv_pip_install_args.extend(["-r", pyproject_path, "--group", "dev"])
        else:
          uv_pip_install_args.append(f"databricks-bundles=={cli_version}")

        try:
          subprocess.run(uv_pip_install_args, check=True, cwd=cwd, stdout=subprocess.DEVNULL, env=env)
        finally:
          if override_file and os.path.exists(override_file):
            os.remove(override_file)
        emit_event("install_dependencies", install_dependencies_start)

        env.pop("DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION", None)
      else:
        env["DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION"] = "1"

      env.pop("DATABRICKS_CLUSTER_ID", None)

      with tempfile.NamedTemporaryFile(mode='w+', dir=tmpdirname, delete=True) as databrickscfg_file:
          cli_args = shlex.split(args)
          custom_profiles = None
          if use_custom_profiles == "true":
            custom_profiles = _cli_runner_get_custom_profiles(
              workspace_url,
              str(getattr(ctx, "workspaceId", None) or ""),
              str(getattr(ctx, "workspaceHostName", None) or ""),
            )
          if custom_profiles:
            databrickscfg_file.write(custom_profiles)
            env["DATABRICKS_BUNDLE_STRICT_PROFILE_MATCH"] = "1"
            cli_command = [cli_exec_path] + cli_args
          else:
            profile_name = "workspace_profile"
            databrickscfg_file.write(f"[workspace_profile]\nhost = {workspace_url}")
            cli_command = [cli_exec_path] + cli_args + ["-p", profile_name]
          databrickscfg_file.flush()
          env["DATABRICKS_CONFIG_FILE"] = databrickscfg_file.name

          tmp_file_path = tmp_file

          run_cli_start = time.monotonic()
          run_cli_phase = "run_cli_genie" if cli_upstream == "genie-code" else "run_cli"
          try:
            _cli_runner_run_cli_command_with_token(
              cli_command,
              cwd,
              tmp_file_path,
              include_stderr_in_error == "true",
              token,
              env,
              cli_upstream=cli_upstream,
              capture_stdout=capture_stdout == "true",
            )
          finally:
            emit_event(run_cli_phase, run_cli_start)


try:
  _databricks_cli_runner()
finally:
  del (
    _databricks_cli_runner,
    _cli_runner_get_venv_path_from_databricks_yml,
    _cli_runner_get_custom_profiles,
    _cli_runner_get_download_url,
    _cli_runner_run_cli_command,
    _cli_runner_run_cli_command_with_token,
  )
None


In [0]:
def _cli_runner_get_venv_path_from_databricks_yml(cwd):
  import yaml

  databricks_yml_path = f"{cwd}/databricks.yml"
  with open(databricks_yml_path, "r") as f:
    data = yaml.safe_load(f)
    return data.get("python", {}).get("venv_path", ".venv")


def _cli_runner_get_custom_profiles(workspace_url, workspace_id, custom_url_host):
  import re

  if not re.fullmatch(r"[0-9]+", workspace_id):
    return None
  profiles = "[__settings__]\ndefault_profile = workspace_classic_id\n"
  profiles += f"\n[workspace_classic]\nhost = {workspace_url}\n"
  profiles += f"\n[workspace_classic_id]\nhost = {workspace_url}\nworkspace_id = {workspace_id}\n"
  custom_url_host = custom_url_host.lower()
  if re.fullmatch(r"[a-z0-9-]+(\.[a-z0-9-]+)+", custom_url_host):
    profiles += f"\n[workspace_custom_host]\nhost = https://{custom_url_host}\nworkspace_id = {workspace_id}\n"
  return profiles


def _cli_runner_run_cli_command(cli_command, cwd, tmp_file_path, include_stderr, env, *, cli_upstream, capture_stdout=False):
  import os
  import subprocess

  stderr_arg = subprocess.PIPE if include_stderr else None
  if tmp_file_path or capture_stdout:
    if tmp_file_path:
      os.makedirs(os.path.dirname(tmp_file_path), exist_ok=True)
    result = subprocess.run(cli_command, cwd=cwd, stdout=subprocess.PIPE, stderr=stderr_arg, text=True, env=env)
    if tmp_file_path:
      with open(tmp_file_path, 'w') as output_file:
        output_file.write(result.stdout)
    print(result.stdout)
  else:
    result = subprocess.run(cli_command, cwd=cwd, stderr=stderr_arg, text=True, env=env)
  if result.returncode != 0:
    error_msg = f"Command {result.args[:3]} failed with exit code {result.returncode}"
    if include_stderr and result.stderr:
      error_msg += f":\n{result.stderr}"
    raise RuntimeError(error_msg)

  if cli_upstream == "genie-code":
    print(result.stderr)

    try:
      with open("/Workspace/.proc/self/status", "r") as status_file:
        status_file.read()
    except Exception as e:
      print(f"Warning: Failed to read /Workspace/.proc/self/status: {e}")


def _cli_runner_run_cli_command_with_token(cli_command, cwd, tmp_file_path, include_stderr, token, env, *, cli_upstream, capture_stdout=False):
  _cli_runner_run_cli_command(
    cli_command,
    cwd,
    tmp_file_path,
    include_stderr,
    {**env, "DATABRICKS_TOKEN": token},
    cli_upstream=cli_upstream,
    capture_stdout=capture_stdout,
  )


def _cli_runner_get_download_url(workspace_url, token, file_api_path, runner_download_error_marker):
  import requests
  import time

  api_create_download_url = f"{workspace_url}/api/2.0/fs/create-download-url"
  headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
  data = {"path": file_api_path}

  max_attempts = 3
  last_error = None
  for attempt in range(max_attempts):
    try:
      response = requests.post(api_create_download_url, headers=headers, json=data)
      if response.status_code == 200:
        return response.json()["url"]
      last_error = f"HTTP {response.status_code} {response.text}"
    except requests.RequestException as e:
      last_error = str(e)
    if attempt < max_attempts - 1:
      time.sleep(2 ** attempt)

  print(f"{runner_download_error_marker}: {file_api_path} {last_error}")
  raise RuntimeError(f"Failed to download {file_api_path}: {last_error}")


def _databricks_cli_runner():
  import hashlib
  import os
  import platform
  import requests
  import shlex
  import shutil
  import subprocess
  import tempfile
  import time
  import zipfile
  from dbruntime.databricks_repl_context import get_context
  from urllib.parse import urlparse

  cli_version = "1.24.0"
  cwd = "/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1"
  args = "bundle summary -o json --include-locations --target dev"
  tmp_file = "/Workspace/Users/deepaksinghal@cyntexa.com/.bundle/claims_lakehouse/.output/async-summary-output-u6p17ymudyk.json"
  capture_stdout = "true"
  dev_cli_snippet = ""
  artifact_download_error_prefix = "DABsRunnerError: Failed to download CLI dependency"
  runner_download_error_marker = "DabsRunnerErrorFallback: Downloading error, using fallback URL"
  allow_python_execution = "false"
  enable_yaml_sync = "true"
  enable_deployment_history = "true"
  include_stderr_in_error = "false"
  cli_upstream = "dabs-in-workspace"
  use_custom_profiles = "false"

  if os.environ.get("IS_SERVERLESS", "").strip().lower() != "true":
    raise RuntimeError("Databricks CLI runner must run on serverless compute")

  setup_runner_environment_start = time.monotonic()
  ctx = get_context()
  token = ctx.apiToken
  workspace_url = f"https://{urlparse(ctx.apiUrl).netloc}"
  cpu_arch = "linux_arm64" if platform.machine() == "aarch64" else "linux_amd64"

  env = os.environ.copy()

  def emit_event(phase, start_time):
    print(f"\n__RUNNER_PHASE__:{phase}:{(time.monotonic() - start_time) * 1000:.0f}", flush=True)

  def get_download_url(file_api_path):
    return _cli_runner_get_download_url(
      workspace_url, token, file_api_path, runner_download_error_marker
    )

  def download_and_extract(file_url, dest_dir):
    response = requests.get(file_url)
    if response.status_code != 200:
      raise RuntimeError(f"HTTP {response.status_code} while downloading artifact: {response.text}")
    tmp_file = f"{dest_dir}.zip"

    with open(tmp_file, "wb") as zip_file:
        zip_file.write(response.content)
    with zipfile.ZipFile(tmp_file, 'r') as zip_ref:
        zip_ref.extractall(dest_dir)

  def download_artifact_if_not_exists(input_file_name, binary_name, dest_path, tmp_dir):
    if os.path.exists(dest_path):
      return

    try:
      os.makedirs(os.path.dirname(dest_path), exist_ok=True)

      path_prefix = f"/DatabricksInternal/Jobs/ArtifactRepository/public/databricks-cli-binaries/{cli_version}"
      download_url = get_download_url(f"{path_prefix}/{input_file_name}.zip")

      tmp_archive_dir = f"{tmp_dir}/{input_file_name}"
      download_and_extract(download_url, tmp_archive_dir)

      src_path = f"{tmp_archive_dir}/{binary_name}"
      if not os.path.exists(src_path):
        available = ", ".join(os.listdir(tmp_archive_dir)) if os.path.isdir(tmp_archive_dir) else "(missing archive directory)"
        raise RuntimeError(f"Expected binary '{binary_name}' not found in archive contents: {available}")

      shutil.move(src_path, dest_path)
      os.chmod(dest_path, 0o755)
    except Exception as e:
      error_msg = f"{artifact_download_error_prefix} {input_file_name}/{binary_name} CLI v{cli_version}: {str(e)}"
      raise RuntimeError(f"{error_msg}")

  tmp_base = os.path.expanduser("~/tmp")
  if not os.path.exists(tmp_base):
    os.makedirs(tmp_base)
  os.chmod(tmp_base, 0o700)

  with tempfile.TemporaryDirectory(dir=tmp_base) as tmpdirname:
      emit_event("setup_runner_environment", setup_runner_environment_start)

      download_artifacts_start = time.monotonic()
      cli_exec_path = f"{tmp_base}/databricks_{cli_version}_{cpu_arch}"
      download_artifact_if_not_exists(
        f"databricks_cli_{cpu_arch}",
        "databricks",
        cli_exec_path,
        tmpdirname
      )
      emit_event("download_artifacts", download_artifacts_start)

      setup_cli_environment_start = time.monotonic()
      cwd = os.path.normpath(f"/Workspace{cwd}")

      if dev_cli_snippet:
        cli_path = cli_exec_path
        exec(dev_cli_snippet, globals(), locals())

      env["DATABRICKS_BUNDLE_TMP"] = f"{tmp_base}/bundle-2-{hashlib.sha256(cwd.encode()).hexdigest()}"
      env["DATABRICKS_CLI_UPSTREAM"] = cli_upstream

      if enable_yaml_sync == "true":
        env["DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC"] = "1"
      else:
        env.pop("DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC", None)

      if enable_deployment_history == "true":
        env["DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY"] = "true"
      else:
        env.pop("DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY", None)

      env["DATABRICKS_BUNDLE_ENGINE"] = "direct"

      emit_event("setup_cli_environment", setup_cli_environment_start)

      if allow_python_execution == "true":
        install_dependencies_start = time.monotonic()
        venv_path = _cli_runner_get_venv_path_from_databricks_yml(cwd)
        venv_folder_path = os.path.normpath(f"{cwd}/{venv_path}")
        venv_bin_path = f"{venv_folder_path}/bin"

        os.makedirs(venv_bin_path, exist_ok=True)

        python3_wrapper_path = f"{venv_bin_path}/python3"
        if not os.path.exists(python3_wrapper_path):
          with open(python3_wrapper_path, "w") as f:
            f.write('#!/usr/bin/env bash\nexec python3 "$@"\n')
          os.chmod(python3_wrapper_path, 0o755)

        pyvenv_cfg_path = f"{venv_folder_path}/pyvenv.cfg"
        if not os.path.exists(pyvenv_cfg_path):
          with open(pyvenv_cfg_path, "w") as f:
            pass

        pyproject_path = f"{cwd}/pyproject.toml"
        uv_pip_install_args = ["uv", "pip", "install", "--quiet", "--python", python3_wrapper_path, "--link-mode=copy"]

        override_file = None
        if os.path.exists(pyproject_path):
          uv_pip_install_args.extend(["-r", pyproject_path, "--group", "dev"])
        else:
          uv_pip_install_args.append(f"databricks-bundles=={cli_version}")

        try:
          subprocess.run(uv_pip_install_args, check=True, cwd=cwd, stdout=subprocess.DEVNULL, env=env)
        finally:
          if override_file and os.path.exists(override_file):
            os.remove(override_file)
        emit_event("install_dependencies", install_dependencies_start)

        env.pop("DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION", None)
      else:
        env["DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION"] = "1"

      env.pop("DATABRICKS_CLUSTER_ID", None)

      with tempfile.NamedTemporaryFile(mode='w+', dir=tmpdirname, delete=True) as databrickscfg_file:
          cli_args = shlex.split(args)
          custom_profiles = None
          if use_custom_profiles == "true":
            custom_profiles = _cli_runner_get_custom_profiles(
              workspace_url,
              str(getattr(ctx, "workspaceId", None) or ""),
              str(getattr(ctx, "workspaceHostName", None) or ""),
            )
          if custom_profiles:
            databrickscfg_file.write(custom_profiles)
            env["DATABRICKS_BUNDLE_STRICT_PROFILE_MATCH"] = "1"
            cli_command = [cli_exec_path] + cli_args
          else:
            profile_name = "workspace_profile"
            databrickscfg_file.write(f"[workspace_profile]\nhost = {workspace_url}")
            cli_command = [cli_exec_path] + cli_args + ["-p", profile_name]
          databrickscfg_file.flush()
          env["DATABRICKS_CONFIG_FILE"] = databrickscfg_file.name

          tmp_file_path = tmp_file

          run_cli_start = time.monotonic()
          run_cli_phase = "run_cli_genie" if cli_upstream == "genie-code" else "run_cli"
          try:
            _cli_runner_run_cli_command_with_token(
              cli_command,
              cwd,
              tmp_file_path,
              include_stderr_in_error == "true",
              token,
              env,
              cli_upstream=cli_upstream,
              capture_stdout=capture_stdout == "true",
            )
          finally:
            emit_event(run_cli_phase, run_cli_start)


try:
  _databricks_cli_runner()
finally:
  del (
    _databricks_cli_runner,
    _cli_runner_get_venv_path_from_databricks_yml,
    _cli_runner_get_custom_profiles,
    _cli_runner_get_download_url,
    _cli_runner_run_cli_command,
    _cli_runner_run_cli_command_with_token,
  )
None


In [0]:
def _cli_runner_get_venv_path_from_databricks_yml(cwd):
  import yaml

  databricks_yml_path = f"{cwd}/databricks.yml"
  with open(databricks_yml_path, "r") as f:
    data = yaml.safe_load(f)
    return data.get("python", {}).get("venv_path", ".venv")


def _cli_runner_get_custom_profiles(workspace_url, workspace_id, custom_url_host):
  import re

  if not re.fullmatch(r"[0-9]+", workspace_id):
    return None
  profiles = "[__settings__]\ndefault_profile = workspace_classic_id\n"
  profiles += f"\n[workspace_classic]\nhost = {workspace_url}\n"
  profiles += f"\n[workspace_classic_id]\nhost = {workspace_url}\nworkspace_id = {workspace_id}\n"
  custom_url_host = custom_url_host.lower()
  if re.fullmatch(r"[a-z0-9-]+(\.[a-z0-9-]+)+", custom_url_host):
    profiles += f"\n[workspace_custom_host]\nhost = https://{custom_url_host}\nworkspace_id = {workspace_id}\n"
  return profiles


def _cli_runner_run_cli_command(cli_command, cwd, tmp_file_path, include_stderr, env, *, cli_upstream, capture_stdout=False):
  import os
  import subprocess

  stderr_arg = subprocess.PIPE if include_stderr else None
  if tmp_file_path or capture_stdout:
    if tmp_file_path:
      os.makedirs(os.path.dirname(tmp_file_path), exist_ok=True)
    result = subprocess.run(cli_command, cwd=cwd, stdout=subprocess.PIPE, stderr=stderr_arg, text=True, env=env)
    if tmp_file_path:
      with open(tmp_file_path, 'w') as output_file:
        output_file.write(result.stdout)
    print(result.stdout)
  else:
    result = subprocess.run(cli_command, cwd=cwd, stderr=stderr_arg, text=True, env=env)
  if result.returncode != 0:
    error_msg = f"Command {result.args[:3]} failed with exit code {result.returncode}"
    if include_stderr and result.stderr:
      error_msg += f":\n{result.stderr}"
    raise RuntimeError(error_msg)

  if cli_upstream == "genie-code":
    print(result.stderr)

    try:
      with open("/Workspace/.proc/self/status", "r") as status_file:
        status_file.read()
    except Exception as e:
      print(f"Warning: Failed to read /Workspace/.proc/self/status: {e}")


def _cli_runner_run_cli_command_with_token(cli_command, cwd, tmp_file_path, include_stderr, token, env, *, cli_upstream, capture_stdout=False):
  _cli_runner_run_cli_command(
    cli_command,
    cwd,
    tmp_file_path,
    include_stderr,
    {**env, "DATABRICKS_TOKEN": token},
    cli_upstream=cli_upstream,
    capture_stdout=capture_stdout,
  )


def _cli_runner_get_download_url(workspace_url, token, file_api_path, runner_download_error_marker):
  import requests
  import time

  api_create_download_url = f"{workspace_url}/api/2.0/fs/create-download-url"
  headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
  data = {"path": file_api_path}

  max_attempts = 3
  last_error = None
  for attempt in range(max_attempts):
    try:
      response = requests.post(api_create_download_url, headers=headers, json=data)
      if response.status_code == 200:
        return response.json()["url"]
      last_error = f"HTTP {response.status_code} {response.text}"
    except requests.RequestException as e:
      last_error = str(e)
    if attempt < max_attempts - 1:
      time.sleep(2 ** attempt)

  print(f"{runner_download_error_marker}: {file_api_path} {last_error}")
  raise RuntimeError(f"Failed to download {file_api_path}: {last_error}")


def _databricks_cli_runner():
  import hashlib
  import os
  import platform
  import requests
  import shlex
  import shutil
  import subprocess
  import tempfile
  import time
  import zipfile
  from dbruntime.databricks_repl_context import get_context
  from urllib.parse import urlparse

  cli_version = "1.24.0"
  cwd = "/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1"
  args = "bundle summary -o json --include-locations --target dev"
  tmp_file = "/Workspace/Users/deepaksinghal@cyntexa.com/.bundle/claims_lakehouse/.output/async-summary-output-m0bwte2i84.json"
  capture_stdout = "true"
  dev_cli_snippet = ""
  artifact_download_error_prefix = "DABsRunnerError: Failed to download CLI dependency"
  runner_download_error_marker = "DabsRunnerErrorFallback: Downloading error, using fallback URL"
  allow_python_execution = "false"
  enable_yaml_sync = "true"
  enable_deployment_history = "true"
  include_stderr_in_error = "false"
  cli_upstream = "dabs-in-workspace"
  use_custom_profiles = "false"

  if os.environ.get("IS_SERVERLESS", "").strip().lower() != "true":
    raise RuntimeError("Databricks CLI runner must run on serverless compute")

  setup_runner_environment_start = time.monotonic()
  ctx = get_context()
  token = ctx.apiToken
  workspace_url = f"https://{urlparse(ctx.apiUrl).netloc}"
  cpu_arch = "linux_arm64" if platform.machine() == "aarch64" else "linux_amd64"

  env = os.environ.copy()

  def emit_event(phase, start_time):
    print(f"\n__RUNNER_PHASE__:{phase}:{(time.monotonic() - start_time) * 1000:.0f}", flush=True)

  def get_download_url(file_api_path):
    return _cli_runner_get_download_url(
      workspace_url, token, file_api_path, runner_download_error_marker
    )

  def download_and_extract(file_url, dest_dir):
    response = requests.get(file_url)
    if response.status_code != 200:
      raise RuntimeError(f"HTTP {response.status_code} while downloading artifact: {response.text}")
    tmp_file = f"{dest_dir}.zip"

    with open(tmp_file, "wb") as zip_file:
        zip_file.write(response.content)
    with zipfile.ZipFile(tmp_file, 'r') as zip_ref:
        zip_ref.extractall(dest_dir)

  def download_artifact_if_not_exists(input_file_name, binary_name, dest_path, tmp_dir):
    if os.path.exists(dest_path):
      return

    try:
      os.makedirs(os.path.dirname(dest_path), exist_ok=True)

      path_prefix = f"/DatabricksInternal/Jobs/ArtifactRepository/public/databricks-cli-binaries/{cli_version}"
      download_url = get_download_url(f"{path_prefix}/{input_file_name}.zip")

      tmp_archive_dir = f"{tmp_dir}/{input_file_name}"
      download_and_extract(download_url, tmp_archive_dir)

      src_path = f"{tmp_archive_dir}/{binary_name}"
      if not os.path.exists(src_path):
        available = ", ".join(os.listdir(tmp_archive_dir)) if os.path.isdir(tmp_archive_dir) else "(missing archive directory)"
        raise RuntimeError(f"Expected binary '{binary_name}' not found in archive contents: {available}")

      shutil.move(src_path, dest_path)
      os.chmod(dest_path, 0o755)
    except Exception as e:
      error_msg = f"{artifact_download_error_prefix} {input_file_name}/{binary_name} CLI v{cli_version}: {str(e)}"
      raise RuntimeError(f"{error_msg}")

  tmp_base = os.path.expanduser("~/tmp")
  if not os.path.exists(tmp_base):
    os.makedirs(tmp_base)
  os.chmod(tmp_base, 0o700)

  with tempfile.TemporaryDirectory(dir=tmp_base) as tmpdirname:
      emit_event("setup_runner_environment", setup_runner_environment_start)

      download_artifacts_start = time.monotonic()
      cli_exec_path = f"{tmp_base}/databricks_{cli_version}_{cpu_arch}"
      download_artifact_if_not_exists(
        f"databricks_cli_{cpu_arch}",
        "databricks",
        cli_exec_path,
        tmpdirname
      )
      emit_event("download_artifacts", download_artifacts_start)

      setup_cli_environment_start = time.monotonic()
      cwd = os.path.normpath(f"/Workspace{cwd}")

      if dev_cli_snippet:
        cli_path = cli_exec_path
        exec(dev_cli_snippet, globals(), locals())

      env["DATABRICKS_BUNDLE_TMP"] = f"{tmp_base}/bundle-2-{hashlib.sha256(cwd.encode()).hexdigest()}"
      env["DATABRICKS_CLI_UPSTREAM"] = cli_upstream

      if enable_yaml_sync == "true":
        env["DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC"] = "1"
      else:
        env.pop("DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC", None)

      if enable_deployment_history == "true":
        env["DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY"] = "true"
      else:
        env.pop("DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY", None)

      env["DATABRICKS_BUNDLE_ENGINE"] = "direct"

      emit_event("setup_cli_environment", setup_cli_environment_start)

      if allow_python_execution == "true":
        install_dependencies_start = time.monotonic()
        venv_path = _cli_runner_get_venv_path_from_databricks_yml(cwd)
        venv_folder_path = os.path.normpath(f"{cwd}/{venv_path}")
        venv_bin_path = f"{venv_folder_path}/bin"

        os.makedirs(venv_bin_path, exist_ok=True)

        python3_wrapper_path = f"{venv_bin_path}/python3"
        if not os.path.exists(python3_wrapper_path):
          with open(python3_wrapper_path, "w") as f:
            f.write('#!/usr/bin/env bash\nexec python3 "$@"\n')
          os.chmod(python3_wrapper_path, 0o755)

        pyvenv_cfg_path = f"{venv_folder_path}/pyvenv.cfg"
        if not os.path.exists(pyvenv_cfg_path):
          with open(pyvenv_cfg_path, "w") as f:
            pass

        pyproject_path = f"{cwd}/pyproject.toml"
        uv_pip_install_args = ["uv", "pip", "install", "--quiet", "--python", python3_wrapper_path, "--link-mode=copy"]

        override_file = None
        if os.path.exists(pyproject_path):
          uv_pip_install_args.extend(["-r", pyproject_path, "--group", "dev"])
        else:
          uv_pip_install_args.append(f"databricks-bundles=={cli_version}")

        try:
          subprocess.run(uv_pip_install_args, check=True, cwd=cwd, stdout=subprocess.DEVNULL, env=env)
        finally:
          if override_file and os.path.exists(override_file):
            os.remove(override_file)
        emit_event("install_dependencies", install_dependencies_start)

        env.pop("DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION", None)
      else:
        env["DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION"] = "1"

      env.pop("DATABRICKS_CLUSTER_ID", None)

      with tempfile.NamedTemporaryFile(mode='w+', dir=tmpdirname, delete=True) as databrickscfg_file:
          cli_args = shlex.split(args)
          custom_profiles = None
          if use_custom_profiles == "true":
            custom_profiles = _cli_runner_get_custom_profiles(
              workspace_url,
              str(getattr(ctx, "workspaceId", None) or ""),
              str(getattr(ctx, "workspaceHostName", None) or ""),
            )
          if custom_profiles:
            databrickscfg_file.write(custom_profiles)
            env["DATABRICKS_BUNDLE_STRICT_PROFILE_MATCH"] = "1"
            cli_command = [cli_exec_path] + cli_args
          else:
            profile_name = "workspace_profile"
            databrickscfg_file.write(f"[workspace_profile]\nhost = {workspace_url}")
            cli_command = [cli_exec_path] + cli_args + ["-p", profile_name]
          databrickscfg_file.flush()
          env["DATABRICKS_CONFIG_FILE"] = databrickscfg_file.name

          tmp_file_path = tmp_file

          run_cli_start = time.monotonic()
          run_cli_phase = "run_cli_genie" if cli_upstream == "genie-code" else "run_cli"
          try:
            _cli_runner_run_cli_command_with_token(
              cli_command,
              cwd,
              tmp_file_path,
              include_stderr_in_error == "true",
              token,
              env,
              cli_upstream=cli_upstream,
              capture_stdout=capture_stdout == "true",
            )
          finally:
            emit_event(run_cli_phase, run_cli_start)


try:
  _databricks_cli_runner()
finally:
  del (
    _databricks_cli_runner,
    _cli_runner_get_venv_path_from_databricks_yml,
    _cli_runner_get_custom_profiles,
    _cli_runner_get_download_url,
    _cli_runner_run_cli_command,
    _cli_runner_run_cli_command_with_token,
  )
None


In [0]:
def _cli_runner_get_venv_path_from_databricks_yml(cwd):
  import yaml

  databricks_yml_path = f"{cwd}/databricks.yml"
  with open(databricks_yml_path, "r") as f:
    data = yaml.safe_load(f)
    return data.get("python", {}).get("venv_path", ".venv")


def _cli_runner_get_custom_profiles(workspace_url, workspace_id, custom_url_host):
  import re

  if not re.fullmatch(r"[0-9]+", workspace_id):
    return None
  profiles = "[__settings__]\ndefault_profile = workspace_classic_id\n"
  profiles += f"\n[workspace_classic]\nhost = {workspace_url}\n"
  profiles += f"\n[workspace_classic_id]\nhost = {workspace_url}\nworkspace_id = {workspace_id}\n"
  custom_url_host = custom_url_host.lower()
  if re.fullmatch(r"[a-z0-9-]+(\.[a-z0-9-]+)+", custom_url_host):
    profiles += f"\n[workspace_custom_host]\nhost = https://{custom_url_host}\nworkspace_id = {workspace_id}\n"
  return profiles


def _cli_runner_run_cli_command(cli_command, cwd, tmp_file_path, include_stderr, env, *, cli_upstream, capture_stdout=False):
  import os
  import subprocess

  stderr_arg = subprocess.PIPE if include_stderr else None
  if tmp_file_path or capture_stdout:
    if tmp_file_path:
      os.makedirs(os.path.dirname(tmp_file_path), exist_ok=True)
    result = subprocess.run(cli_command, cwd=cwd, stdout=subprocess.PIPE, stderr=stderr_arg, text=True, env=env)
    if tmp_file_path:
      with open(tmp_file_path, 'w') as output_file:
        output_file.write(result.stdout)
    print(result.stdout)
  else:
    result = subprocess.run(cli_command, cwd=cwd, stderr=stderr_arg, text=True, env=env)
  if result.returncode != 0:
    error_msg = f"Command {result.args[:3]} failed with exit code {result.returncode}"
    if include_stderr and result.stderr:
      error_msg += f":\n{result.stderr}"
    raise RuntimeError(error_msg)

  if cli_upstream == "genie-code":
    print(result.stderr)

    try:
      with open("/Workspace/.proc/self/status", "r") as status_file:
        status_file.read()
    except Exception as e:
      print(f"Warning: Failed to read /Workspace/.proc/self/status: {e}")


def _cli_runner_run_cli_command_with_token(cli_command, cwd, tmp_file_path, include_stderr, token, env, *, cli_upstream, capture_stdout=False):
  _cli_runner_run_cli_command(
    cli_command,
    cwd,
    tmp_file_path,
    include_stderr,
    {**env, "DATABRICKS_TOKEN": token},
    cli_upstream=cli_upstream,
    capture_stdout=capture_stdout,
  )


def _cli_runner_get_download_url(workspace_url, token, file_api_path, runner_download_error_marker):
  import requests
  import time

  api_create_download_url = f"{workspace_url}/api/2.0/fs/create-download-url"
  headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
  data = {"path": file_api_path}

  max_attempts = 3
  last_error = None
  for attempt in range(max_attempts):
    try:
      response = requests.post(api_create_download_url, headers=headers, json=data)
      if response.status_code == 200:
        return response.json()["url"]
      last_error = f"HTTP {response.status_code} {response.text}"
    except requests.RequestException as e:
      last_error = str(e)
    if attempt < max_attempts - 1:
      time.sleep(2 ** attempt)

  print(f"{runner_download_error_marker}: {file_api_path} {last_error}")
  raise RuntimeError(f"Failed to download {file_api_path}: {last_error}")


def _databricks_cli_runner():
  import hashlib
  import os
  import platform
  import requests
  import shlex
  import shutil
  import subprocess
  import tempfile
  import time
  import zipfile
  from dbruntime.databricks_repl_context import get_context
  from urllib.parse import urlparse

  cli_version = "1.24.0"
  cwd = "/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1"
  args = "bundle summary -o json --include-locations --target dev"
  tmp_file = "/Workspace/Users/deepaksinghal@cyntexa.com/.bundle/claims_lakehouse/.output/async-summary-output-fuc34dm883u.json"
  capture_stdout = "true"
  dev_cli_snippet = ""
  artifact_download_error_prefix = "DABsRunnerError: Failed to download CLI dependency"
  runner_download_error_marker = "DabsRunnerErrorFallback: Downloading error, using fallback URL"
  allow_python_execution = "false"
  enable_yaml_sync = "true"
  enable_deployment_history = "true"
  include_stderr_in_error = "false"
  cli_upstream = "dabs-in-workspace"
  use_custom_profiles = "false"

  if os.environ.get("IS_SERVERLESS", "").strip().lower() != "true":
    raise RuntimeError("Databricks CLI runner must run on serverless compute")

  setup_runner_environment_start = time.monotonic()
  ctx = get_context()
  token = ctx.apiToken
  workspace_url = f"https://{urlparse(ctx.apiUrl).netloc}"
  cpu_arch = "linux_arm64" if platform.machine() == "aarch64" else "linux_amd64"

  env = os.environ.copy()

  def emit_event(phase, start_time):
    print(f"\n__RUNNER_PHASE__:{phase}:{(time.monotonic() - start_time) * 1000:.0f}", flush=True)

  def get_download_url(file_api_path):
    return _cli_runner_get_download_url(
      workspace_url, token, file_api_path, runner_download_error_marker
    )

  def download_and_extract(file_url, dest_dir):
    response = requests.get(file_url)
    if response.status_code != 200:
      raise RuntimeError(f"HTTP {response.status_code} while downloading artifact: {response.text}")
    tmp_file = f"{dest_dir}.zip"

    with open(tmp_file, "wb") as zip_file:
        zip_file.write(response.content)
    with zipfile.ZipFile(tmp_file, 'r') as zip_ref:
        zip_ref.extractall(dest_dir)

  def download_artifact_if_not_exists(input_file_name, binary_name, dest_path, tmp_dir):
    if os.path.exists(dest_path):
      return

    try:
      os.makedirs(os.path.dirname(dest_path), exist_ok=True)

      path_prefix = f"/DatabricksInternal/Jobs/ArtifactRepository/public/databricks-cli-binaries/{cli_version}"
      download_url = get_download_url(f"{path_prefix}/{input_file_name}.zip")

      tmp_archive_dir = f"{tmp_dir}/{input_file_name}"
      download_and_extract(download_url, tmp_archive_dir)

      src_path = f"{tmp_archive_dir}/{binary_name}"
      if not os.path.exists(src_path):
        available = ", ".join(os.listdir(tmp_archive_dir)) if os.path.isdir(tmp_archive_dir) else "(missing archive directory)"
        raise RuntimeError(f"Expected binary '{binary_name}' not found in archive contents: {available}")

      shutil.move(src_path, dest_path)
      os.chmod(dest_path, 0o755)
    except Exception as e:
      error_msg = f"{artifact_download_error_prefix} {input_file_name}/{binary_name} CLI v{cli_version}: {str(e)}"
      raise RuntimeError(f"{error_msg}")

  tmp_base = os.path.expanduser("~/tmp")
  if not os.path.exists(tmp_base):
    os.makedirs(tmp_base)
  os.chmod(tmp_base, 0o700)

  with tempfile.TemporaryDirectory(dir=tmp_base) as tmpdirname:
      emit_event("setup_runner_environment", setup_runner_environment_start)

      download_artifacts_start = time.monotonic()
      cli_exec_path = f"{tmp_base}/databricks_{cli_version}_{cpu_arch}"
      download_artifact_if_not_exists(
        f"databricks_cli_{cpu_arch}",
        "databricks",
        cli_exec_path,
        tmpdirname
      )
      emit_event("download_artifacts", download_artifacts_start)

      setup_cli_environment_start = time.monotonic()
      cwd = os.path.normpath(f"/Workspace{cwd}")

      if dev_cli_snippet:
        cli_path = cli_exec_path
        exec(dev_cli_snippet, globals(), locals())

      env["DATABRICKS_BUNDLE_TMP"] = f"{tmp_base}/bundle-2-{hashlib.sha256(cwd.encode()).hexdigest()}"
      env["DATABRICKS_CLI_UPSTREAM"] = cli_upstream

      if enable_yaml_sync == "true":
        env["DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC"] = "1"
      else:
        env.pop("DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC", None)

      if enable_deployment_history == "true":
        env["DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY"] = "true"
      else:
        env.pop("DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY", None)

      env["DATABRICKS_BUNDLE_ENGINE"] = "direct"

      emit_event("setup_cli_environment", setup_cli_environment_start)

      if allow_python_execution == "true":
        install_dependencies_start = time.monotonic()
        venv_path = _cli_runner_get_venv_path_from_databricks_yml(cwd)
        venv_folder_path = os.path.normpath(f"{cwd}/{venv_path}")
        venv_bin_path = f"{venv_folder_path}/bin"

        os.makedirs(venv_bin_path, exist_ok=True)

        python3_wrapper_path = f"{venv_bin_path}/python3"
        if not os.path.exists(python3_wrapper_path):
          with open(python3_wrapper_path, "w") as f:
            f.write('#!/usr/bin/env bash\nexec python3 "$@"\n')
          os.chmod(python3_wrapper_path, 0o755)

        pyvenv_cfg_path = f"{venv_folder_path}/pyvenv.cfg"
        if not os.path.exists(pyvenv_cfg_path):
          with open(pyvenv_cfg_path, "w") as f:
            pass

        pyproject_path = f"{cwd}/pyproject.toml"
        uv_pip_install_args = ["uv", "pip", "install", "--quiet", "--python", python3_wrapper_path, "--link-mode=copy"]

        override_file = None
        if os.path.exists(pyproject_path):
          uv_pip_install_args.extend(["-r", pyproject_path, "--group", "dev"])
        else:
          uv_pip_install_args.append(f"databricks-bundles=={cli_version}")

        try:
          subprocess.run(uv_pip_install_args, check=True, cwd=cwd, stdout=subprocess.DEVNULL, env=env)
        finally:
          if override_file and os.path.exists(override_file):
            os.remove(override_file)
        emit_event("install_dependencies", install_dependencies_start)

        env.pop("DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION", None)
      else:
        env["DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION"] = "1"

      env.pop("DATABRICKS_CLUSTER_ID", None)

      with tempfile.NamedTemporaryFile(mode='w+', dir=tmpdirname, delete=True) as databrickscfg_file:
          cli_args = shlex.split(args)
          custom_profiles = None
          if use_custom_profiles == "true":
            custom_profiles = _cli_runner_get_custom_profiles(
              workspace_url,
              str(getattr(ctx, "workspaceId", None) or ""),
              str(getattr(ctx, "workspaceHostName", None) or ""),
            )
          if custom_profiles:
            databrickscfg_file.write(custom_profiles)
            env["DATABRICKS_BUNDLE_STRICT_PROFILE_MATCH"] = "1"
            cli_command = [cli_exec_path] + cli_args
          else:
            profile_name = "workspace_profile"
            databrickscfg_file.write(f"[workspace_profile]\nhost = {workspace_url}")
            cli_command = [cli_exec_path] + cli_args + ["-p", profile_name]
          databrickscfg_file.flush()
          env["DATABRICKS_CONFIG_FILE"] = databrickscfg_file.name

          tmp_file_path = tmp_file

          run_cli_start = time.monotonic()
          run_cli_phase = "run_cli_genie" if cli_upstream == "genie-code" else "run_cli"
          try:
            _cli_runner_run_cli_command_with_token(
              cli_command,
              cwd,
              tmp_file_path,
              include_stderr_in_error == "true",
              token,
              env,
              cli_upstream=cli_upstream,
              capture_stdout=capture_stdout == "true",
            )
          finally:
            emit_event(run_cli_phase, run_cli_start)


try:
  _databricks_cli_runner()
finally:
  del (
    _databricks_cli_runner,
    _cli_runner_get_venv_path_from_databricks_yml,
    _cli_runner_get_custom_profiles,
    _cli_runner_get_download_url,
    _cli_runner_run_cli_command,
    _cli_runner_run_cli_command_with_token,
  )
None


In [0]:
def _cli_runner_get_venv_path_from_databricks_yml(cwd):
  import yaml

  databricks_yml_path = f"{cwd}/databricks.yml"
  with open(databricks_yml_path, "r") as f:
    data = yaml.safe_load(f)
    return data.get("python", {}).get("venv_path", ".venv")


def _cli_runner_get_custom_profiles(workspace_url, workspace_id, custom_url_host):
  import re

  if not re.fullmatch(r"[0-9]+", workspace_id):
    return None
  profiles = "[__settings__]\ndefault_profile = workspace_classic_id\n"
  profiles += f"\n[workspace_classic]\nhost = {workspace_url}\n"
  profiles += f"\n[workspace_classic_id]\nhost = {workspace_url}\nworkspace_id = {workspace_id}\n"
  custom_url_host = custom_url_host.lower()
  if re.fullmatch(r"[a-z0-9-]+(\.[a-z0-9-]+)+", custom_url_host):
    profiles += f"\n[workspace_custom_host]\nhost = https://{custom_url_host}\nworkspace_id = {workspace_id}\n"
  return profiles


def _cli_runner_run_cli_command(cli_command, cwd, tmp_file_path, include_stderr, env, *, cli_upstream, capture_stdout=False):
  import os
  import subprocess

  stderr_arg = subprocess.PIPE if include_stderr else None
  if tmp_file_path or capture_stdout:
    if tmp_file_path:
      os.makedirs(os.path.dirname(tmp_file_path), exist_ok=True)
    result = subprocess.run(cli_command, cwd=cwd, stdout=subprocess.PIPE, stderr=stderr_arg, text=True, env=env)
    if tmp_file_path:
      with open(tmp_file_path, 'w') as output_file:
        output_file.write(result.stdout)
    print(result.stdout)
  else:
    result = subprocess.run(cli_command, cwd=cwd, stderr=stderr_arg, text=True, env=env)
  if result.returncode != 0:
    error_msg = f"Command {result.args[:3]} failed with exit code {result.returncode}"
    if include_stderr and result.stderr:
      error_msg += f":\n{result.stderr}"
    raise RuntimeError(error_msg)

  if cli_upstream == "genie-code":
    print(result.stderr)

    try:
      with open("/Workspace/.proc/self/status", "r") as status_file:
        status_file.read()
    except Exception as e:
      print(f"Warning: Failed to read /Workspace/.proc/self/status: {e}")


def _cli_runner_run_cli_command_with_token(cli_command, cwd, tmp_file_path, include_stderr, token, env, *, cli_upstream, capture_stdout=False):
  _cli_runner_run_cli_command(
    cli_command,
    cwd,
    tmp_file_path,
    include_stderr,
    {**env, "DATABRICKS_TOKEN": token},
    cli_upstream=cli_upstream,
    capture_stdout=capture_stdout,
  )


def _cli_runner_get_download_url(workspace_url, token, file_api_path, runner_download_error_marker):
  import requests
  import time

  api_create_download_url = f"{workspace_url}/api/2.0/fs/create-download-url"
  headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
  data = {"path": file_api_path}

  max_attempts = 3
  last_error = None
  for attempt in range(max_attempts):
    try:
      response = requests.post(api_create_download_url, headers=headers, json=data)
      if response.status_code == 200:
        return response.json()["url"]
      last_error = f"HTTP {response.status_code} {response.text}"
    except requests.RequestException as e:
      last_error = str(e)
    if attempt < max_attempts - 1:
      time.sleep(2 ** attempt)

  print(f"{runner_download_error_marker}: {file_api_path} {last_error}")
  raise RuntimeError(f"Failed to download {file_api_path}: {last_error}")


def _databricks_cli_runner():
  import hashlib
  import os
  import platform
  import requests
  import shlex
  import shutil
  import subprocess
  import tempfile
  import time
  import zipfile
  from dbruntime.databricks_repl_context import get_context
  from urllib.parse import urlparse

  cli_version = "1.24.0"
  cwd = "/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1"
  args = "bundle debug list-targets -o json"
  tmp_file = "/Workspace/Users/deepaksinghal@cyntexa.com/.bundle/claims_lakehouse/.output/list-targets-output-516kuhxcnom.json"
  capture_stdout = "true"
  dev_cli_snippet = ""
  artifact_download_error_prefix = "DABsRunnerError: Failed to download CLI dependency"
  runner_download_error_marker = "DabsRunnerErrorFallback: Downloading error, using fallback URL"
  allow_python_execution = "false"
  enable_yaml_sync = "true"
  enable_deployment_history = "true"
  include_stderr_in_error = "false"
  cli_upstream = "dabs-in-workspace"
  use_custom_profiles = "false"

  if os.environ.get("IS_SERVERLESS", "").strip().lower() != "true":
    raise RuntimeError("Databricks CLI runner must run on serverless compute")

  setup_runner_environment_start = time.monotonic()
  ctx = get_context()
  token = ctx.apiToken
  workspace_url = f"https://{urlparse(ctx.apiUrl).netloc}"
  cpu_arch = "linux_arm64" if platform.machine() == "aarch64" else "linux_amd64"

  env = os.environ.copy()

  def emit_event(phase, start_time):
    print(f"\n__RUNNER_PHASE__:{phase}:{(time.monotonic() - start_time) * 1000:.0f}", flush=True)

  def get_download_url(file_api_path):
    return _cli_runner_get_download_url(
      workspace_url, token, file_api_path, runner_download_error_marker
    )

  def download_and_extract(file_url, dest_dir):
    response = requests.get(file_url)
    if response.status_code != 200:
      raise RuntimeError(f"HTTP {response.status_code} while downloading artifact: {response.text}")
    tmp_file = f"{dest_dir}.zip"

    with open(tmp_file, "wb") as zip_file:
        zip_file.write(response.content)
    with zipfile.ZipFile(tmp_file, 'r') as zip_ref:
        zip_ref.extractall(dest_dir)

  def download_artifact_if_not_exists(input_file_name, binary_name, dest_path, tmp_dir):
    if os.path.exists(dest_path):
      return

    try:
      os.makedirs(os.path.dirname(dest_path), exist_ok=True)

      path_prefix = f"/DatabricksInternal/Jobs/ArtifactRepository/public/databricks-cli-binaries/{cli_version}"
      download_url = get_download_url(f"{path_prefix}/{input_file_name}.zip")

      tmp_archive_dir = f"{tmp_dir}/{input_file_name}"
      download_and_extract(download_url, tmp_archive_dir)

      src_path = f"{tmp_archive_dir}/{binary_name}"
      if not os.path.exists(src_path):
        available = ", ".join(os.listdir(tmp_archive_dir)) if os.path.isdir(tmp_archive_dir) else "(missing archive directory)"
        raise RuntimeError(f"Expected binary '{binary_name}' not found in archive contents: {available}")

      shutil.move(src_path, dest_path)
      os.chmod(dest_path, 0o755)
    except Exception as e:
      error_msg = f"{artifact_download_error_prefix} {input_file_name}/{binary_name} CLI v{cli_version}: {str(e)}"
      raise RuntimeError(f"{error_msg}")

  tmp_base = os.path.expanduser("~/tmp")
  if not os.path.exists(tmp_base):
    os.makedirs(tmp_base)
  os.chmod(tmp_base, 0o700)

  with tempfile.TemporaryDirectory(dir=tmp_base) as tmpdirname:
      emit_event("setup_runner_environment", setup_runner_environment_start)

      download_artifacts_start = time.monotonic()
      cli_exec_path = f"{tmp_base}/databricks_{cli_version}_{cpu_arch}"
      download_artifact_if_not_exists(
        f"databricks_cli_{cpu_arch}",
        "databricks",
        cli_exec_path,
        tmpdirname
      )
      emit_event("download_artifacts", download_artifacts_start)

      setup_cli_environment_start = time.monotonic()
      cwd = os.path.normpath(f"/Workspace{cwd}")

      if dev_cli_snippet:
        cli_path = cli_exec_path
        exec(dev_cli_snippet, globals(), locals())

      env["DATABRICKS_BUNDLE_TMP"] = f"{tmp_base}/bundle-2-{hashlib.sha256(cwd.encode()).hexdigest()}"
      env["DATABRICKS_CLI_UPSTREAM"] = cli_upstream

      if enable_yaml_sync == "true":
        env["DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC"] = "1"
      else:
        env.pop("DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC", None)

      if enable_deployment_history == "true":
        env["DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY"] = "true"
      else:
        env.pop("DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY", None)

      env["DATABRICKS_BUNDLE_ENGINE"] = "direct"

      emit_event("setup_cli_environment", setup_cli_environment_start)

      if allow_python_execution == "true":
        install_dependencies_start = time.monotonic()
        venv_path = _cli_runner_get_venv_path_from_databricks_yml(cwd)
        venv_folder_path = os.path.normpath(f"{cwd}/{venv_path}")
        venv_bin_path = f"{venv_folder_path}/bin"

        os.makedirs(venv_bin_path, exist_ok=True)

        python3_wrapper_path = f"{venv_bin_path}/python3"
        if not os.path.exists(python3_wrapper_path):
          with open(python3_wrapper_path, "w") as f:
            f.write('#!/usr/bin/env bash\nexec python3 "$@"\n')
          os.chmod(python3_wrapper_path, 0o755)

        pyvenv_cfg_path = f"{venv_folder_path}/pyvenv.cfg"
        if not os.path.exists(pyvenv_cfg_path):
          with open(pyvenv_cfg_path, "w") as f:
            pass

        pyproject_path = f"{cwd}/pyproject.toml"
        uv_pip_install_args = ["uv", "pip", "install", "--quiet", "--python", python3_wrapper_path, "--link-mode=copy"]

        override_file = None
        if os.path.exists(pyproject_path):
          uv_pip_install_args.extend(["-r", pyproject_path, "--group", "dev"])
        else:
          uv_pip_install_args.append(f"databricks-bundles=={cli_version}")

        try:
          subprocess.run(uv_pip_install_args, check=True, cwd=cwd, stdout=subprocess.DEVNULL, env=env)
        finally:
          if override_file and os.path.exists(override_file):
            os.remove(override_file)
        emit_event("install_dependencies", install_dependencies_start)

        env.pop("DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION", None)
      else:
        env["DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION"] = "1"

      env.pop("DATABRICKS_CLUSTER_ID", None)

      with tempfile.NamedTemporaryFile(mode='w+', dir=tmpdirname, delete=True) as databrickscfg_file:
          cli_args = shlex.split(args)
          custom_profiles = None
          if use_custom_profiles == "true":
            custom_profiles = _cli_runner_get_custom_profiles(
              workspace_url,
              str(getattr(ctx, "workspaceId", None) or ""),
              str(getattr(ctx, "workspaceHostName", None) or ""),
            )
          if custom_profiles:
            databrickscfg_file.write(custom_profiles)
            env["DATABRICKS_BUNDLE_STRICT_PROFILE_MATCH"] = "1"
            cli_command = [cli_exec_path] + cli_args
          else:
            profile_name = "workspace_profile"
            databrickscfg_file.write(f"[workspace_profile]\nhost = {workspace_url}")
            cli_command = [cli_exec_path] + cli_args + ["-p", profile_name]
          databrickscfg_file.flush()
          env["DATABRICKS_CONFIG_FILE"] = databrickscfg_file.name

          tmp_file_path = tmp_file

          run_cli_start = time.monotonic()
          run_cli_phase = "run_cli_genie" if cli_upstream == "genie-code" else "run_cli"
          try:
            _cli_runner_run_cli_command_with_token(
              cli_command,
              cwd,
              tmp_file_path,
              include_stderr_in_error == "true",
              token,
              env,
              cli_upstream=cli_upstream,
              capture_stdout=capture_stdout == "true",
            )
          finally:
            emit_event(run_cli_phase, run_cli_start)


try:
  _databricks_cli_runner()
finally:
  del (
    _databricks_cli_runner,
    _cli_runner_get_venv_path_from_databricks_yml,
    _cli_runner_get_custom_profiles,
    _cli_runner_get_download_url,
    _cli_runner_run_cli_command,
    _cli_runner_run_cli_command_with_token,
  )
None


In [0]:
def _cli_runner_get_venv_path_from_databricks_yml(cwd):
  import yaml

  databricks_yml_path = f"{cwd}/databricks.yml"
  with open(databricks_yml_path, "r") as f:
    data = yaml.safe_load(f)
    return data.get("python", {}).get("venv_path", ".venv")


def _cli_runner_get_custom_profiles(workspace_url, workspace_id, custom_url_host):
  import re

  if not re.fullmatch(r"[0-9]+", workspace_id):
    return None
  profiles = "[__settings__]\ndefault_profile = workspace_classic_id\n"
  profiles += f"\n[workspace_classic]\nhost = {workspace_url}\n"
  profiles += f"\n[workspace_classic_id]\nhost = {workspace_url}\nworkspace_id = {workspace_id}\n"
  custom_url_host = custom_url_host.lower()
  if re.fullmatch(r"[a-z0-9-]+(\.[a-z0-9-]+)+", custom_url_host):
    profiles += f"\n[workspace_custom_host]\nhost = https://{custom_url_host}\nworkspace_id = {workspace_id}\n"
  return profiles


def _cli_runner_run_cli_command(cli_command, cwd, tmp_file_path, include_stderr, env, *, cli_upstream, capture_stdout=False):
  import os
  import subprocess

  stderr_arg = subprocess.PIPE if include_stderr else None
  if tmp_file_path or capture_stdout:
    if tmp_file_path:
      os.makedirs(os.path.dirname(tmp_file_path), exist_ok=True)
    result = subprocess.run(cli_command, cwd=cwd, stdout=subprocess.PIPE, stderr=stderr_arg, text=True, env=env)
    if tmp_file_path:
      with open(tmp_file_path, 'w') as output_file:
        output_file.write(result.stdout)
    print(result.stdout)
  else:
    result = subprocess.run(cli_command, cwd=cwd, stderr=stderr_arg, text=True, env=env)
  if result.returncode != 0:
    error_msg = f"Command {result.args[:3]} failed with exit code {result.returncode}"
    if include_stderr and result.stderr:
      error_msg += f":\n{result.stderr}"
    raise RuntimeError(error_msg)

  if cli_upstream == "genie-code":
    print(result.stderr)

    try:
      with open("/Workspace/.proc/self/status", "r") as status_file:
        status_file.read()
    except Exception as e:
      print(f"Warning: Failed to read /Workspace/.proc/self/status: {e}")


def _cli_runner_run_cli_command_with_token(cli_command, cwd, tmp_file_path, include_stderr, token, env, *, cli_upstream, capture_stdout=False):
  _cli_runner_run_cli_command(
    cli_command,
    cwd,
    tmp_file_path,
    include_stderr,
    {**env, "DATABRICKS_TOKEN": token},
    cli_upstream=cli_upstream,
    capture_stdout=capture_stdout,
  )


def _cli_runner_get_download_url(workspace_url, token, file_api_path, runner_download_error_marker):
  import requests
  import time

  api_create_download_url = f"{workspace_url}/api/2.0/fs/create-download-url"
  headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
  data = {"path": file_api_path}

  max_attempts = 3
  last_error = None
  for attempt in range(max_attempts):
    try:
      response = requests.post(api_create_download_url, headers=headers, json=data)
      if response.status_code == 200:
        return response.json()["url"]
      last_error = f"HTTP {response.status_code} {response.text}"
    except requests.RequestException as e:
      last_error = str(e)
    if attempt < max_attempts - 1:
      time.sleep(2 ** attempt)

  print(f"{runner_download_error_marker}: {file_api_path} {last_error}")
  raise RuntimeError(f"Failed to download {file_api_path}: {last_error}")


def _databricks_cli_runner():
  import hashlib
  import os
  import platform
  import requests
  import shlex
  import shutil
  import subprocess
  import tempfile
  import time
  import zipfile
  from dbruntime.databricks_repl_context import get_context
  from urllib.parse import urlparse

  cli_version = "1.24.0"
  cwd = "/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1"
  args = "bundle deploy --target dev"
  tmp_file = ""
  capture_stdout = "false"
  dev_cli_snippet = ""
  artifact_download_error_prefix = "DABsRunnerError: Failed to download CLI dependency"
  runner_download_error_marker = "DabsRunnerErrorFallback: Downloading error, using fallback URL"
  allow_python_execution = "false"
  enable_yaml_sync = "true"
  enable_deployment_history = "true"
  include_stderr_in_error = "false"
  cli_upstream = "dabs-in-workspace"
  use_custom_profiles = "false"

  if os.environ.get("IS_SERVERLESS", "").strip().lower() != "true":
    raise RuntimeError("Databricks CLI runner must run on serverless compute")

  setup_runner_environment_start = time.monotonic()
  ctx = get_context()
  token = ctx.apiToken
  workspace_url = f"https://{urlparse(ctx.apiUrl).netloc}"
  cpu_arch = "linux_arm64" if platform.machine() == "aarch64" else "linux_amd64"

  env = os.environ.copy()

  def emit_event(phase, start_time):
    print(f"\n__RUNNER_PHASE__:{phase}:{(time.monotonic() - start_time) * 1000:.0f}", flush=True)

  def get_download_url(file_api_path):
    return _cli_runner_get_download_url(
      workspace_url, token, file_api_path, runner_download_error_marker
    )

  def download_and_extract(file_url, dest_dir):
    response = requests.get(file_url)
    if response.status_code != 200:
      raise RuntimeError(f"HTTP {response.status_code} while downloading artifact: {response.text}")
    tmp_file = f"{dest_dir}.zip"

    with open(tmp_file, "wb") as zip_file:
        zip_file.write(response.content)
    with zipfile.ZipFile(tmp_file, 'r') as zip_ref:
        zip_ref.extractall(dest_dir)

  def download_artifact_if_not_exists(input_file_name, binary_name, dest_path, tmp_dir):
    if os.path.exists(dest_path):
      return

    try:
      os.makedirs(os.path.dirname(dest_path), exist_ok=True)

      path_prefix = f"/DatabricksInternal/Jobs/ArtifactRepository/public/databricks-cli-binaries/{cli_version}"
      download_url = get_download_url(f"{path_prefix}/{input_file_name}.zip")

      tmp_archive_dir = f"{tmp_dir}/{input_file_name}"
      download_and_extract(download_url, tmp_archive_dir)

      src_path = f"{tmp_archive_dir}/{binary_name}"
      if not os.path.exists(src_path):
        available = ", ".join(os.listdir(tmp_archive_dir)) if os.path.isdir(tmp_archive_dir) else "(missing archive directory)"
        raise RuntimeError(f"Expected binary '{binary_name}' not found in archive contents: {available}")

      shutil.move(src_path, dest_path)
      os.chmod(dest_path, 0o755)
    except Exception as e:
      error_msg = f"{artifact_download_error_prefix} {input_file_name}/{binary_name} CLI v{cli_version}: {str(e)}"
      raise RuntimeError(f"{error_msg}")

  tmp_base = os.path.expanduser("~/tmp")
  if not os.path.exists(tmp_base):
    os.makedirs(tmp_base)
  os.chmod(tmp_base, 0o700)

  with tempfile.TemporaryDirectory(dir=tmp_base) as tmpdirname:
      emit_event("setup_runner_environment", setup_runner_environment_start)

      download_artifacts_start = time.monotonic()
      cli_exec_path = f"{tmp_base}/databricks_{cli_version}_{cpu_arch}"
      download_artifact_if_not_exists(
        f"databricks_cli_{cpu_arch}",
        "databricks",
        cli_exec_path,
        tmpdirname
      )
      emit_event("download_artifacts", download_artifacts_start)

      setup_cli_environment_start = time.monotonic()
      cwd = os.path.normpath(f"/Workspace{cwd}")

      if dev_cli_snippet:
        cli_path = cli_exec_path
        exec(dev_cli_snippet, globals(), locals())

      env["DATABRICKS_BUNDLE_TMP"] = f"{tmp_base}/bundle-2-{hashlib.sha256(cwd.encode()).hexdigest()}"
      env["DATABRICKS_CLI_UPSTREAM"] = cli_upstream

      if enable_yaml_sync == "true":
        env["DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC"] = "1"
      else:
        env.pop("DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC", None)

      if enable_deployment_history == "true":
        env["DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY"] = "true"
      else:
        env.pop("DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY", None)

      env["DATABRICKS_BUNDLE_ENGINE"] = "direct"

      emit_event("setup_cli_environment", setup_cli_environment_start)

      if allow_python_execution == "true":
        install_dependencies_start = time.monotonic()
        venv_path = _cli_runner_get_venv_path_from_databricks_yml(cwd)
        venv_folder_path = os.path.normpath(f"{cwd}/{venv_path}")
        venv_bin_path = f"{venv_folder_path}/bin"

        os.makedirs(venv_bin_path, exist_ok=True)

        python3_wrapper_path = f"{venv_bin_path}/python3"
        if not os.path.exists(python3_wrapper_path):
          with open(python3_wrapper_path, "w") as f:
            f.write('#!/usr/bin/env bash\nexec python3 "$@"\n')
          os.chmod(python3_wrapper_path, 0o755)

        pyvenv_cfg_path = f"{venv_folder_path}/pyvenv.cfg"
        if not os.path.exists(pyvenv_cfg_path):
          with open(pyvenv_cfg_path, "w") as f:
            pass

        pyproject_path = f"{cwd}/pyproject.toml"
        uv_pip_install_args = ["uv", "pip", "install", "--quiet", "--python", python3_wrapper_path, "--link-mode=copy"]

        override_file = None
        if os.path.exists(pyproject_path):
          uv_pip_install_args.extend(["-r", pyproject_path, "--group", "dev"])
        else:
          uv_pip_install_args.append(f"databricks-bundles=={cli_version}")

        try:
          subprocess.run(uv_pip_install_args, check=True, cwd=cwd, stdout=subprocess.DEVNULL, env=env)
        finally:
          if override_file and os.path.exists(override_file):
            os.remove(override_file)
        emit_event("install_dependencies", install_dependencies_start)

        env.pop("DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION", None)
      else:
        env["DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION"] = "1"

      env.pop("DATABRICKS_CLUSTER_ID", None)

      with tempfile.NamedTemporaryFile(mode='w+', dir=tmpdirname, delete=True) as databrickscfg_file:
          cli_args = shlex.split(args)
          custom_profiles = None
          if use_custom_profiles == "true":
            custom_profiles = _cli_runner_get_custom_profiles(
              workspace_url,
              str(getattr(ctx, "workspaceId", None) or ""),
              str(getattr(ctx, "workspaceHostName", None) or ""),
            )
          if custom_profiles:
            databrickscfg_file.write(custom_profiles)
            env["DATABRICKS_BUNDLE_STRICT_PROFILE_MATCH"] = "1"
            cli_command = [cli_exec_path] + cli_args
          else:
            profile_name = "workspace_profile"
            databrickscfg_file.write(f"[workspace_profile]\nhost = {workspace_url}")
            cli_command = [cli_exec_path] + cli_args + ["-p", profile_name]
          databrickscfg_file.flush()
          env["DATABRICKS_CONFIG_FILE"] = databrickscfg_file.name

          tmp_file_path = tmp_file

          run_cli_start = time.monotonic()
          run_cli_phase = "run_cli_genie" if cli_upstream == "genie-code" else "run_cli"
          try:
            _cli_runner_run_cli_command_with_token(
              cli_command,
              cwd,
              tmp_file_path,
              include_stderr_in_error == "true",
              token,
              env,
              cli_upstream=cli_upstream,
              capture_stdout=capture_stdout == "true",
            )
          finally:
            emit_event(run_cli_phase, run_cli_start)


try:
  _databricks_cli_runner()
finally:
  del (
    _databricks_cli_runner,
    _cli_runner_get_venv_path_from_databricks_yml,
    _cli_runner_get_custom_profiles,
    _cli_runner_get_download_url,
    _cli_runner_run_cli_command,
    _cli_runner_run_cli_command_with_token,
  )
None


In [0]:
def _cli_runner_get_venv_path_from_databricks_yml(cwd):
  import yaml

  databricks_yml_path = f"{cwd}/databricks.yml"
  with open(databricks_yml_path, "r") as f:
    data = yaml.safe_load(f)
    return data.get("python", {}).get("venv_path", ".venv")


def _cli_runner_get_custom_profiles(workspace_url, workspace_id, custom_url_host):
  import re

  if not re.fullmatch(r"[0-9]+", workspace_id):
    return None
  profiles = "[__settings__]\ndefault_profile = workspace_classic_id\n"
  profiles += f"\n[workspace_classic]\nhost = {workspace_url}\n"
  profiles += f"\n[workspace_classic_id]\nhost = {workspace_url}\nworkspace_id = {workspace_id}\n"
  custom_url_host = custom_url_host.lower()
  if re.fullmatch(r"[a-z0-9-]+(\.[a-z0-9-]+)+", custom_url_host):
    profiles += f"\n[workspace_custom_host]\nhost = https://{custom_url_host}\nworkspace_id = {workspace_id}\n"
  return profiles


def _cli_runner_run_cli_command(cli_command, cwd, tmp_file_path, include_stderr, env, *, cli_upstream, capture_stdout=False):
  import os
  import subprocess

  stderr_arg = subprocess.PIPE if include_stderr else None
  if tmp_file_path or capture_stdout:
    if tmp_file_path:
      os.makedirs(os.path.dirname(tmp_file_path), exist_ok=True)
    result = subprocess.run(cli_command, cwd=cwd, stdout=subprocess.PIPE, stderr=stderr_arg, text=True, env=env)
    if tmp_file_path:
      with open(tmp_file_path, 'w') as output_file:
        output_file.write(result.stdout)
    print(result.stdout)
  else:
    result = subprocess.run(cli_command, cwd=cwd, stderr=stderr_arg, text=True, env=env)
  if result.returncode != 0:
    error_msg = f"Command {result.args[:3]} failed with exit code {result.returncode}"
    if include_stderr and result.stderr:
      error_msg += f":\n{result.stderr}"
    raise RuntimeError(error_msg)

  if cli_upstream == "genie-code":
    print(result.stderr)

    try:
      with open("/Workspace/.proc/self/status", "r") as status_file:
        status_file.read()
    except Exception as e:
      print(f"Warning: Failed to read /Workspace/.proc/self/status: {e}")


def _cli_runner_run_cli_command_with_token(cli_command, cwd, tmp_file_path, include_stderr, token, env, *, cli_upstream, capture_stdout=False):
  _cli_runner_run_cli_command(
    cli_command,
    cwd,
    tmp_file_path,
    include_stderr,
    {**env, "DATABRICKS_TOKEN": token},
    cli_upstream=cli_upstream,
    capture_stdout=capture_stdout,
  )


def _cli_runner_get_download_url(workspace_url, token, file_api_path, runner_download_error_marker):
  import requests
  import time

  api_create_download_url = f"{workspace_url}/api/2.0/fs/create-download-url"
  headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
  data = {"path": file_api_path}

  max_attempts = 3
  last_error = None
  for attempt in range(max_attempts):
    try:
      response = requests.post(api_create_download_url, headers=headers, json=data)
      if response.status_code == 200:
        return response.json()["url"]
      last_error = f"HTTP {response.status_code} {response.text}"
    except requests.RequestException as e:
      last_error = str(e)
    if attempt < max_attempts - 1:
      time.sleep(2 ** attempt)

  print(f"{runner_download_error_marker}: {file_api_path} {last_error}")
  raise RuntimeError(f"Failed to download {file_api_path}: {last_error}")


def _databricks_cli_runner():
  import hashlib
  import os
  import platform
  import requests
  import shlex
  import shutil
  import subprocess
  import tempfile
  import time
  import zipfile
  from dbruntime.databricks_repl_context import get_context
  from urllib.parse import urlparse

  cli_version = "1.24.0"
  cwd = "/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1"
  args = "bundle deploy --target dev"
  tmp_file = ""
  capture_stdout = "false"
  dev_cli_snippet = ""
  artifact_download_error_prefix = "DABsRunnerError: Failed to download CLI dependency"
  runner_download_error_marker = "DabsRunnerErrorFallback: Downloading error, using fallback URL"
  allow_python_execution = "false"
  enable_yaml_sync = "true"
  enable_deployment_history = "true"
  include_stderr_in_error = "false"
  cli_upstream = "dabs-in-workspace"
  use_custom_profiles = "false"

  if os.environ.get("IS_SERVERLESS", "").strip().lower() != "true":
    raise RuntimeError("Databricks CLI runner must run on serverless compute")

  setup_runner_environment_start = time.monotonic()
  ctx = get_context()
  token = ctx.apiToken
  workspace_url = f"https://{urlparse(ctx.apiUrl).netloc}"
  cpu_arch = "linux_arm64" if platform.machine() == "aarch64" else "linux_amd64"

  env = os.environ.copy()

  def emit_event(phase, start_time):
    print(f"\n__RUNNER_PHASE__:{phase}:{(time.monotonic() - start_time) * 1000:.0f}", flush=True)

  def get_download_url(file_api_path):
    return _cli_runner_get_download_url(
      workspace_url, token, file_api_path, runner_download_error_marker
    )

  def download_and_extract(file_url, dest_dir):
    response = requests.get(file_url)
    if response.status_code != 200:
      raise RuntimeError(f"HTTP {response.status_code} while downloading artifact: {response.text}")
    tmp_file = f"{dest_dir}.zip"

    with open(tmp_file, "wb") as zip_file:
        zip_file.write(response.content)
    with zipfile.ZipFile(tmp_file, 'r') as zip_ref:
        zip_ref.extractall(dest_dir)

  def download_artifact_if_not_exists(input_file_name, binary_name, dest_path, tmp_dir):
    if os.path.exists(dest_path):
      return

    try:
      os.makedirs(os.path.dirname(dest_path), exist_ok=True)

      path_prefix = f"/DatabricksInternal/Jobs/ArtifactRepository/public/databricks-cli-binaries/{cli_version}"
      download_url = get_download_url(f"{path_prefix}/{input_file_name}.zip")

      tmp_archive_dir = f"{tmp_dir}/{input_file_name}"
      download_and_extract(download_url, tmp_archive_dir)

      src_path = f"{tmp_archive_dir}/{binary_name}"
      if not os.path.exists(src_path):
        available = ", ".join(os.listdir(tmp_archive_dir)) if os.path.isdir(tmp_archive_dir) else "(missing archive directory)"
        raise RuntimeError(f"Expected binary '{binary_name}' not found in archive contents: {available}")

      shutil.move(src_path, dest_path)
      os.chmod(dest_path, 0o755)
    except Exception as e:
      error_msg = f"{artifact_download_error_prefix} {input_file_name}/{binary_name} CLI v{cli_version}: {str(e)}"
      raise RuntimeError(f"{error_msg}")

  tmp_base = os.path.expanduser("~/tmp")
  if not os.path.exists(tmp_base):
    os.makedirs(tmp_base)
  os.chmod(tmp_base, 0o700)

  with tempfile.TemporaryDirectory(dir=tmp_base) as tmpdirname:
      emit_event("setup_runner_environment", setup_runner_environment_start)

      download_artifacts_start = time.monotonic()
      cli_exec_path = f"{tmp_base}/databricks_{cli_version}_{cpu_arch}"
      download_artifact_if_not_exists(
        f"databricks_cli_{cpu_arch}",
        "databricks",
        cli_exec_path,
        tmpdirname
      )
      emit_event("download_artifacts", download_artifacts_start)

      setup_cli_environment_start = time.monotonic()
      cwd = os.path.normpath(f"/Workspace{cwd}")

      if dev_cli_snippet:
        cli_path = cli_exec_path
        exec(dev_cli_snippet, globals(), locals())

      env["DATABRICKS_BUNDLE_TMP"] = f"{tmp_base}/bundle-2-{hashlib.sha256(cwd.encode()).hexdigest()}"
      env["DATABRICKS_CLI_UPSTREAM"] = cli_upstream

      if enable_yaml_sync == "true":
        env["DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC"] = "1"
      else:
        env.pop("DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC", None)

      if enable_deployment_history == "true":
        env["DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY"] = "true"
      else:
        env.pop("DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY", None)

      env["DATABRICKS_BUNDLE_ENGINE"] = "direct"

      emit_event("setup_cli_environment", setup_cli_environment_start)

      if allow_python_execution == "true":
        install_dependencies_start = time.monotonic()
        venv_path = _cli_runner_get_venv_path_from_databricks_yml(cwd)
        venv_folder_path = os.path.normpath(f"{cwd}/{venv_path}")
        venv_bin_path = f"{venv_folder_path}/bin"

        os.makedirs(venv_bin_path, exist_ok=True)

        python3_wrapper_path = f"{venv_bin_path}/python3"
        if not os.path.exists(python3_wrapper_path):
          with open(python3_wrapper_path, "w") as f:
            f.write('#!/usr/bin/env bash\nexec python3 "$@"\n')
          os.chmod(python3_wrapper_path, 0o755)

        pyvenv_cfg_path = f"{venv_folder_path}/pyvenv.cfg"
        if not os.path.exists(pyvenv_cfg_path):
          with open(pyvenv_cfg_path, "w") as f:
            pass

        pyproject_path = f"{cwd}/pyproject.toml"
        uv_pip_install_args = ["uv", "pip", "install", "--quiet", "--python", python3_wrapper_path, "--link-mode=copy"]

        override_file = None
        if os.path.exists(pyproject_path):
          uv_pip_install_args.extend(["-r", pyproject_path, "--group", "dev"])
        else:
          uv_pip_install_args.append(f"databricks-bundles=={cli_version}")

        try:
          subprocess.run(uv_pip_install_args, check=True, cwd=cwd, stdout=subprocess.DEVNULL, env=env)
        finally:
          if override_file and os.path.exists(override_file):
            os.remove(override_file)
        emit_event("install_dependencies", install_dependencies_start)

        env.pop("DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION", None)
      else:
        env["DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION"] = "1"

      env.pop("DATABRICKS_CLUSTER_ID", None)

      with tempfile.NamedTemporaryFile(mode='w+', dir=tmpdirname, delete=True) as databrickscfg_file:
          cli_args = shlex.split(args)
          custom_profiles = None
          if use_custom_profiles == "true":
            custom_profiles = _cli_runner_get_custom_profiles(
              workspace_url,
              str(getattr(ctx, "workspaceId", None) or ""),
              str(getattr(ctx, "workspaceHostName", None) or ""),
            )
          if custom_profiles:
            databrickscfg_file.write(custom_profiles)
            env["DATABRICKS_BUNDLE_STRICT_PROFILE_MATCH"] = "1"
            cli_command = [cli_exec_path] + cli_args
          else:
            profile_name = "workspace_profile"
            databrickscfg_file.write(f"[workspace_profile]\nhost = {workspace_url}")
            cli_command = [cli_exec_path] + cli_args + ["-p", profile_name]
          databrickscfg_file.flush()
          env["DATABRICKS_CONFIG_FILE"] = databrickscfg_file.name

          tmp_file_path = tmp_file

          run_cli_start = time.monotonic()
          run_cli_phase = "run_cli_genie" if cli_upstream == "genie-code" else "run_cli"
          try:
            _cli_runner_run_cli_command_with_token(
              cli_command,
              cwd,
              tmp_file_path,
              include_stderr_in_error == "true",
              token,
              env,
              cli_upstream=cli_upstream,
              capture_stdout=capture_stdout == "true",
            )
          finally:
            emit_event(run_cli_phase, run_cli_start)


try:
  _databricks_cli_runner()
finally:
  del (
    _databricks_cli_runner,
    _cli_runner_get_venv_path_from_databricks_yml,
    _cli_runner_get_custom_profiles,
    _cli_runner_get_download_url,
    _cli_runner_run_cli_command,
    _cli_runner_run_cli_command_with_token,
  )
None


In [0]:
def _cli_runner_get_venv_path_from_databricks_yml(cwd):
  import yaml

  databricks_yml_path = f"{cwd}/databricks.yml"
  with open(databricks_yml_path, "r") as f:
    data = yaml.safe_load(f)
    return data.get("python", {}).get("venv_path", ".venv")


def _cli_runner_get_custom_profiles(workspace_url, workspace_id, custom_url_host):
  import re

  if not re.fullmatch(r"[0-9]+", workspace_id):
    return None
  profiles = "[__settings__]\ndefault_profile = workspace_classic_id\n"
  profiles += f"\n[workspace_classic]\nhost = {workspace_url}\n"
  profiles += f"\n[workspace_classic_id]\nhost = {workspace_url}\nworkspace_id = {workspace_id}\n"
  custom_url_host = custom_url_host.lower()
  if re.fullmatch(r"[a-z0-9-]+(\.[a-z0-9-]+)+", custom_url_host):
    profiles += f"\n[workspace_custom_host]\nhost = https://{custom_url_host}\nworkspace_id = {workspace_id}\n"
  return profiles


def _cli_runner_run_cli_command(cli_command, cwd, tmp_file_path, include_stderr, env, *, cli_upstream, capture_stdout=False):
  import os
  import subprocess

  stderr_arg = subprocess.PIPE if include_stderr else None
  if tmp_file_path or capture_stdout:
    if tmp_file_path:
      os.makedirs(os.path.dirname(tmp_file_path), exist_ok=True)
    result = subprocess.run(cli_command, cwd=cwd, stdout=subprocess.PIPE, stderr=stderr_arg, text=True, env=env)
    if tmp_file_path:
      with open(tmp_file_path, 'w') as output_file:
        output_file.write(result.stdout)
    print(result.stdout)
  else:
    result = subprocess.run(cli_command, cwd=cwd, stderr=stderr_arg, text=True, env=env)
  if result.returncode != 0:
    error_msg = f"Command {result.args[:3]} failed with exit code {result.returncode}"
    if include_stderr and result.stderr:
      error_msg += f":\n{result.stderr}"
    raise RuntimeError(error_msg)

  if cli_upstream == "genie-code":
    print(result.stderr)

    try:
      with open("/Workspace/.proc/self/status", "r") as status_file:
        status_file.read()
    except Exception as e:
      print(f"Warning: Failed to read /Workspace/.proc/self/status: {e}")


def _cli_runner_run_cli_command_with_token(cli_command, cwd, tmp_file_path, include_stderr, token, env, *, cli_upstream, capture_stdout=False):
  _cli_runner_run_cli_command(
    cli_command,
    cwd,
    tmp_file_path,
    include_stderr,
    {**env, "DATABRICKS_TOKEN": token},
    cli_upstream=cli_upstream,
    capture_stdout=capture_stdout,
  )


def _cli_runner_get_download_url(workspace_url, token, file_api_path, runner_download_error_marker):
  import requests
  import time

  api_create_download_url = f"{workspace_url}/api/2.0/fs/create-download-url"
  headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
  data = {"path": file_api_path}

  max_attempts = 3
  last_error = None
  for attempt in range(max_attempts):
    try:
      response = requests.post(api_create_download_url, headers=headers, json=data)
      if response.status_code == 200:
        return response.json()["url"]
      last_error = f"HTTP {response.status_code} {response.text}"
    except requests.RequestException as e:
      last_error = str(e)
    if attempt < max_attempts - 1:
      time.sleep(2 ** attempt)

  print(f"{runner_download_error_marker}: {file_api_path} {last_error}")
  raise RuntimeError(f"Failed to download {file_api_path}: {last_error}")


def _databricks_cli_runner():
  import hashlib
  import os
  import platform
  import requests
  import shlex
  import shutil
  import subprocess
  import tempfile
  import time
  import zipfile
  from dbruntime.databricks_repl_context import get_context
  from urllib.parse import urlparse

  cli_version = "1.24.0"
  cwd = "/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1"
  args = "bundle summary -o json --include-locations --target dev"
  tmp_file = "/Workspace/Users/deepaksinghal@cyntexa.com/.bundle/claims_lakehouse/.output/async-summary-output-l5lrejcpx2m.json"
  capture_stdout = "true"
  dev_cli_snippet = ""
  artifact_download_error_prefix = "DABsRunnerError: Failed to download CLI dependency"
  runner_download_error_marker = "DabsRunnerErrorFallback: Downloading error, using fallback URL"
  allow_python_execution = "false"
  enable_yaml_sync = "true"
  enable_deployment_history = "true"
  include_stderr_in_error = "false"
  cli_upstream = "dabs-in-workspace"
  use_custom_profiles = "false"

  if os.environ.get("IS_SERVERLESS", "").strip().lower() != "true":
    raise RuntimeError("Databricks CLI runner must run on serverless compute")

  setup_runner_environment_start = time.monotonic()
  ctx = get_context()
  token = ctx.apiToken
  workspace_url = f"https://{urlparse(ctx.apiUrl).netloc}"
  cpu_arch = "linux_arm64" if platform.machine() == "aarch64" else "linux_amd64"

  env = os.environ.copy()

  def emit_event(phase, start_time):
    print(f"\n__RUNNER_PHASE__:{phase}:{(time.monotonic() - start_time) * 1000:.0f}", flush=True)

  def get_download_url(file_api_path):
    return _cli_runner_get_download_url(
      workspace_url, token, file_api_path, runner_download_error_marker
    )

  def download_and_extract(file_url, dest_dir):
    response = requests.get(file_url)
    if response.status_code != 200:
      raise RuntimeError(f"HTTP {response.status_code} while downloading artifact: {response.text}")
    tmp_file = f"{dest_dir}.zip"

    with open(tmp_file, "wb") as zip_file:
        zip_file.write(response.content)
    with zipfile.ZipFile(tmp_file, 'r') as zip_ref:
        zip_ref.extractall(dest_dir)

  def download_artifact_if_not_exists(input_file_name, binary_name, dest_path, tmp_dir):
    if os.path.exists(dest_path):
      return

    try:
      os.makedirs(os.path.dirname(dest_path), exist_ok=True)

      path_prefix = f"/DatabricksInternal/Jobs/ArtifactRepository/public/databricks-cli-binaries/{cli_version}"
      download_url = get_download_url(f"{path_prefix}/{input_file_name}.zip")

      tmp_archive_dir = f"{tmp_dir}/{input_file_name}"
      download_and_extract(download_url, tmp_archive_dir)

      src_path = f"{tmp_archive_dir}/{binary_name}"
      if not os.path.exists(src_path):
        available = ", ".join(os.listdir(tmp_archive_dir)) if os.path.isdir(tmp_archive_dir) else "(missing archive directory)"
        raise RuntimeError(f"Expected binary '{binary_name}' not found in archive contents: {available}")

      shutil.move(src_path, dest_path)
      os.chmod(dest_path, 0o755)
    except Exception as e:
      error_msg = f"{artifact_download_error_prefix} {input_file_name}/{binary_name} CLI v{cli_version}: {str(e)}"
      raise RuntimeError(f"{error_msg}")

  tmp_base = os.path.expanduser("~/tmp")
  if not os.path.exists(tmp_base):
    os.makedirs(tmp_base)
  os.chmod(tmp_base, 0o700)

  with tempfile.TemporaryDirectory(dir=tmp_base) as tmpdirname:
      emit_event("setup_runner_environment", setup_runner_environment_start)

      download_artifacts_start = time.monotonic()
      cli_exec_path = f"{tmp_base}/databricks_{cli_version}_{cpu_arch}"
      download_artifact_if_not_exists(
        f"databricks_cli_{cpu_arch}",
        "databricks",
        cli_exec_path,
        tmpdirname
      )
      emit_event("download_artifacts", download_artifacts_start)

      setup_cli_environment_start = time.monotonic()
      cwd = os.path.normpath(f"/Workspace{cwd}")

      if dev_cli_snippet:
        cli_path = cli_exec_path
        exec(dev_cli_snippet, globals(), locals())

      env["DATABRICKS_BUNDLE_TMP"] = f"{tmp_base}/bundle-2-{hashlib.sha256(cwd.encode()).hexdigest()}"
      env["DATABRICKS_CLI_UPSTREAM"] = cli_upstream

      if enable_yaml_sync == "true":
        env["DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC"] = "1"
      else:
        env.pop("DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC", None)

      if enable_deployment_history == "true":
        env["DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY"] = "true"
      else:
        env.pop("DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY", None)

      env["DATABRICKS_BUNDLE_ENGINE"] = "direct"

      emit_event("setup_cli_environment", setup_cli_environment_start)

      if allow_python_execution == "true":
        install_dependencies_start = time.monotonic()
        venv_path = _cli_runner_get_venv_path_from_databricks_yml(cwd)
        venv_folder_path = os.path.normpath(f"{cwd}/{venv_path}")
        venv_bin_path = f"{venv_folder_path}/bin"

        os.makedirs(venv_bin_path, exist_ok=True)

        python3_wrapper_path = f"{venv_bin_path}/python3"
        if not os.path.exists(python3_wrapper_path):
          with open(python3_wrapper_path, "w") as f:
            f.write('#!/usr/bin/env bash\nexec python3 "$@"\n')
          os.chmod(python3_wrapper_path, 0o755)

        pyvenv_cfg_path = f"{venv_folder_path}/pyvenv.cfg"
        if not os.path.exists(pyvenv_cfg_path):
          with open(pyvenv_cfg_path, "w") as f:
            pass

        pyproject_path = f"{cwd}/pyproject.toml"
        uv_pip_install_args = ["uv", "pip", "install", "--quiet", "--python", python3_wrapper_path, "--link-mode=copy"]

        override_file = None
        if os.path.exists(pyproject_path):
          uv_pip_install_args.extend(["-r", pyproject_path, "--group", "dev"])
        else:
          uv_pip_install_args.append(f"databricks-bundles=={cli_version}")

        try:
          subprocess.run(uv_pip_install_args, check=True, cwd=cwd, stdout=subprocess.DEVNULL, env=env)
        finally:
          if override_file and os.path.exists(override_file):
            os.remove(override_file)
        emit_event("install_dependencies", install_dependencies_start)

        env.pop("DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION", None)
      else:
        env["DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION"] = "1"

      env.pop("DATABRICKS_CLUSTER_ID", None)

      with tempfile.NamedTemporaryFile(mode='w+', dir=tmpdirname, delete=True) as databrickscfg_file:
          cli_args = shlex.split(args)
          custom_profiles = None
          if use_custom_profiles == "true":
            custom_profiles = _cli_runner_get_custom_profiles(
              workspace_url,
              str(getattr(ctx, "workspaceId", None) or ""),
              str(getattr(ctx, "workspaceHostName", None) or ""),
            )
          if custom_profiles:
            databrickscfg_file.write(custom_profiles)
            env["DATABRICKS_BUNDLE_STRICT_PROFILE_MATCH"] = "1"
            cli_command = [cli_exec_path] + cli_args
          else:
            profile_name = "workspace_profile"
            databrickscfg_file.write(f"[workspace_profile]\nhost = {workspace_url}")
            cli_command = [cli_exec_path] + cli_args + ["-p", profile_name]
          databrickscfg_file.flush()
          env["DATABRICKS_CONFIG_FILE"] = databrickscfg_file.name

          tmp_file_path = tmp_file

          run_cli_start = time.monotonic()
          run_cli_phase = "run_cli_genie" if cli_upstream == "genie-code" else "run_cli"
          try:
            _cli_runner_run_cli_command_with_token(
              cli_command,
              cwd,
              tmp_file_path,
              include_stderr_in_error == "true",
              token,
              env,
              cli_upstream=cli_upstream,
              capture_stdout=capture_stdout == "true",
            )
          finally:
            emit_event(run_cli_phase, run_cli_start)


try:
  _databricks_cli_runner()
finally:
  del (
    _databricks_cli_runner,
    _cli_runner_get_venv_path_from_databricks_yml,
    _cli_runner_get_custom_profiles,
    _cli_runner_get_download_url,
    _cli_runner_run_cli_command,
    _cli_runner_run_cli_command_with_token,
  )
None


In [0]:
def _cli_runner_get_venv_path_from_databricks_yml(cwd):
  import yaml

  databricks_yml_path = f"{cwd}/databricks.yml"
  with open(databricks_yml_path, "r") as f:
    data = yaml.safe_load(f)
    return data.get("python", {}).get("venv_path", ".venv")


def _cli_runner_get_custom_profiles(workspace_url, workspace_id, custom_url_host):
  import re

  if not re.fullmatch(r"[0-9]+", workspace_id):
    return None
  profiles = "[__settings__]\ndefault_profile = workspace_classic_id\n"
  profiles += f"\n[workspace_classic]\nhost = {workspace_url}\n"
  profiles += f"\n[workspace_classic_id]\nhost = {workspace_url}\nworkspace_id = {workspace_id}\n"
  custom_url_host = custom_url_host.lower()
  if re.fullmatch(r"[a-z0-9-]+(\.[a-z0-9-]+)+", custom_url_host):
    profiles += f"\n[workspace_custom_host]\nhost = https://{custom_url_host}\nworkspace_id = {workspace_id}\n"
  return profiles


def _cli_runner_run_cli_command(cli_command, cwd, tmp_file_path, include_stderr, env, *, cli_upstream, capture_stdout=False):
  import os
  import subprocess

  stderr_arg = subprocess.PIPE if include_stderr else None
  if tmp_file_path or capture_stdout:
    if tmp_file_path:
      os.makedirs(os.path.dirname(tmp_file_path), exist_ok=True)
    result = subprocess.run(cli_command, cwd=cwd, stdout=subprocess.PIPE, stderr=stderr_arg, text=True, env=env)
    if tmp_file_path:
      with open(tmp_file_path, 'w') as output_file:
        output_file.write(result.stdout)
    print(result.stdout)
  else:
    result = subprocess.run(cli_command, cwd=cwd, stderr=stderr_arg, text=True, env=env)
  if result.returncode != 0:
    error_msg = f"Command {result.args[:3]} failed with exit code {result.returncode}"
    if include_stderr and result.stderr:
      error_msg += f":\n{result.stderr}"
    raise RuntimeError(error_msg)

  if cli_upstream == "genie-code":
    print(result.stderr)

    try:
      with open("/Workspace/.proc/self/status", "r") as status_file:
        status_file.read()
    except Exception as e:
      print(f"Warning: Failed to read /Workspace/.proc/self/status: {e}")


def _cli_runner_run_cli_command_with_token(cli_command, cwd, tmp_file_path, include_stderr, token, env, *, cli_upstream, capture_stdout=False):
  _cli_runner_run_cli_command(
    cli_command,
    cwd,
    tmp_file_path,
    include_stderr,
    {**env, "DATABRICKS_TOKEN": token},
    cli_upstream=cli_upstream,
    capture_stdout=capture_stdout,
  )


def _cli_runner_get_download_url(workspace_url, token, file_api_path, runner_download_error_marker):
  import requests
  import time

  api_create_download_url = f"{workspace_url}/api/2.0/fs/create-download-url"
  headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
  data = {"path": file_api_path}

  max_attempts = 3
  last_error = None
  for attempt in range(max_attempts):
    try:
      response = requests.post(api_create_download_url, headers=headers, json=data)
      if response.status_code == 200:
        return response.json()["url"]
      last_error = f"HTTP {response.status_code} {response.text}"
    except requests.RequestException as e:
      last_error = str(e)
    if attempt < max_attempts - 1:
      time.sleep(2 ** attempt)

  print(f"{runner_download_error_marker}: {file_api_path} {last_error}")
  raise RuntimeError(f"Failed to download {file_api_path}: {last_error}")


def _databricks_cli_runner():
  import hashlib
  import os
  import platform
  import requests
  import shlex
  import shutil
  import subprocess
  import tempfile
  import time
  import zipfile
  from dbruntime.databricks_repl_context import get_context
  from urllib.parse import urlparse

  cli_version = "1.24.0"
  cwd = "/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1"
  args = "bundle summary -o json --include-locations --target dev"
  tmp_file = "/Workspace/Users/deepaksinghal@cyntexa.com/.bundle/claims_lakehouse/.output/async-summary-output-huuyadizhmr.json"
  capture_stdout = "true"
  dev_cli_snippet = ""
  artifact_download_error_prefix = "DABsRunnerError: Failed to download CLI dependency"
  runner_download_error_marker = "DabsRunnerErrorFallback: Downloading error, using fallback URL"
  allow_python_execution = "false"
  enable_yaml_sync = "true"
  enable_deployment_history = "true"
  include_stderr_in_error = "false"
  cli_upstream = "dabs-in-workspace"
  use_custom_profiles = "false"

  if os.environ.get("IS_SERVERLESS", "").strip().lower() != "true":
    raise RuntimeError("Databricks CLI runner must run on serverless compute")

  setup_runner_environment_start = time.monotonic()
  ctx = get_context()
  token = ctx.apiToken
  workspace_url = f"https://{urlparse(ctx.apiUrl).netloc}"
  cpu_arch = "linux_arm64" if platform.machine() == "aarch64" else "linux_amd64"

  env = os.environ.copy()

  def emit_event(phase, start_time):
    print(f"\n__RUNNER_PHASE__:{phase}:{(time.monotonic() - start_time) * 1000:.0f}", flush=True)

  def get_download_url(file_api_path):
    return _cli_runner_get_download_url(
      workspace_url, token, file_api_path, runner_download_error_marker
    )

  def download_and_extract(file_url, dest_dir):
    response = requests.get(file_url)
    if response.status_code != 200:
      raise RuntimeError(f"HTTP {response.status_code} while downloading artifact: {response.text}")
    tmp_file = f"{dest_dir}.zip"

    with open(tmp_file, "wb") as zip_file:
        zip_file.write(response.content)
    with zipfile.ZipFile(tmp_file, 'r') as zip_ref:
        zip_ref.extractall(dest_dir)

  def download_artifact_if_not_exists(input_file_name, binary_name, dest_path, tmp_dir):
    if os.path.exists(dest_path):
      return

    try:
      os.makedirs(os.path.dirname(dest_path), exist_ok=True)

      path_prefix = f"/DatabricksInternal/Jobs/ArtifactRepository/public/databricks-cli-binaries/{cli_version}"
      download_url = get_download_url(f"{path_prefix}/{input_file_name}.zip")

      tmp_archive_dir = f"{tmp_dir}/{input_file_name}"
      download_and_extract(download_url, tmp_archive_dir)

      src_path = f"{tmp_archive_dir}/{binary_name}"
      if not os.path.exists(src_path):
        available = ", ".join(os.listdir(tmp_archive_dir)) if os.path.isdir(tmp_archive_dir) else "(missing archive directory)"
        raise RuntimeError(f"Expected binary '{binary_name}' not found in archive contents: {available}")

      shutil.move(src_path, dest_path)
      os.chmod(dest_path, 0o755)
    except Exception as e:
      error_msg = f"{artifact_download_error_prefix} {input_file_name}/{binary_name} CLI v{cli_version}: {str(e)}"
      raise RuntimeError(f"{error_msg}")

  tmp_base = os.path.expanduser("~/tmp")
  if not os.path.exists(tmp_base):
    os.makedirs(tmp_base)
  os.chmod(tmp_base, 0o700)

  with tempfile.TemporaryDirectory(dir=tmp_base) as tmpdirname:
      emit_event("setup_runner_environment", setup_runner_environment_start)

      download_artifacts_start = time.monotonic()
      cli_exec_path = f"{tmp_base}/databricks_{cli_version}_{cpu_arch}"
      download_artifact_if_not_exists(
        f"databricks_cli_{cpu_arch}",
        "databricks",
        cli_exec_path,
        tmpdirname
      )
      emit_event("download_artifacts", download_artifacts_start)

      setup_cli_environment_start = time.monotonic()
      cwd = os.path.normpath(f"/Workspace{cwd}")

      if dev_cli_snippet:
        cli_path = cli_exec_path
        exec(dev_cli_snippet, globals(), locals())

      env["DATABRICKS_BUNDLE_TMP"] = f"{tmp_base}/bundle-2-{hashlib.sha256(cwd.encode()).hexdigest()}"
      env["DATABRICKS_CLI_UPSTREAM"] = cli_upstream

      if enable_yaml_sync == "true":
        env["DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC"] = "1"
      else:
        env.pop("DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC", None)

      if enable_deployment_history == "true":
        env["DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY"] = "true"
      else:
        env.pop("DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY", None)

      env["DATABRICKS_BUNDLE_ENGINE"] = "direct"

      emit_event("setup_cli_environment", setup_cli_environment_start)

      if allow_python_execution == "true":
        install_dependencies_start = time.monotonic()
        venv_path = _cli_runner_get_venv_path_from_databricks_yml(cwd)
        venv_folder_path = os.path.normpath(f"{cwd}/{venv_path}")
        venv_bin_path = f"{venv_folder_path}/bin"

        os.makedirs(venv_bin_path, exist_ok=True)

        python3_wrapper_path = f"{venv_bin_path}/python3"
        if not os.path.exists(python3_wrapper_path):
          with open(python3_wrapper_path, "w") as f:
            f.write('#!/usr/bin/env bash\nexec python3 "$@"\n')
          os.chmod(python3_wrapper_path, 0o755)

        pyvenv_cfg_path = f"{venv_folder_path}/pyvenv.cfg"
        if not os.path.exists(pyvenv_cfg_path):
          with open(pyvenv_cfg_path, "w") as f:
            pass

        pyproject_path = f"{cwd}/pyproject.toml"
        uv_pip_install_args = ["uv", "pip", "install", "--quiet", "--python", python3_wrapper_path, "--link-mode=copy"]

        override_file = None
        if os.path.exists(pyproject_path):
          uv_pip_install_args.extend(["-r", pyproject_path, "--group", "dev"])
        else:
          uv_pip_install_args.append(f"databricks-bundles=={cli_version}")

        try:
          subprocess.run(uv_pip_install_args, check=True, cwd=cwd, stdout=subprocess.DEVNULL, env=env)
        finally:
          if override_file and os.path.exists(override_file):
            os.remove(override_file)
        emit_event("install_dependencies", install_dependencies_start)

        env.pop("DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION", None)
      else:
        env["DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION"] = "1"

      env.pop("DATABRICKS_CLUSTER_ID", None)

      with tempfile.NamedTemporaryFile(mode='w+', dir=tmpdirname, delete=True) as databrickscfg_file:
          cli_args = shlex.split(args)
          custom_profiles = None
          if use_custom_profiles == "true":
            custom_profiles = _cli_runner_get_custom_profiles(
              workspace_url,
              str(getattr(ctx, "workspaceId", None) or ""),
              str(getattr(ctx, "workspaceHostName", None) or ""),
            )
          if custom_profiles:
            databrickscfg_file.write(custom_profiles)
            env["DATABRICKS_BUNDLE_STRICT_PROFILE_MATCH"] = "1"
            cli_command = [cli_exec_path] + cli_args
          else:
            profile_name = "workspace_profile"
            databrickscfg_file.write(f"[workspace_profile]\nhost = {workspace_url}")
            cli_command = [cli_exec_path] + cli_args + ["-p", profile_name]
          databrickscfg_file.flush()
          env["DATABRICKS_CONFIG_FILE"] = databrickscfg_file.name

          tmp_file_path = tmp_file

          run_cli_start = time.monotonic()
          run_cli_phase = "run_cli_genie" if cli_upstream == "genie-code" else "run_cli"
          try:
            _cli_runner_run_cli_command_with_token(
              cli_command,
              cwd,
              tmp_file_path,
              include_stderr_in_error == "true",
              token,
              env,
              cli_upstream=cli_upstream,
              capture_stdout=capture_stdout == "true",
            )
          finally:
            emit_event(run_cli_phase, run_cli_start)


try:
  _databricks_cli_runner()
finally:
  del (
    _databricks_cli_runner,
    _cli_runner_get_venv_path_from_databricks_yml,
    _cli_runner_get_custom_profiles,
    _cli_runner_get_download_url,
    _cli_runner_run_cli_command,
    _cli_runner_run_cli_command_with_token,
  )
None


In [0]:
def _cli_runner_get_venv_path_from_databricks_yml(cwd):
  import yaml

  databricks_yml_path = f"{cwd}/databricks.yml"
  with open(databricks_yml_path, "r") as f:
    data = yaml.safe_load(f)
    return data.get("python", {}).get("venv_path", ".venv")


def _cli_runner_get_custom_profiles(workspace_url, workspace_id, custom_url_host):
  import re

  if not re.fullmatch(r"[0-9]+", workspace_id):
    return None
  profiles = "[__settings__]\ndefault_profile = workspace_classic_id\n"
  profiles += f"\n[workspace_classic]\nhost = {workspace_url}\n"
  profiles += f"\n[workspace_classic_id]\nhost = {workspace_url}\nworkspace_id = {workspace_id}\n"
  custom_url_host = custom_url_host.lower()
  if re.fullmatch(r"[a-z0-9-]+(\.[a-z0-9-]+)+", custom_url_host):
    profiles += f"\n[workspace_custom_host]\nhost = https://{custom_url_host}\nworkspace_id = {workspace_id}\n"
  return profiles


def _cli_runner_run_cli_command(cli_command, cwd, tmp_file_path, include_stderr, env, *, cli_upstream, capture_stdout=False):
  import os
  import subprocess

  stderr_arg = subprocess.PIPE if include_stderr else None
  if tmp_file_path or capture_stdout:
    if tmp_file_path:
      os.makedirs(os.path.dirname(tmp_file_path), exist_ok=True)
    result = subprocess.run(cli_command, cwd=cwd, stdout=subprocess.PIPE, stderr=stderr_arg, text=True, env=env)
    if tmp_file_path:
      with open(tmp_file_path, 'w') as output_file:
        output_file.write(result.stdout)
    print(result.stdout)
  else:
    result = subprocess.run(cli_command, cwd=cwd, stderr=stderr_arg, text=True, env=env)
  if result.returncode != 0:
    error_msg = f"Command {result.args[:3]} failed with exit code {result.returncode}"
    if include_stderr and result.stderr:
      error_msg += f":\n{result.stderr}"
    raise RuntimeError(error_msg)

  if cli_upstream == "genie-code":
    print(result.stderr)

    try:
      with open("/Workspace/.proc/self/status", "r") as status_file:
        status_file.read()
    except Exception as e:
      print(f"Warning: Failed to read /Workspace/.proc/self/status: {e}")


def _cli_runner_run_cli_command_with_token(cli_command, cwd, tmp_file_path, include_stderr, token, env, *, cli_upstream, capture_stdout=False):
  _cli_runner_run_cli_command(
    cli_command,
    cwd,
    tmp_file_path,
    include_stderr,
    {**env, "DATABRICKS_TOKEN": token},
    cli_upstream=cli_upstream,
    capture_stdout=capture_stdout,
  )


def _cli_runner_get_download_url(workspace_url, token, file_api_path, runner_download_error_marker):
  import requests
  import time

  api_create_download_url = f"{workspace_url}/api/2.0/fs/create-download-url"
  headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
  data = {"path": file_api_path}

  max_attempts = 3
  last_error = None
  for attempt in range(max_attempts):
    try:
      response = requests.post(api_create_download_url, headers=headers, json=data)
      if response.status_code == 200:
        return response.json()["url"]
      last_error = f"HTTP {response.status_code} {response.text}"
    except requests.RequestException as e:
      last_error = str(e)
    if attempt < max_attempts - 1:
      time.sleep(2 ** attempt)

  print(f"{runner_download_error_marker}: {file_api_path} {last_error}")
  raise RuntimeError(f"Failed to download {file_api_path}: {last_error}")


def _databricks_cli_runner():
  import hashlib
  import os
  import platform
  import requests
  import shlex
  import shutil
  import subprocess
  import tempfile
  import time
  import zipfile
  from dbruntime.databricks_repl_context import get_context
  from urllib.parse import urlparse

  cli_version = "1.24.0"
  cwd = "/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1"
  args = "bundle debug list-targets -o json"
  tmp_file = "/Workspace/Users/deepaksinghal@cyntexa.com/.bundle/claims_lakehouse/.output/list-targets-output-q0auiludycl.json"
  capture_stdout = "true"
  dev_cli_snippet = ""
  artifact_download_error_prefix = "DABsRunnerError: Failed to download CLI dependency"
  runner_download_error_marker = "DabsRunnerErrorFallback: Downloading error, using fallback URL"
  allow_python_execution = "false"
  enable_yaml_sync = "true"
  enable_deployment_history = "true"
  include_stderr_in_error = "false"
  cli_upstream = "dabs-in-workspace"
  use_custom_profiles = "false"

  if os.environ.get("IS_SERVERLESS", "").strip().lower() != "true":
    raise RuntimeError("Databricks CLI runner must run on serverless compute")

  setup_runner_environment_start = time.monotonic()
  ctx = get_context()
  token = ctx.apiToken
  workspace_url = f"https://{urlparse(ctx.apiUrl).netloc}"
  cpu_arch = "linux_arm64" if platform.machine() == "aarch64" else "linux_amd64"

  env = os.environ.copy()

  def emit_event(phase, start_time):
    print(f"\n__RUNNER_PHASE__:{phase}:{(time.monotonic() - start_time) * 1000:.0f}", flush=True)

  def get_download_url(file_api_path):
    return _cli_runner_get_download_url(
      workspace_url, token, file_api_path, runner_download_error_marker
    )

  def download_and_extract(file_url, dest_dir):
    response = requests.get(file_url)
    if response.status_code != 200:
      raise RuntimeError(f"HTTP {response.status_code} while downloading artifact: {response.text}")
    tmp_file = f"{dest_dir}.zip"

    with open(tmp_file, "wb") as zip_file:
        zip_file.write(response.content)
    with zipfile.ZipFile(tmp_file, 'r') as zip_ref:
        zip_ref.extractall(dest_dir)

  def download_artifact_if_not_exists(input_file_name, binary_name, dest_path, tmp_dir):
    if os.path.exists(dest_path):
      return

    try:
      os.makedirs(os.path.dirname(dest_path), exist_ok=True)

      path_prefix = f"/DatabricksInternal/Jobs/ArtifactRepository/public/databricks-cli-binaries/{cli_version}"
      download_url = get_download_url(f"{path_prefix}/{input_file_name}.zip")

      tmp_archive_dir = f"{tmp_dir}/{input_file_name}"
      download_and_extract(download_url, tmp_archive_dir)

      src_path = f"{tmp_archive_dir}/{binary_name}"
      if not os.path.exists(src_path):
        available = ", ".join(os.listdir(tmp_archive_dir)) if os.path.isdir(tmp_archive_dir) else "(missing archive directory)"
        raise RuntimeError(f"Expected binary '{binary_name}' not found in archive contents: {available}")

      shutil.move(src_path, dest_path)
      os.chmod(dest_path, 0o755)
    except Exception as e:
      error_msg = f"{artifact_download_error_prefix} {input_file_name}/{binary_name} CLI v{cli_version}: {str(e)}"
      raise RuntimeError(f"{error_msg}")

  tmp_base = os.path.expanduser("~/tmp")
  if not os.path.exists(tmp_base):
    os.makedirs(tmp_base)
  os.chmod(tmp_base, 0o700)

  with tempfile.TemporaryDirectory(dir=tmp_base) as tmpdirname:
      emit_event("setup_runner_environment", setup_runner_environment_start)

      download_artifacts_start = time.monotonic()
      cli_exec_path = f"{tmp_base}/databricks_{cli_version}_{cpu_arch}"
      download_artifact_if_not_exists(
        f"databricks_cli_{cpu_arch}",
        "databricks",
        cli_exec_path,
        tmpdirname
      )
      emit_event("download_artifacts", download_artifacts_start)

      setup_cli_environment_start = time.monotonic()
      cwd = os.path.normpath(f"/Workspace{cwd}")

      if dev_cli_snippet:
        cli_path = cli_exec_path
        exec(dev_cli_snippet, globals(), locals())

      env["DATABRICKS_BUNDLE_TMP"] = f"{tmp_base}/bundle-2-{hashlib.sha256(cwd.encode()).hexdigest()}"
      env["DATABRICKS_CLI_UPSTREAM"] = cli_upstream

      if enable_yaml_sync == "true":
        env["DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC"] = "1"
      else:
        env.pop("DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC", None)

      if enable_deployment_history == "true":
        env["DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY"] = "true"
      else:
        env.pop("DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY", None)

      env["DATABRICKS_BUNDLE_ENGINE"] = "direct"

      emit_event("setup_cli_environment", setup_cli_environment_start)

      if allow_python_execution == "true":
        install_dependencies_start = time.monotonic()
        venv_path = _cli_runner_get_venv_path_from_databricks_yml(cwd)
        venv_folder_path = os.path.normpath(f"{cwd}/{venv_path}")
        venv_bin_path = f"{venv_folder_path}/bin"

        os.makedirs(venv_bin_path, exist_ok=True)

        python3_wrapper_path = f"{venv_bin_path}/python3"
        if not os.path.exists(python3_wrapper_path):
          with open(python3_wrapper_path, "w") as f:
            f.write('#!/usr/bin/env bash\nexec python3 "$@"\n')
          os.chmod(python3_wrapper_path, 0o755)

        pyvenv_cfg_path = f"{venv_folder_path}/pyvenv.cfg"
        if not os.path.exists(pyvenv_cfg_path):
          with open(pyvenv_cfg_path, "w") as f:
            pass

        pyproject_path = f"{cwd}/pyproject.toml"
        uv_pip_install_args = ["uv", "pip", "install", "--quiet", "--python", python3_wrapper_path, "--link-mode=copy"]

        override_file = None
        if os.path.exists(pyproject_path):
          uv_pip_install_args.extend(["-r", pyproject_path, "--group", "dev"])
        else:
          uv_pip_install_args.append(f"databricks-bundles=={cli_version}")

        try:
          subprocess.run(uv_pip_install_args, check=True, cwd=cwd, stdout=subprocess.DEVNULL, env=env)
        finally:
          if override_file and os.path.exists(override_file):
            os.remove(override_file)
        emit_event("install_dependencies", install_dependencies_start)

        env.pop("DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION", None)
      else:
        env["DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION"] = "1"

      env.pop("DATABRICKS_CLUSTER_ID", None)

      with tempfile.NamedTemporaryFile(mode='w+', dir=tmpdirname, delete=True) as databrickscfg_file:
          cli_args = shlex.split(args)
          custom_profiles = None
          if use_custom_profiles == "true":
            custom_profiles = _cli_runner_get_custom_profiles(
              workspace_url,
              str(getattr(ctx, "workspaceId", None) or ""),
              str(getattr(ctx, "workspaceHostName", None) or ""),
            )
          if custom_profiles:
            databrickscfg_file.write(custom_profiles)
            env["DATABRICKS_BUNDLE_STRICT_PROFILE_MATCH"] = "1"
            cli_command = [cli_exec_path] + cli_args
          else:
            profile_name = "workspace_profile"
            databrickscfg_file.write(f"[workspace_profile]\nhost = {workspace_url}")
            cli_command = [cli_exec_path] + cli_args + ["-p", profile_name]
          databrickscfg_file.flush()
          env["DATABRICKS_CONFIG_FILE"] = databrickscfg_file.name

          tmp_file_path = tmp_file

          run_cli_start = time.monotonic()
          run_cli_phase = "run_cli_genie" if cli_upstream == "genie-code" else "run_cli"
          try:
            _cli_runner_run_cli_command_with_token(
              cli_command,
              cwd,
              tmp_file_path,
              include_stderr_in_error == "true",
              token,
              env,
              cli_upstream=cli_upstream,
              capture_stdout=capture_stdout == "true",
            )
          finally:
            emit_event(run_cli_phase, run_cli_start)


try:
  _databricks_cli_runner()
finally:
  del (
    _databricks_cli_runner,
    _cli_runner_get_venv_path_from_databricks_yml,
    _cli_runner_get_custom_profiles,
    _cli_runner_get_download_url,
    _cli_runner_run_cli_command,
    _cli_runner_run_cli_command_with_token,
  )
None


In [0]:
def _cli_runner_get_venv_path_from_databricks_yml(cwd):
  import yaml

  databricks_yml_path = f"{cwd}/databricks.yml"
  with open(databricks_yml_path, "r") as f:
    data = yaml.safe_load(f)
    return data.get("python", {}).get("venv_path", ".venv")


def _cli_runner_get_custom_profiles(workspace_url, workspace_id, custom_url_host):
  import re

  if not re.fullmatch(r"[0-9]+", workspace_id):
    return None
  profiles = "[__settings__]\ndefault_profile = workspace_classic_id\n"
  profiles += f"\n[workspace_classic]\nhost = {workspace_url}\n"
  profiles += f"\n[workspace_classic_id]\nhost = {workspace_url}\nworkspace_id = {workspace_id}\n"
  custom_url_host = custom_url_host.lower()
  if re.fullmatch(r"[a-z0-9-]+(\.[a-z0-9-]+)+", custom_url_host):
    profiles += f"\n[workspace_custom_host]\nhost = https://{custom_url_host}\nworkspace_id = {workspace_id}\n"
  return profiles


def _cli_runner_run_cli_command(cli_command, cwd, tmp_file_path, include_stderr, env, *, cli_upstream, capture_stdout=False):
  import os
  import subprocess

  stderr_arg = subprocess.PIPE if include_stderr else None
  if tmp_file_path or capture_stdout:
    if tmp_file_path:
      os.makedirs(os.path.dirname(tmp_file_path), exist_ok=True)
    result = subprocess.run(cli_command, cwd=cwd, stdout=subprocess.PIPE, stderr=stderr_arg, text=True, env=env)
    if tmp_file_path:
      with open(tmp_file_path, 'w') as output_file:
        output_file.write(result.stdout)
    print(result.stdout)
  else:
    result = subprocess.run(cli_command, cwd=cwd, stderr=stderr_arg, text=True, env=env)
  if result.returncode != 0:
    error_msg = f"Command {result.args[:3]} failed with exit code {result.returncode}"
    if include_stderr and result.stderr:
      error_msg += f":\n{result.stderr}"
    raise RuntimeError(error_msg)

  if cli_upstream == "genie-code":
    print(result.stderr)

    try:
      with open("/Workspace/.proc/self/status", "r") as status_file:
        status_file.read()
    except Exception as e:
      print(f"Warning: Failed to read /Workspace/.proc/self/status: {e}")


def _cli_runner_run_cli_command_with_token(cli_command, cwd, tmp_file_path, include_stderr, token, env, *, cli_upstream, capture_stdout=False):
  _cli_runner_run_cli_command(
    cli_command,
    cwd,
    tmp_file_path,
    include_stderr,
    {**env, "DATABRICKS_TOKEN": token},
    cli_upstream=cli_upstream,
    capture_stdout=capture_stdout,
  )


def _cli_runner_get_download_url(workspace_url, token, file_api_path, runner_download_error_marker):
  import requests
  import time

  api_create_download_url = f"{workspace_url}/api/2.0/fs/create-download-url"
  headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
  data = {"path": file_api_path}

  max_attempts = 3
  last_error = None
  for attempt in range(max_attempts):
    try:
      response = requests.post(api_create_download_url, headers=headers, json=data)
      if response.status_code == 200:
        return response.json()["url"]
      last_error = f"HTTP {response.status_code} {response.text}"
    except requests.RequestException as e:
      last_error = str(e)
    if attempt < max_attempts - 1:
      time.sleep(2 ** attempt)

  print(f"{runner_download_error_marker}: {file_api_path} {last_error}")
  raise RuntimeError(f"Failed to download {file_api_path}: {last_error}")


def _databricks_cli_runner():
  import hashlib
  import os
  import platform
  import requests
  import shlex
  import shutil
  import subprocess
  import tempfile
  import time
  import zipfile
  from dbruntime.databricks_repl_context import get_context
  from urllib.parse import urlparse

  cli_version = "1.24.0"
  cwd = "/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1"
  args = "bundle debug list-targets -o json"
  tmp_file = "/Workspace/Users/deepaksinghal@cyntexa.com/.bundle/claims_lakehouse/.output/list-targets-output-h6ad3qh3ycc.json"
  capture_stdout = "true"
  dev_cli_snippet = ""
  artifact_download_error_prefix = "DABsRunnerError: Failed to download CLI dependency"
  runner_download_error_marker = "DabsRunnerErrorFallback: Downloading error, using fallback URL"
  allow_python_execution = "false"
  enable_yaml_sync = "true"
  enable_deployment_history = "true"
  include_stderr_in_error = "false"
  cli_upstream = "dabs-in-workspace"
  use_custom_profiles = "false"

  if os.environ.get("IS_SERVERLESS", "").strip().lower() != "true":
    raise RuntimeError("Databricks CLI runner must run on serverless compute")

  setup_runner_environment_start = time.monotonic()
  ctx = get_context()
  token = ctx.apiToken
  workspace_url = f"https://{urlparse(ctx.apiUrl).netloc}"
  cpu_arch = "linux_arm64" if platform.machine() == "aarch64" else "linux_amd64"

  env = os.environ.copy()

  def emit_event(phase, start_time):
    print(f"\n__RUNNER_PHASE__:{phase}:{(time.monotonic() - start_time) * 1000:.0f}", flush=True)

  def get_download_url(file_api_path):
    return _cli_runner_get_download_url(
      workspace_url, token, file_api_path, runner_download_error_marker
    )

  def download_and_extract(file_url, dest_dir):
    response = requests.get(file_url)
    if response.status_code != 200:
      raise RuntimeError(f"HTTP {response.status_code} while downloading artifact: {response.text}")
    tmp_file = f"{dest_dir}.zip"

    with open(tmp_file, "wb") as zip_file:
        zip_file.write(response.content)
    with zipfile.ZipFile(tmp_file, 'r') as zip_ref:
        zip_ref.extractall(dest_dir)

  def download_artifact_if_not_exists(input_file_name, binary_name, dest_path, tmp_dir):
    if os.path.exists(dest_path):
      return

    try:
      os.makedirs(os.path.dirname(dest_path), exist_ok=True)

      path_prefix = f"/DatabricksInternal/Jobs/ArtifactRepository/public/databricks-cli-binaries/{cli_version}"
      download_url = get_download_url(f"{path_prefix}/{input_file_name}.zip")

      tmp_archive_dir = f"{tmp_dir}/{input_file_name}"
      download_and_extract(download_url, tmp_archive_dir)

      src_path = f"{tmp_archive_dir}/{binary_name}"
      if not os.path.exists(src_path):
        available = ", ".join(os.listdir(tmp_archive_dir)) if os.path.isdir(tmp_archive_dir) else "(missing archive directory)"
        raise RuntimeError(f"Expected binary '{binary_name}' not found in archive contents: {available}")

      shutil.move(src_path, dest_path)
      os.chmod(dest_path, 0o755)
    except Exception as e:
      error_msg = f"{artifact_download_error_prefix} {input_file_name}/{binary_name} CLI v{cli_version}: {str(e)}"
      raise RuntimeError(f"{error_msg}")

  tmp_base = os.path.expanduser("~/tmp")
  if not os.path.exists(tmp_base):
    os.makedirs(tmp_base)
  os.chmod(tmp_base, 0o700)

  with tempfile.TemporaryDirectory(dir=tmp_base) as tmpdirname:
      emit_event("setup_runner_environment", setup_runner_environment_start)

      download_artifacts_start = time.monotonic()
      cli_exec_path = f"{tmp_base}/databricks_{cli_version}_{cpu_arch}"
      download_artifact_if_not_exists(
        f"databricks_cli_{cpu_arch}",
        "databricks",
        cli_exec_path,
        tmpdirname
      )
      emit_event("download_artifacts", download_artifacts_start)

      setup_cli_environment_start = time.monotonic()
      cwd = os.path.normpath(f"/Workspace{cwd}")

      if dev_cli_snippet:
        cli_path = cli_exec_path
        exec(dev_cli_snippet, globals(), locals())

      env["DATABRICKS_BUNDLE_TMP"] = f"{tmp_base}/bundle-2-{hashlib.sha256(cwd.encode()).hexdigest()}"
      env["DATABRICKS_CLI_UPSTREAM"] = cli_upstream

      if enable_yaml_sync == "true":
        env["DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC"] = "1"
      else:
        env.pop("DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC", None)

      if enable_deployment_history == "true":
        env["DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY"] = "true"
      else:
        env.pop("DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY", None)

      env["DATABRICKS_BUNDLE_ENGINE"] = "direct"

      emit_event("setup_cli_environment", setup_cli_environment_start)

      if allow_python_execution == "true":
        install_dependencies_start = time.monotonic()
        venv_path = _cli_runner_get_venv_path_from_databricks_yml(cwd)
        venv_folder_path = os.path.normpath(f"{cwd}/{venv_path}")
        venv_bin_path = f"{venv_folder_path}/bin"

        os.makedirs(venv_bin_path, exist_ok=True)

        python3_wrapper_path = f"{venv_bin_path}/python3"
        if not os.path.exists(python3_wrapper_path):
          with open(python3_wrapper_path, "w") as f:
            f.write('#!/usr/bin/env bash\nexec python3 "$@"\n')
          os.chmod(python3_wrapper_path, 0o755)

        pyvenv_cfg_path = f"{venv_folder_path}/pyvenv.cfg"
        if not os.path.exists(pyvenv_cfg_path):
          with open(pyvenv_cfg_path, "w") as f:
            pass

        pyproject_path = f"{cwd}/pyproject.toml"
        uv_pip_install_args = ["uv", "pip", "install", "--quiet", "--python", python3_wrapper_path, "--link-mode=copy"]

        override_file = None
        if os.path.exists(pyproject_path):
          uv_pip_install_args.extend(["-r", pyproject_path, "--group", "dev"])
        else:
          uv_pip_install_args.append(f"databricks-bundles=={cli_version}")

        try:
          subprocess.run(uv_pip_install_args, check=True, cwd=cwd, stdout=subprocess.DEVNULL, env=env)
        finally:
          if override_file and os.path.exists(override_file):
            os.remove(override_file)
        emit_event("install_dependencies", install_dependencies_start)

        env.pop("DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION", None)
      else:
        env["DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION"] = "1"

      env.pop("DATABRICKS_CLUSTER_ID", None)

      with tempfile.NamedTemporaryFile(mode='w+', dir=tmpdirname, delete=True) as databrickscfg_file:
          cli_args = shlex.split(args)
          custom_profiles = None
          if use_custom_profiles == "true":
            custom_profiles = _cli_runner_get_custom_profiles(
              workspace_url,
              str(getattr(ctx, "workspaceId", None) or ""),
              str(getattr(ctx, "workspaceHostName", None) or ""),
            )
          if custom_profiles:
            databrickscfg_file.write(custom_profiles)
            env["DATABRICKS_BUNDLE_STRICT_PROFILE_MATCH"] = "1"
            cli_command = [cli_exec_path] + cli_args
          else:
            profile_name = "workspace_profile"
            databrickscfg_file.write(f"[workspace_profile]\nhost = {workspace_url}")
            cli_command = [cli_exec_path] + cli_args + ["-p", profile_name]
          databrickscfg_file.flush()
          env["DATABRICKS_CONFIG_FILE"] = databrickscfg_file.name

          tmp_file_path = tmp_file

          run_cli_start = time.monotonic()
          run_cli_phase = "run_cli_genie" if cli_upstream == "genie-code" else "run_cli"
          try:
            _cli_runner_run_cli_command_with_token(
              cli_command,
              cwd,
              tmp_file_path,
              include_stderr_in_error == "true",
              token,
              env,
              cli_upstream=cli_upstream,
              capture_stdout=capture_stdout == "true",
            )
          finally:
            emit_event(run_cli_phase, run_cli_start)


try:
  _databricks_cli_runner()
finally:
  del (
    _databricks_cli_runner,
    _cli_runner_get_venv_path_from_databricks_yml,
    _cli_runner_get_custom_profiles,
    _cli_runner_get_download_url,
    _cli_runner_run_cli_command,
    _cli_runner_run_cli_command_with_token,
  )
None


In [0]:
def _cli_runner_get_venv_path_from_databricks_yml(cwd):
  import yaml

  databricks_yml_path = f"{cwd}/databricks.yml"
  with open(databricks_yml_path, "r") as f:
    data = yaml.safe_load(f)
    return data.get("python", {}).get("venv_path", ".venv")


def _cli_runner_get_custom_profiles(workspace_url, workspace_id, custom_url_host):
  import re

  if not re.fullmatch(r"[0-9]+", workspace_id):
    return None
  profiles = "[__settings__]\ndefault_profile = workspace_classic_id\n"
  profiles += f"\n[workspace_classic]\nhost = {workspace_url}\n"
  profiles += f"\n[workspace_classic_id]\nhost = {workspace_url}\nworkspace_id = {workspace_id}\n"
  custom_url_host = custom_url_host.lower()
  if re.fullmatch(r"[a-z0-9-]+(\.[a-z0-9-]+)+", custom_url_host):
    profiles += f"\n[workspace_custom_host]\nhost = https://{custom_url_host}\nworkspace_id = {workspace_id}\n"
  return profiles


def _cli_runner_run_cli_command(cli_command, cwd, tmp_file_path, include_stderr, env, *, cli_upstream, capture_stdout=False):
  import os
  import subprocess

  stderr_arg = subprocess.PIPE if include_stderr else None
  if tmp_file_path or capture_stdout:
    if tmp_file_path:
      os.makedirs(os.path.dirname(tmp_file_path), exist_ok=True)
    result = subprocess.run(cli_command, cwd=cwd, stdout=subprocess.PIPE, stderr=stderr_arg, text=True, env=env)
    if tmp_file_path:
      with open(tmp_file_path, 'w') as output_file:
        output_file.write(result.stdout)
    print(result.stdout)
  else:
    result = subprocess.run(cli_command, cwd=cwd, stderr=stderr_arg, text=True, env=env)
  if result.returncode != 0:
    error_msg = f"Command {result.args[:3]} failed with exit code {result.returncode}"
    if include_stderr and result.stderr:
      error_msg += f":\n{result.stderr}"
    raise RuntimeError(error_msg)

  if cli_upstream == "genie-code":
    print(result.stderr)

    try:
      with open("/Workspace/.proc/self/status", "r") as status_file:
        status_file.read()
    except Exception as e:
      print(f"Warning: Failed to read /Workspace/.proc/self/status: {e}")


def _cli_runner_run_cli_command_with_token(cli_command, cwd, tmp_file_path, include_stderr, token, env, *, cli_upstream, capture_stdout=False):
  _cli_runner_run_cli_command(
    cli_command,
    cwd,
    tmp_file_path,
    include_stderr,
    {**env, "DATABRICKS_TOKEN": token},
    cli_upstream=cli_upstream,
    capture_stdout=capture_stdout,
  )


def _cli_runner_get_download_url(workspace_url, token, file_api_path, runner_download_error_marker):
  import requests
  import time

  api_create_download_url = f"{workspace_url}/api/2.0/fs/create-download-url"
  headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
  data = {"path": file_api_path}

  max_attempts = 3
  last_error = None
  for attempt in range(max_attempts):
    try:
      response = requests.post(api_create_download_url, headers=headers, json=data)
      if response.status_code == 200:
        return response.json()["url"]
      last_error = f"HTTP {response.status_code} {response.text}"
    except requests.RequestException as e:
      last_error = str(e)
    if attempt < max_attempts - 1:
      time.sleep(2 ** attempt)

  print(f"{runner_download_error_marker}: {file_api_path} {last_error}")
  raise RuntimeError(f"Failed to download {file_api_path}: {last_error}")


def _databricks_cli_runner():
  import hashlib
  import os
  import platform
  import requests
  import shlex
  import shutil
  import subprocess
  import tempfile
  import time
  import zipfile
  from dbruntime.databricks_repl_context import get_context
  from urllib.parse import urlparse

  cli_version = "1.24.0"
  cwd = "/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1"
  args = "bundle summary -o json --include-locations --target dev"
  tmp_file = "/Workspace/Users/deepaksinghal@cyntexa.com/.bundle/claims_lakehouse/.output/async-summary-output-g7v7okz1ti9.json"
  capture_stdout = "true"
  dev_cli_snippet = ""
  artifact_download_error_prefix = "DABsRunnerError: Failed to download CLI dependency"
  runner_download_error_marker = "DabsRunnerErrorFallback: Downloading error, using fallback URL"
  allow_python_execution = "false"
  enable_yaml_sync = "true"
  enable_deployment_history = "true"
  include_stderr_in_error = "false"
  cli_upstream = "dabs-in-workspace"
  use_custom_profiles = "false"

  if os.environ.get("IS_SERVERLESS", "").strip().lower() != "true":
    raise RuntimeError("Databricks CLI runner must run on serverless compute")

  setup_runner_environment_start = time.monotonic()
  ctx = get_context()
  token = ctx.apiToken
  workspace_url = f"https://{urlparse(ctx.apiUrl).netloc}"
  cpu_arch = "linux_arm64" if platform.machine() == "aarch64" else "linux_amd64"

  env = os.environ.copy()

  def emit_event(phase, start_time):
    print(f"\n__RUNNER_PHASE__:{phase}:{(time.monotonic() - start_time) * 1000:.0f}", flush=True)

  def get_download_url(file_api_path):
    return _cli_runner_get_download_url(
      workspace_url, token, file_api_path, runner_download_error_marker
    )

  def download_and_extract(file_url, dest_dir):
    response = requests.get(file_url)
    if response.status_code != 200:
      raise RuntimeError(f"HTTP {response.status_code} while downloading artifact: {response.text}")
    tmp_file = f"{dest_dir}.zip"

    with open(tmp_file, "wb") as zip_file:
        zip_file.write(response.content)
    with zipfile.ZipFile(tmp_file, 'r') as zip_ref:
        zip_ref.extractall(dest_dir)

  def download_artifact_if_not_exists(input_file_name, binary_name, dest_path, tmp_dir):
    if os.path.exists(dest_path):
      return

    try:
      os.makedirs(os.path.dirname(dest_path), exist_ok=True)

      path_prefix = f"/DatabricksInternal/Jobs/ArtifactRepository/public/databricks-cli-binaries/{cli_version}"
      download_url = get_download_url(f"{path_prefix}/{input_file_name}.zip")

      tmp_archive_dir = f"{tmp_dir}/{input_file_name}"
      download_and_extract(download_url, tmp_archive_dir)

      src_path = f"{tmp_archive_dir}/{binary_name}"
      if not os.path.exists(src_path):
        available = ", ".join(os.listdir(tmp_archive_dir)) if os.path.isdir(tmp_archive_dir) else "(missing archive directory)"
        raise RuntimeError(f"Expected binary '{binary_name}' not found in archive contents: {available}")

      shutil.move(src_path, dest_path)
      os.chmod(dest_path, 0o755)
    except Exception as e:
      error_msg = f"{artifact_download_error_prefix} {input_file_name}/{binary_name} CLI v{cli_version}: {str(e)}"
      raise RuntimeError(f"{error_msg}")

  tmp_base = os.path.expanduser("~/tmp")
  if not os.path.exists(tmp_base):
    os.makedirs(tmp_base)
  os.chmod(tmp_base, 0o700)

  with tempfile.TemporaryDirectory(dir=tmp_base) as tmpdirname:
      emit_event("setup_runner_environment", setup_runner_environment_start)

      download_artifacts_start = time.monotonic()
      cli_exec_path = f"{tmp_base}/databricks_{cli_version}_{cpu_arch}"
      download_artifact_if_not_exists(
        f"databricks_cli_{cpu_arch}",
        "databricks",
        cli_exec_path,
        tmpdirname
      )
      emit_event("download_artifacts", download_artifacts_start)

      setup_cli_environment_start = time.monotonic()
      cwd = os.path.normpath(f"/Workspace{cwd}")

      if dev_cli_snippet:
        cli_path = cli_exec_path
        exec(dev_cli_snippet, globals(), locals())

      env["DATABRICKS_BUNDLE_TMP"] = f"{tmp_base}/bundle-2-{hashlib.sha256(cwd.encode()).hexdigest()}"
      env["DATABRICKS_CLI_UPSTREAM"] = cli_upstream

      if enable_yaml_sync == "true":
        env["DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC"] = "1"
      else:
        env.pop("DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC", None)

      if enable_deployment_history == "true":
        env["DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY"] = "true"
      else:
        env.pop("DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY", None)

      env["DATABRICKS_BUNDLE_ENGINE"] = "direct"

      emit_event("setup_cli_environment", setup_cli_environment_start)

      if allow_python_execution == "true":
        install_dependencies_start = time.monotonic()
        venv_path = _cli_runner_get_venv_path_from_databricks_yml(cwd)
        venv_folder_path = os.path.normpath(f"{cwd}/{venv_path}")
        venv_bin_path = f"{venv_folder_path}/bin"

        os.makedirs(venv_bin_path, exist_ok=True)

        python3_wrapper_path = f"{venv_bin_path}/python3"
        if not os.path.exists(python3_wrapper_path):
          with open(python3_wrapper_path, "w") as f:
            f.write('#!/usr/bin/env bash\nexec python3 "$@"\n')
          os.chmod(python3_wrapper_path, 0o755)

        pyvenv_cfg_path = f"{venv_folder_path}/pyvenv.cfg"
        if not os.path.exists(pyvenv_cfg_path):
          with open(pyvenv_cfg_path, "w") as f:
            pass

        pyproject_path = f"{cwd}/pyproject.toml"
        uv_pip_install_args = ["uv", "pip", "install", "--quiet", "--python", python3_wrapper_path, "--link-mode=copy"]

        override_file = None
        if os.path.exists(pyproject_path):
          uv_pip_install_args.extend(["-r", pyproject_path, "--group", "dev"])
        else:
          uv_pip_install_args.append(f"databricks-bundles=={cli_version}")

        try:
          subprocess.run(uv_pip_install_args, check=True, cwd=cwd, stdout=subprocess.DEVNULL, env=env)
        finally:
          if override_file and os.path.exists(override_file):
            os.remove(override_file)
        emit_event("install_dependencies", install_dependencies_start)

        env.pop("DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION", None)
      else:
        env["DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION"] = "1"

      env.pop("DATABRICKS_CLUSTER_ID", None)

      with tempfile.NamedTemporaryFile(mode='w+', dir=tmpdirname, delete=True) as databrickscfg_file:
          cli_args = shlex.split(args)
          custom_profiles = None
          if use_custom_profiles == "true":
            custom_profiles = _cli_runner_get_custom_profiles(
              workspace_url,
              str(getattr(ctx, "workspaceId", None) or ""),
              str(getattr(ctx, "workspaceHostName", None) or ""),
            )
          if custom_profiles:
            databrickscfg_file.write(custom_profiles)
            env["DATABRICKS_BUNDLE_STRICT_PROFILE_MATCH"] = "1"
            cli_command = [cli_exec_path] + cli_args
          else:
            profile_name = "workspace_profile"
            databrickscfg_file.write(f"[workspace_profile]\nhost = {workspace_url}")
            cli_command = [cli_exec_path] + cli_args + ["-p", profile_name]
          databrickscfg_file.flush()
          env["DATABRICKS_CONFIG_FILE"] = databrickscfg_file.name

          tmp_file_path = tmp_file

          run_cli_start = time.monotonic()
          run_cli_phase = "run_cli_genie" if cli_upstream == "genie-code" else "run_cli"
          try:
            _cli_runner_run_cli_command_with_token(
              cli_command,
              cwd,
              tmp_file_path,
              include_stderr_in_error == "true",
              token,
              env,
              cli_upstream=cli_upstream,
              capture_stdout=capture_stdout == "true",
            )
          finally:
            emit_event(run_cli_phase, run_cli_start)


try:
  _databricks_cli_runner()
finally:
  del (
    _databricks_cli_runner,
    _cli_runner_get_venv_path_from_databricks_yml,
    _cli_runner_get_custom_profiles,
    _cli_runner_get_download_url,
    _cli_runner_run_cli_command,
    _cli_runner_run_cli_command_with_token,
  )
None


In [0]:
def _cli_runner_get_venv_path_from_databricks_yml(cwd):
  import yaml

  databricks_yml_path = f"{cwd}/databricks.yml"
  with open(databricks_yml_path, "r") as f:
    data = yaml.safe_load(f)
    return data.get("python", {}).get("venv_path", ".venv")


def _cli_runner_get_custom_profiles(workspace_url, workspace_id, custom_url_host):
  import re

  if not re.fullmatch(r"[0-9]+", workspace_id):
    return None
  profiles = "[__settings__]\ndefault_profile = workspace_classic_id\n"
  profiles += f"\n[workspace_classic]\nhost = {workspace_url}\n"
  profiles += f"\n[workspace_classic_id]\nhost = {workspace_url}\nworkspace_id = {workspace_id}\n"
  custom_url_host = custom_url_host.lower()
  if re.fullmatch(r"[a-z0-9-]+(\.[a-z0-9-]+)+", custom_url_host):
    profiles += f"\n[workspace_custom_host]\nhost = https://{custom_url_host}\nworkspace_id = {workspace_id}\n"
  return profiles


def _cli_runner_run_cli_command(cli_command, cwd, tmp_file_path, include_stderr, env, *, cli_upstream, capture_stdout=False):
  import os
  import subprocess

  stderr_arg = subprocess.PIPE if include_stderr else None
  if tmp_file_path or capture_stdout:
    if tmp_file_path:
      os.makedirs(os.path.dirname(tmp_file_path), exist_ok=True)
    result = subprocess.run(cli_command, cwd=cwd, stdout=subprocess.PIPE, stderr=stderr_arg, text=True, env=env)
    if tmp_file_path:
      with open(tmp_file_path, 'w') as output_file:
        output_file.write(result.stdout)
    print(result.stdout)
  else:
    result = subprocess.run(cli_command, cwd=cwd, stderr=stderr_arg, text=True, env=env)
  if result.returncode != 0:
    error_msg = f"Command {result.args[:3]} failed with exit code {result.returncode}"
    if include_stderr and result.stderr:
      error_msg += f":\n{result.stderr}"
    raise RuntimeError(error_msg)

  if cli_upstream == "genie-code":
    print(result.stderr)

    try:
      with open("/Workspace/.proc/self/status", "r") as status_file:
        status_file.read()
    except Exception as e:
      print(f"Warning: Failed to read /Workspace/.proc/self/status: {e}")


def _cli_runner_run_cli_command_with_token(cli_command, cwd, tmp_file_path, include_stderr, token, env, *, cli_upstream, capture_stdout=False):
  _cli_runner_run_cli_command(
    cli_command,
    cwd,
    tmp_file_path,
    include_stderr,
    {**env, "DATABRICKS_TOKEN": token},
    cli_upstream=cli_upstream,
    capture_stdout=capture_stdout,
  )


def _cli_runner_get_download_url(workspace_url, token, file_api_path, runner_download_error_marker):
  import requests
  import time

  api_create_download_url = f"{workspace_url}/api/2.0/fs/create-download-url"
  headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
  data = {"path": file_api_path}

  max_attempts = 3
  last_error = None
  for attempt in range(max_attempts):
    try:
      response = requests.post(api_create_download_url, headers=headers, json=data)
      if response.status_code == 200:
        return response.json()["url"]
      last_error = f"HTTP {response.status_code} {response.text}"
    except requests.RequestException as e:
      last_error = str(e)
    if attempt < max_attempts - 1:
      time.sleep(2 ** attempt)

  print(f"{runner_download_error_marker}: {file_api_path} {last_error}")
  raise RuntimeError(f"Failed to download {file_api_path}: {last_error}")


def _databricks_cli_runner():
  import hashlib
  import os
  import platform
  import requests
  import shlex
  import shutil
  import subprocess
  import tempfile
  import time
  import zipfile
  from dbruntime.databricks_repl_context import get_context
  from urllib.parse import urlparse

  cli_version = "1.24.0"
  cwd = "/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1"
  args = "bundle summary -o json --include-locations --target dev"
  tmp_file = "/Workspace/Users/deepaksinghal@cyntexa.com/.bundle/claims_lakehouse/.output/async-summary-output-jgw53l25lb.json"
  capture_stdout = "true"
  dev_cli_snippet = ""
  artifact_download_error_prefix = "DABsRunnerError: Failed to download CLI dependency"
  runner_download_error_marker = "DabsRunnerErrorFallback: Downloading error, using fallback URL"
  allow_python_execution = "false"
  enable_yaml_sync = "true"
  enable_deployment_history = "true"
  include_stderr_in_error = "false"
  cli_upstream = "dabs-in-workspace"
  use_custom_profiles = "false"

  if os.environ.get("IS_SERVERLESS", "").strip().lower() != "true":
    raise RuntimeError("Databricks CLI runner must run on serverless compute")

  setup_runner_environment_start = time.monotonic()
  ctx = get_context()
  token = ctx.apiToken
  workspace_url = f"https://{urlparse(ctx.apiUrl).netloc}"
  cpu_arch = "linux_arm64" if platform.machine() == "aarch64" else "linux_amd64"

  env = os.environ.copy()

  def emit_event(phase, start_time):
    print(f"\n__RUNNER_PHASE__:{phase}:{(time.monotonic() - start_time) * 1000:.0f}", flush=True)

  def get_download_url(file_api_path):
    return _cli_runner_get_download_url(
      workspace_url, token, file_api_path, runner_download_error_marker
    )

  def download_and_extract(file_url, dest_dir):
    response = requests.get(file_url)
    if response.status_code != 200:
      raise RuntimeError(f"HTTP {response.status_code} while downloading artifact: {response.text}")
    tmp_file = f"{dest_dir}.zip"

    with open(tmp_file, "wb") as zip_file:
        zip_file.write(response.content)
    with zipfile.ZipFile(tmp_file, 'r') as zip_ref:
        zip_ref.extractall(dest_dir)

  def download_artifact_if_not_exists(input_file_name, binary_name, dest_path, tmp_dir):
    if os.path.exists(dest_path):
      return

    try:
      os.makedirs(os.path.dirname(dest_path), exist_ok=True)

      path_prefix = f"/DatabricksInternal/Jobs/ArtifactRepository/public/databricks-cli-binaries/{cli_version}"
      download_url = get_download_url(f"{path_prefix}/{input_file_name}.zip")

      tmp_archive_dir = f"{tmp_dir}/{input_file_name}"
      download_and_extract(download_url, tmp_archive_dir)

      src_path = f"{tmp_archive_dir}/{binary_name}"
      if not os.path.exists(src_path):
        available = ", ".join(os.listdir(tmp_archive_dir)) if os.path.isdir(tmp_archive_dir) else "(missing archive directory)"
        raise RuntimeError(f"Expected binary '{binary_name}' not found in archive contents: {available}")

      shutil.move(src_path, dest_path)
      os.chmod(dest_path, 0o755)
    except Exception as e:
      error_msg = f"{artifact_download_error_prefix} {input_file_name}/{binary_name} CLI v{cli_version}: {str(e)}"
      raise RuntimeError(f"{error_msg}")

  tmp_base = os.path.expanduser("~/tmp")
  if not os.path.exists(tmp_base):
    os.makedirs(tmp_base)
  os.chmod(tmp_base, 0o700)

  with tempfile.TemporaryDirectory(dir=tmp_base) as tmpdirname:
      emit_event("setup_runner_environment", setup_runner_environment_start)

      download_artifacts_start = time.monotonic()
      cli_exec_path = f"{tmp_base}/databricks_{cli_version}_{cpu_arch}"
      download_artifact_if_not_exists(
        f"databricks_cli_{cpu_arch}",
        "databricks",
        cli_exec_path,
        tmpdirname
      )
      emit_event("download_artifacts", download_artifacts_start)

      setup_cli_environment_start = time.monotonic()
      cwd = os.path.normpath(f"/Workspace{cwd}")

      if dev_cli_snippet:
        cli_path = cli_exec_path
        exec(dev_cli_snippet, globals(), locals())

      env["DATABRICKS_BUNDLE_TMP"] = f"{tmp_base}/bundle-2-{hashlib.sha256(cwd.encode()).hexdigest()}"
      env["DATABRICKS_CLI_UPSTREAM"] = cli_upstream

      if enable_yaml_sync == "true":
        env["DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC"] = "1"
      else:
        env.pop("DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC", None)

      if enable_deployment_history == "true":
        env["DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY"] = "true"
      else:
        env.pop("DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY", None)

      env["DATABRICKS_BUNDLE_ENGINE"] = "direct"

      emit_event("setup_cli_environment", setup_cli_environment_start)

      if allow_python_execution == "true":
        install_dependencies_start = time.monotonic()
        venv_path = _cli_runner_get_venv_path_from_databricks_yml(cwd)
        venv_folder_path = os.path.normpath(f"{cwd}/{venv_path}")
        venv_bin_path = f"{venv_folder_path}/bin"

        os.makedirs(venv_bin_path, exist_ok=True)

        python3_wrapper_path = f"{venv_bin_path}/python3"
        if not os.path.exists(python3_wrapper_path):
          with open(python3_wrapper_path, "w") as f:
            f.write('#!/usr/bin/env bash\nexec python3 "$@"\n')
          os.chmod(python3_wrapper_path, 0o755)

        pyvenv_cfg_path = f"{venv_folder_path}/pyvenv.cfg"
        if not os.path.exists(pyvenv_cfg_path):
          with open(pyvenv_cfg_path, "w") as f:
            pass

        pyproject_path = f"{cwd}/pyproject.toml"
        uv_pip_install_args = ["uv", "pip", "install", "--quiet", "--python", python3_wrapper_path, "--link-mode=copy"]

        override_file = None
        if os.path.exists(pyproject_path):
          uv_pip_install_args.extend(["-r", pyproject_path, "--group", "dev"])
        else:
          uv_pip_install_args.append(f"databricks-bundles=={cli_version}")

        try:
          subprocess.run(uv_pip_install_args, check=True, cwd=cwd, stdout=subprocess.DEVNULL, env=env)
        finally:
          if override_file and os.path.exists(override_file):
            os.remove(override_file)
        emit_event("install_dependencies", install_dependencies_start)

        env.pop("DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION", None)
      else:
        env["DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION"] = "1"

      env.pop("DATABRICKS_CLUSTER_ID", None)

      with tempfile.NamedTemporaryFile(mode='w+', dir=tmpdirname, delete=True) as databrickscfg_file:
          cli_args = shlex.split(args)
          custom_profiles = None
          if use_custom_profiles == "true":
            custom_profiles = _cli_runner_get_custom_profiles(
              workspace_url,
              str(getattr(ctx, "workspaceId", None) or ""),
              str(getattr(ctx, "workspaceHostName", None) or ""),
            )
          if custom_profiles:
            databrickscfg_file.write(custom_profiles)
            env["DATABRICKS_BUNDLE_STRICT_PROFILE_MATCH"] = "1"
            cli_command = [cli_exec_path] + cli_args
          else:
            profile_name = "workspace_profile"
            databrickscfg_file.write(f"[workspace_profile]\nhost = {workspace_url}")
            cli_command = [cli_exec_path] + cli_args + ["-p", profile_name]
          databrickscfg_file.flush()
          env["DATABRICKS_CONFIG_FILE"] = databrickscfg_file.name

          tmp_file_path = tmp_file

          run_cli_start = time.monotonic()
          run_cli_phase = "run_cli_genie" if cli_upstream == "genie-code" else "run_cli"
          try:
            _cli_runner_run_cli_command_with_token(
              cli_command,
              cwd,
              tmp_file_path,
              include_stderr_in_error == "true",
              token,
              env,
              cli_upstream=cli_upstream,
              capture_stdout=capture_stdout == "true",
            )
          finally:
            emit_event(run_cli_phase, run_cli_start)


try:
  _databricks_cli_runner()
finally:
  del (
    _databricks_cli_runner,
    _cli_runner_get_venv_path_from_databricks_yml,
    _cli_runner_get_custom_profiles,
    _cli_runner_get_download_url,
    _cli_runner_run_cli_command,
    _cli_runner_run_cli_command_with_token,
  )
None


In [0]:
def _cli_runner_get_venv_path_from_databricks_yml(cwd):
  import yaml

  databricks_yml_path = f"{cwd}/databricks.yml"
  with open(databricks_yml_path, "r") as f:
    data = yaml.safe_load(f)
    return data.get("python", {}).get("venv_path", ".venv")


def _cli_runner_get_custom_profiles(workspace_url, workspace_id, custom_url_host):
  import re

  if not re.fullmatch(r"[0-9]+", workspace_id):
    return None
  profiles = "[__settings__]\ndefault_profile = workspace_classic_id\n"
  profiles += f"\n[workspace_classic]\nhost = {workspace_url}\n"
  profiles += f"\n[workspace_classic_id]\nhost = {workspace_url}\nworkspace_id = {workspace_id}\n"
  custom_url_host = custom_url_host.lower()
  if re.fullmatch(r"[a-z0-9-]+(\.[a-z0-9-]+)+", custom_url_host):
    profiles += f"\n[workspace_custom_host]\nhost = https://{custom_url_host}\nworkspace_id = {workspace_id}\n"
  return profiles


def _cli_runner_run_cli_command(cli_command, cwd, tmp_file_path, include_stderr, env, *, cli_upstream, capture_stdout=False):
  import os
  import subprocess

  stderr_arg = subprocess.PIPE if include_stderr else None
  if tmp_file_path or capture_stdout:
    if tmp_file_path:
      os.makedirs(os.path.dirname(tmp_file_path), exist_ok=True)
    result = subprocess.run(cli_command, cwd=cwd, stdout=subprocess.PIPE, stderr=stderr_arg, text=True, env=env)
    if tmp_file_path:
      with open(tmp_file_path, 'w') as output_file:
        output_file.write(result.stdout)
    print(result.stdout)
  else:
    result = subprocess.run(cli_command, cwd=cwd, stderr=stderr_arg, text=True, env=env)
  if result.returncode != 0:
    error_msg = f"Command {result.args[:3]} failed with exit code {result.returncode}"
    if include_stderr and result.stderr:
      error_msg += f":\n{result.stderr}"
    raise RuntimeError(error_msg)

  if cli_upstream == "genie-code":
    print(result.stderr)

    try:
      with open("/Workspace/.proc/self/status", "r") as status_file:
        status_file.read()
    except Exception as e:
      print(f"Warning: Failed to read /Workspace/.proc/self/status: {e}")


def _cli_runner_run_cli_command_with_token(cli_command, cwd, tmp_file_path, include_stderr, token, env, *, cli_upstream, capture_stdout=False):
  _cli_runner_run_cli_command(
    cli_command,
    cwd,
    tmp_file_path,
    include_stderr,
    {**env, "DATABRICKS_TOKEN": token},
    cli_upstream=cli_upstream,
    capture_stdout=capture_stdout,
  )


def _cli_runner_get_download_url(workspace_url, token, file_api_path, runner_download_error_marker):
  import requests
  import time

  api_create_download_url = f"{workspace_url}/api/2.0/fs/create-download-url"
  headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
  data = {"path": file_api_path}

  max_attempts = 3
  last_error = None
  for attempt in range(max_attempts):
    try:
      response = requests.post(api_create_download_url, headers=headers, json=data)
      if response.status_code == 200:
        return response.json()["url"]
      last_error = f"HTTP {response.status_code} {response.text}"
    except requests.RequestException as e:
      last_error = str(e)
    if attempt < max_attempts - 1:
      time.sleep(2 ** attempt)

  print(f"{runner_download_error_marker}: {file_api_path} {last_error}")
  raise RuntimeError(f"Failed to download {file_api_path}: {last_error}")


def _databricks_cli_runner():
  import hashlib
  import os
  import platform
  import requests
  import shlex
  import shutil
  import subprocess
  import tempfile
  import time
  import zipfile
  from dbruntime.databricks_repl_context import get_context
  from urllib.parse import urlparse

  cli_version = "1.24.0"
  cwd = "/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1"
  args = "bundle debug list-targets -o json"
  tmp_file = "/Workspace/Users/deepaksinghal@cyntexa.com/.bundle/claims_lakehouse/.output/list-targets-output-cepoctji16n.json"
  capture_stdout = "true"
  dev_cli_snippet = ""
  artifact_download_error_prefix = "DABsRunnerError: Failed to download CLI dependency"
  runner_download_error_marker = "DabsRunnerErrorFallback: Downloading error, using fallback URL"
  allow_python_execution = "false"
  enable_yaml_sync = "true"
  enable_deployment_history = "true"
  include_stderr_in_error = "false"
  cli_upstream = "dabs-in-workspace"
  use_custom_profiles = "false"

  if os.environ.get("IS_SERVERLESS", "").strip().lower() != "true":
    raise RuntimeError("Databricks CLI runner must run on serverless compute")

  setup_runner_environment_start = time.monotonic()
  ctx = get_context()
  token = ctx.apiToken
  workspace_url = f"https://{urlparse(ctx.apiUrl).netloc}"
  cpu_arch = "linux_arm64" if platform.machine() == "aarch64" else "linux_amd64"

  env = os.environ.copy()

  def emit_event(phase, start_time):
    print(f"\n__RUNNER_PHASE__:{phase}:{(time.monotonic() - start_time) * 1000:.0f}", flush=True)

  def get_download_url(file_api_path):
    return _cli_runner_get_download_url(
      workspace_url, token, file_api_path, runner_download_error_marker
    )

  def download_and_extract(file_url, dest_dir):
    response = requests.get(file_url)
    if response.status_code != 200:
      raise RuntimeError(f"HTTP {response.status_code} while downloading artifact: {response.text}")
    tmp_file = f"{dest_dir}.zip"

    with open(tmp_file, "wb") as zip_file:
        zip_file.write(response.content)
    with zipfile.ZipFile(tmp_file, 'r') as zip_ref:
        zip_ref.extractall(dest_dir)

  def download_artifact_if_not_exists(input_file_name, binary_name, dest_path, tmp_dir):
    if os.path.exists(dest_path):
      return

    try:
      os.makedirs(os.path.dirname(dest_path), exist_ok=True)

      path_prefix = f"/DatabricksInternal/Jobs/ArtifactRepository/public/databricks-cli-binaries/{cli_version}"
      download_url = get_download_url(f"{path_prefix}/{input_file_name}.zip")

      tmp_archive_dir = f"{tmp_dir}/{input_file_name}"
      download_and_extract(download_url, tmp_archive_dir)

      src_path = f"{tmp_archive_dir}/{binary_name}"
      if not os.path.exists(src_path):
        available = ", ".join(os.listdir(tmp_archive_dir)) if os.path.isdir(tmp_archive_dir) else "(missing archive directory)"
        raise RuntimeError(f"Expected binary '{binary_name}' not found in archive contents: {available}")

      shutil.move(src_path, dest_path)
      os.chmod(dest_path, 0o755)
    except Exception as e:
      error_msg = f"{artifact_download_error_prefix} {input_file_name}/{binary_name} CLI v{cli_version}: {str(e)}"
      raise RuntimeError(f"{error_msg}")

  tmp_base = os.path.expanduser("~/tmp")
  if not os.path.exists(tmp_base):
    os.makedirs(tmp_base)
  os.chmod(tmp_base, 0o700)

  with tempfile.TemporaryDirectory(dir=tmp_base) as tmpdirname:
      emit_event("setup_runner_environment", setup_runner_environment_start)

      download_artifacts_start = time.monotonic()
      cli_exec_path = f"{tmp_base}/databricks_{cli_version}_{cpu_arch}"
      download_artifact_if_not_exists(
        f"databricks_cli_{cpu_arch}",
        "databricks",
        cli_exec_path,
        tmpdirname
      )
      emit_event("download_artifacts", download_artifacts_start)

      setup_cli_environment_start = time.monotonic()
      cwd = os.path.normpath(f"/Workspace{cwd}")

      if dev_cli_snippet:
        cli_path = cli_exec_path
        exec(dev_cli_snippet, globals(), locals())

      env["DATABRICKS_BUNDLE_TMP"] = f"{tmp_base}/bundle-2-{hashlib.sha256(cwd.encode()).hexdigest()}"
      env["DATABRICKS_CLI_UPSTREAM"] = cli_upstream

      if enable_yaml_sync == "true":
        env["DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC"] = "1"
      else:
        env.pop("DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC", None)

      if enable_deployment_history == "true":
        env["DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY"] = "true"
      else:
        env.pop("DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY", None)

      env["DATABRICKS_BUNDLE_ENGINE"] = "direct"

      emit_event("setup_cli_environment", setup_cli_environment_start)

      if allow_python_execution == "true":
        install_dependencies_start = time.monotonic()
        venv_path = _cli_runner_get_venv_path_from_databricks_yml(cwd)
        venv_folder_path = os.path.normpath(f"{cwd}/{venv_path}")
        venv_bin_path = f"{venv_folder_path}/bin"

        os.makedirs(venv_bin_path, exist_ok=True)

        python3_wrapper_path = f"{venv_bin_path}/python3"
        if not os.path.exists(python3_wrapper_path):
          with open(python3_wrapper_path, "w") as f:
            f.write('#!/usr/bin/env bash\nexec python3 "$@"\n')
          os.chmod(python3_wrapper_path, 0o755)

        pyvenv_cfg_path = f"{venv_folder_path}/pyvenv.cfg"
        if not os.path.exists(pyvenv_cfg_path):
          with open(pyvenv_cfg_path, "w") as f:
            pass

        pyproject_path = f"{cwd}/pyproject.toml"
        uv_pip_install_args = ["uv", "pip", "install", "--quiet", "--python", python3_wrapper_path, "--link-mode=copy"]

        override_file = None
        if os.path.exists(pyproject_path):
          uv_pip_install_args.extend(["-r", pyproject_path, "--group", "dev"])
        else:
          uv_pip_install_args.append(f"databricks-bundles=={cli_version}")

        try:
          subprocess.run(uv_pip_install_args, check=True, cwd=cwd, stdout=subprocess.DEVNULL, env=env)
        finally:
          if override_file and os.path.exists(override_file):
            os.remove(override_file)
        emit_event("install_dependencies", install_dependencies_start)

        env.pop("DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION", None)
      else:
        env["DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION"] = "1"

      env.pop("DATABRICKS_CLUSTER_ID", None)

      with tempfile.NamedTemporaryFile(mode='w+', dir=tmpdirname, delete=True) as databrickscfg_file:
          cli_args = shlex.split(args)
          custom_profiles = None
          if use_custom_profiles == "true":
            custom_profiles = _cli_runner_get_custom_profiles(
              workspace_url,
              str(getattr(ctx, "workspaceId", None) or ""),
              str(getattr(ctx, "workspaceHostName", None) or ""),
            )
          if custom_profiles:
            databrickscfg_file.write(custom_profiles)
            env["DATABRICKS_BUNDLE_STRICT_PROFILE_MATCH"] = "1"
            cli_command = [cli_exec_path] + cli_args
          else:
            profile_name = "workspace_profile"
            databrickscfg_file.write(f"[workspace_profile]\nhost = {workspace_url}")
            cli_command = [cli_exec_path] + cli_args + ["-p", profile_name]
          databrickscfg_file.flush()
          env["DATABRICKS_CONFIG_FILE"] = databrickscfg_file.name

          tmp_file_path = tmp_file

          run_cli_start = time.monotonic()
          run_cli_phase = "run_cli_genie" if cli_upstream == "genie-code" else "run_cli"
          try:
            _cli_runner_run_cli_command_with_token(
              cli_command,
              cwd,
              tmp_file_path,
              include_stderr_in_error == "true",
              token,
              env,
              cli_upstream=cli_upstream,
              capture_stdout=capture_stdout == "true",
            )
          finally:
            emit_event(run_cli_phase, run_cli_start)


try:
  _databricks_cli_runner()
finally:
  del (
    _databricks_cli_runner,
    _cli_runner_get_venv_path_from_databricks_yml,
    _cli_runner_get_custom_profiles,
    _cli_runner_get_download_url,
    _cli_runner_run_cli_command,
    _cli_runner_run_cli_command_with_token,
  )
None


In [0]:
def _cli_runner_get_venv_path_from_databricks_yml(cwd):
  import yaml

  databricks_yml_path = f"{cwd}/databricks.yml"
  with open(databricks_yml_path, "r") as f:
    data = yaml.safe_load(f)
    return data.get("python", {}).get("venv_path", ".venv")


def _cli_runner_get_custom_profiles(workspace_url, workspace_id, custom_url_host):
  import re

  if not re.fullmatch(r"[0-9]+", workspace_id):
    return None
  profiles = "[__settings__]\ndefault_profile = workspace_classic_id\n"
  profiles += f"\n[workspace_classic]\nhost = {workspace_url}\n"
  profiles += f"\n[workspace_classic_id]\nhost = {workspace_url}\nworkspace_id = {workspace_id}\n"
  custom_url_host = custom_url_host.lower()
  if re.fullmatch(r"[a-z0-9-]+(\.[a-z0-9-]+)+", custom_url_host):
    profiles += f"\n[workspace_custom_host]\nhost = https://{custom_url_host}\nworkspace_id = {workspace_id}\n"
  return profiles


def _cli_runner_run_cli_command(cli_command, cwd, tmp_file_path, include_stderr, env, *, cli_upstream, capture_stdout=False):
  import os
  import subprocess

  stderr_arg = subprocess.PIPE if include_stderr else None
  if tmp_file_path or capture_stdout:
    if tmp_file_path:
      os.makedirs(os.path.dirname(tmp_file_path), exist_ok=True)
    result = subprocess.run(cli_command, cwd=cwd, stdout=subprocess.PIPE, stderr=stderr_arg, text=True, env=env)
    if tmp_file_path:
      with open(tmp_file_path, 'w') as output_file:
        output_file.write(result.stdout)
    print(result.stdout)
  else:
    result = subprocess.run(cli_command, cwd=cwd, stderr=stderr_arg, text=True, env=env)
  if result.returncode != 0:
    error_msg = f"Command {result.args[:3]} failed with exit code {result.returncode}"
    if include_stderr and result.stderr:
      error_msg += f":\n{result.stderr}"
    raise RuntimeError(error_msg)

  if cli_upstream == "genie-code":
    print(result.stderr)

    try:
      with open("/Workspace/.proc/self/status", "r") as status_file:
        status_file.read()
    except Exception as e:
      print(f"Warning: Failed to read /Workspace/.proc/self/status: {e}")


def _cli_runner_run_cli_command_with_token(cli_command, cwd, tmp_file_path, include_stderr, token, env, *, cli_upstream, capture_stdout=False):
  _cli_runner_run_cli_command(
    cli_command,
    cwd,
    tmp_file_path,
    include_stderr,
    {**env, "DATABRICKS_TOKEN": token},
    cli_upstream=cli_upstream,
    capture_stdout=capture_stdout,
  )


def _cli_runner_get_download_url(workspace_url, token, file_api_path, runner_download_error_marker):
  import requests
  import time

  api_create_download_url = f"{workspace_url}/api/2.0/fs/create-download-url"
  headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
  data = {"path": file_api_path}

  max_attempts = 3
  last_error = None
  for attempt in range(max_attempts):
    try:
      response = requests.post(api_create_download_url, headers=headers, json=data)
      if response.status_code == 200:
        return response.json()["url"]
      last_error = f"HTTP {response.status_code} {response.text}"
    except requests.RequestException as e:
      last_error = str(e)
    if attempt < max_attempts - 1:
      time.sleep(2 ** attempt)

  print(f"{runner_download_error_marker}: {file_api_path} {last_error}")
  raise RuntimeError(f"Failed to download {file_api_path}: {last_error}")


def _databricks_cli_runner():
  import hashlib
  import os
  import platform
  import requests
  import shlex
  import shutil
  import subprocess
  import tempfile
  import time
  import zipfile
  from dbruntime.databricks_repl_context import get_context
  from urllib.parse import urlparse

  cli_version = "1.24.0"
  cwd = "/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1"
  args = "bundle summary -o json --include-locations --target dev"
  tmp_file = "/Workspace/Users/deepaksinghal@cyntexa.com/.bundle/claims_lakehouse/.output/async-summary-output-ujjupjphvr.json"
  capture_stdout = "true"
  dev_cli_snippet = ""
  artifact_download_error_prefix = "DABsRunnerError: Failed to download CLI dependency"
  runner_download_error_marker = "DabsRunnerErrorFallback: Downloading error, using fallback URL"
  allow_python_execution = "false"
  enable_yaml_sync = "true"
  enable_deployment_history = "true"
  include_stderr_in_error = "false"
  cli_upstream = "dabs-in-workspace"
  use_custom_profiles = "false"

  if os.environ.get("IS_SERVERLESS", "").strip().lower() != "true":
    raise RuntimeError("Databricks CLI runner must run on serverless compute")

  setup_runner_environment_start = time.monotonic()
  ctx = get_context()
  token = ctx.apiToken
  workspace_url = f"https://{urlparse(ctx.apiUrl).netloc}"
  cpu_arch = "linux_arm64" if platform.machine() == "aarch64" else "linux_amd64"

  env = os.environ.copy()

  def emit_event(phase, start_time):
    print(f"\n__RUNNER_PHASE__:{phase}:{(time.monotonic() - start_time) * 1000:.0f}", flush=True)

  def get_download_url(file_api_path):
    return _cli_runner_get_download_url(
      workspace_url, token, file_api_path, runner_download_error_marker
    )

  def download_and_extract(file_url, dest_dir):
    response = requests.get(file_url)
    if response.status_code != 200:
      raise RuntimeError(f"HTTP {response.status_code} while downloading artifact: {response.text}")
    tmp_file = f"{dest_dir}.zip"

    with open(tmp_file, "wb") as zip_file:
        zip_file.write(response.content)
    with zipfile.ZipFile(tmp_file, 'r') as zip_ref:
        zip_ref.extractall(dest_dir)

  def download_artifact_if_not_exists(input_file_name, binary_name, dest_path, tmp_dir):
    if os.path.exists(dest_path):
      return

    try:
      os.makedirs(os.path.dirname(dest_path), exist_ok=True)

      path_prefix = f"/DatabricksInternal/Jobs/ArtifactRepository/public/databricks-cli-binaries/{cli_version}"
      download_url = get_download_url(f"{path_prefix}/{input_file_name}.zip")

      tmp_archive_dir = f"{tmp_dir}/{input_file_name}"
      download_and_extract(download_url, tmp_archive_dir)

      src_path = f"{tmp_archive_dir}/{binary_name}"
      if not os.path.exists(src_path):
        available = ", ".join(os.listdir(tmp_archive_dir)) if os.path.isdir(tmp_archive_dir) else "(missing archive directory)"
        raise RuntimeError(f"Expected binary '{binary_name}' not found in archive contents: {available}")

      shutil.move(src_path, dest_path)
      os.chmod(dest_path, 0o755)
    except Exception as e:
      error_msg = f"{artifact_download_error_prefix} {input_file_name}/{binary_name} CLI v{cli_version}: {str(e)}"
      raise RuntimeError(f"{error_msg}")

  tmp_base = os.path.expanduser("~/tmp")
  if not os.path.exists(tmp_base):
    os.makedirs(tmp_base)
  os.chmod(tmp_base, 0o700)

  with tempfile.TemporaryDirectory(dir=tmp_base) as tmpdirname:
      emit_event("setup_runner_environment", setup_runner_environment_start)

      download_artifacts_start = time.monotonic()
      cli_exec_path = f"{tmp_base}/databricks_{cli_version}_{cpu_arch}"
      download_artifact_if_not_exists(
        f"databricks_cli_{cpu_arch}",
        "databricks",
        cli_exec_path,
        tmpdirname
      )
      emit_event("download_artifacts", download_artifacts_start)

      setup_cli_environment_start = time.monotonic()
      cwd = os.path.normpath(f"/Workspace{cwd}")

      if dev_cli_snippet:
        cli_path = cli_exec_path
        exec(dev_cli_snippet, globals(), locals())

      env["DATABRICKS_BUNDLE_TMP"] = f"{tmp_base}/bundle-2-{hashlib.sha256(cwd.encode()).hexdigest()}"
      env["DATABRICKS_CLI_UPSTREAM"] = cli_upstream

      if enable_yaml_sync == "true":
        env["DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC"] = "1"
      else:
        env.pop("DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC", None)

      if enable_deployment_history == "true":
        env["DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY"] = "true"
      else:
        env.pop("DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY", None)

      env["DATABRICKS_BUNDLE_ENGINE"] = "direct"

      emit_event("setup_cli_environment", setup_cli_environment_start)

      if allow_python_execution == "true":
        install_dependencies_start = time.monotonic()
        venv_path = _cli_runner_get_venv_path_from_databricks_yml(cwd)
        venv_folder_path = os.path.normpath(f"{cwd}/{venv_path}")
        venv_bin_path = f"{venv_folder_path}/bin"

        os.makedirs(venv_bin_path, exist_ok=True)

        python3_wrapper_path = f"{venv_bin_path}/python3"
        if not os.path.exists(python3_wrapper_path):
          with open(python3_wrapper_path, "w") as f:
            f.write('#!/usr/bin/env bash\nexec python3 "$@"\n')
          os.chmod(python3_wrapper_path, 0o755)

        pyvenv_cfg_path = f"{venv_folder_path}/pyvenv.cfg"
        if not os.path.exists(pyvenv_cfg_path):
          with open(pyvenv_cfg_path, "w") as f:
            pass

        pyproject_path = f"{cwd}/pyproject.toml"
        uv_pip_install_args = ["uv", "pip", "install", "--quiet", "--python", python3_wrapper_path, "--link-mode=copy"]

        override_file = None
        if os.path.exists(pyproject_path):
          uv_pip_install_args.extend(["-r", pyproject_path, "--group", "dev"])
        else:
          uv_pip_install_args.append(f"databricks-bundles=={cli_version}")

        try:
          subprocess.run(uv_pip_install_args, check=True, cwd=cwd, stdout=subprocess.DEVNULL, env=env)
        finally:
          if override_file and os.path.exists(override_file):
            os.remove(override_file)
        emit_event("install_dependencies", install_dependencies_start)

        env.pop("DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION", None)
      else:
        env["DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION"] = "1"

      env.pop("DATABRICKS_CLUSTER_ID", None)

      with tempfile.NamedTemporaryFile(mode='w+', dir=tmpdirname, delete=True) as databrickscfg_file:
          cli_args = shlex.split(args)
          custom_profiles = None
          if use_custom_profiles == "true":
            custom_profiles = _cli_runner_get_custom_profiles(
              workspace_url,
              str(getattr(ctx, "workspaceId", None) or ""),
              str(getattr(ctx, "workspaceHostName", None) or ""),
            )
          if custom_profiles:
            databrickscfg_file.write(custom_profiles)
            env["DATABRICKS_BUNDLE_STRICT_PROFILE_MATCH"] = "1"
            cli_command = [cli_exec_path] + cli_args
          else:
            profile_name = "workspace_profile"
            databrickscfg_file.write(f"[workspace_profile]\nhost = {workspace_url}")
            cli_command = [cli_exec_path] + cli_args + ["-p", profile_name]
          databrickscfg_file.flush()
          env["DATABRICKS_CONFIG_FILE"] = databrickscfg_file.name

          tmp_file_path = tmp_file

          run_cli_start = time.monotonic()
          run_cli_phase = "run_cli_genie" if cli_upstream == "genie-code" else "run_cli"
          try:
            _cli_runner_run_cli_command_with_token(
              cli_command,
              cwd,
              tmp_file_path,
              include_stderr_in_error == "true",
              token,
              env,
              cli_upstream=cli_upstream,
              capture_stdout=capture_stdout == "true",
            )
          finally:
            emit_event(run_cli_phase, run_cli_start)


try:
  _databricks_cli_runner()
finally:
  del (
    _databricks_cli_runner,
    _cli_runner_get_venv_path_from_databricks_yml,
    _cli_runner_get_custom_profiles,
    _cli_runner_get_download_url,
    _cli_runner_run_cli_command,
    _cli_runner_run_cli_command_with_token,
  )
None


In [0]:
def _cli_runner_get_venv_path_from_databricks_yml(cwd):
  import yaml

  databricks_yml_path = f"{cwd}/databricks.yml"
  with open(databricks_yml_path, "r") as f:
    data = yaml.safe_load(f)
    return data.get("python", {}).get("venv_path", ".venv")


def _cli_runner_get_custom_profiles(workspace_url, workspace_id, custom_url_host):
  import re

  if not re.fullmatch(r"[0-9]+", workspace_id):
    return None
  profiles = "[__settings__]\ndefault_profile = workspace_classic_id\n"
  profiles += f"\n[workspace_classic]\nhost = {workspace_url}\n"
  profiles += f"\n[workspace_classic_id]\nhost = {workspace_url}\nworkspace_id = {workspace_id}\n"
  custom_url_host = custom_url_host.lower()
  if re.fullmatch(r"[a-z0-9-]+(\.[a-z0-9-]+)+", custom_url_host):
    profiles += f"\n[workspace_custom_host]\nhost = https://{custom_url_host}\nworkspace_id = {workspace_id}\n"
  return profiles


def _cli_runner_run_cli_command(cli_command, cwd, tmp_file_path, include_stderr, env, *, cli_upstream, capture_stdout=False):
  import os
  import subprocess

  stderr_arg = subprocess.PIPE if include_stderr else None
  if tmp_file_path or capture_stdout:
    if tmp_file_path:
      os.makedirs(os.path.dirname(tmp_file_path), exist_ok=True)
    result = subprocess.run(cli_command, cwd=cwd, stdout=subprocess.PIPE, stderr=stderr_arg, text=True, env=env)
    if tmp_file_path:
      with open(tmp_file_path, 'w') as output_file:
        output_file.write(result.stdout)
    print(result.stdout)
  else:
    result = subprocess.run(cli_command, cwd=cwd, stderr=stderr_arg, text=True, env=env)
  if result.returncode != 0:
    error_msg = f"Command {result.args[:3]} failed with exit code {result.returncode}"
    if include_stderr and result.stderr:
      error_msg += f":\n{result.stderr}"
    raise RuntimeError(error_msg)

  if cli_upstream == "genie-code":
    print(result.stderr)

    try:
      with open("/Workspace/.proc/self/status", "r") as status_file:
        status_file.read()
    except Exception as e:
      print(f"Warning: Failed to read /Workspace/.proc/self/status: {e}")


def _cli_runner_run_cli_command_with_token(cli_command, cwd, tmp_file_path, include_stderr, token, env, *, cli_upstream, capture_stdout=False):
  _cli_runner_run_cli_command(
    cli_command,
    cwd,
    tmp_file_path,
    include_stderr,
    {**env, "DATABRICKS_TOKEN": token},
    cli_upstream=cli_upstream,
    capture_stdout=capture_stdout,
  )


def _cli_runner_get_download_url(workspace_url, token, file_api_path, runner_download_error_marker):
  import requests
  import time

  api_create_download_url = f"{workspace_url}/api/2.0/fs/create-download-url"
  headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
  data = {"path": file_api_path}

  max_attempts = 3
  last_error = None
  for attempt in range(max_attempts):
    try:
      response = requests.post(api_create_download_url, headers=headers, json=data)
      if response.status_code == 200:
        return response.json()["url"]
      last_error = f"HTTP {response.status_code} {response.text}"
    except requests.RequestException as e:
      last_error = str(e)
    if attempt < max_attempts - 1:
      time.sleep(2 ** attempt)

  print(f"{runner_download_error_marker}: {file_api_path} {last_error}")
  raise RuntimeError(f"Failed to download {file_api_path}: {last_error}")


def _databricks_cli_runner():
  import hashlib
  import os
  import platform
  import requests
  import shlex
  import shutil
  import subprocess
  import tempfile
  import time
  import zipfile
  from dbruntime.databricks_repl_context import get_context
  from urllib.parse import urlparse

  cli_version = "1.24.0"
  cwd = "/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1"
  args = "bundle debug list-targets -o json"
  tmp_file = "/Workspace/Users/deepaksinghal@cyntexa.com/.bundle/claims_lakehouse/.output/list-targets-output-4j3z8g0kov.json"
  capture_stdout = "true"
  dev_cli_snippet = ""
  artifact_download_error_prefix = "DABsRunnerError: Failed to download CLI dependency"
  runner_download_error_marker = "DabsRunnerErrorFallback: Downloading error, using fallback URL"
  allow_python_execution = "false"
  enable_yaml_sync = "true"
  enable_deployment_history = "true"
  include_stderr_in_error = "false"
  cli_upstream = "dabs-in-workspace"
  use_custom_profiles = "false"

  if os.environ.get("IS_SERVERLESS", "").strip().lower() != "true":
    raise RuntimeError("Databricks CLI runner must run on serverless compute")

  setup_runner_environment_start = time.monotonic()
  ctx = get_context()
  token = ctx.apiToken
  workspace_url = f"https://{urlparse(ctx.apiUrl).netloc}"
  cpu_arch = "linux_arm64" if platform.machine() == "aarch64" else "linux_amd64"

  env = os.environ.copy()

  def emit_event(phase, start_time):
    print(f"\n__RUNNER_PHASE__:{phase}:{(time.monotonic() - start_time) * 1000:.0f}", flush=True)

  def get_download_url(file_api_path):
    return _cli_runner_get_download_url(
      workspace_url, token, file_api_path, runner_download_error_marker
    )

  def download_and_extract(file_url, dest_dir):
    response = requests.get(file_url)
    if response.status_code != 200:
      raise RuntimeError(f"HTTP {response.status_code} while downloading artifact: {response.text}")
    tmp_file = f"{dest_dir}.zip"

    with open(tmp_file, "wb") as zip_file:
        zip_file.write(response.content)
    with zipfile.ZipFile(tmp_file, 'r') as zip_ref:
        zip_ref.extractall(dest_dir)

  def download_artifact_if_not_exists(input_file_name, binary_name, dest_path, tmp_dir):
    if os.path.exists(dest_path):
      return

    try:
      os.makedirs(os.path.dirname(dest_path), exist_ok=True)

      path_prefix = f"/DatabricksInternal/Jobs/ArtifactRepository/public/databricks-cli-binaries/{cli_version}"
      download_url = get_download_url(f"{path_prefix}/{input_file_name}.zip")

      tmp_archive_dir = f"{tmp_dir}/{input_file_name}"
      download_and_extract(download_url, tmp_archive_dir)

      src_path = f"{tmp_archive_dir}/{binary_name}"
      if not os.path.exists(src_path):
        available = ", ".join(os.listdir(tmp_archive_dir)) if os.path.isdir(tmp_archive_dir) else "(missing archive directory)"
        raise RuntimeError(f"Expected binary '{binary_name}' not found in archive contents: {available}")

      shutil.move(src_path, dest_path)
      os.chmod(dest_path, 0o755)
    except Exception as e:
      error_msg = f"{artifact_download_error_prefix} {input_file_name}/{binary_name} CLI v{cli_version}: {str(e)}"
      raise RuntimeError(f"{error_msg}")

  tmp_base = os.path.expanduser("~/tmp")
  if not os.path.exists(tmp_base):
    os.makedirs(tmp_base)
  os.chmod(tmp_base, 0o700)

  with tempfile.TemporaryDirectory(dir=tmp_base) as tmpdirname:
      emit_event("setup_runner_environment", setup_runner_environment_start)

      download_artifacts_start = time.monotonic()
      cli_exec_path = f"{tmp_base}/databricks_{cli_version}_{cpu_arch}"
      download_artifact_if_not_exists(
        f"databricks_cli_{cpu_arch}",
        "databricks",
        cli_exec_path,
        tmpdirname
      )
      emit_event("download_artifacts", download_artifacts_start)

      setup_cli_environment_start = time.monotonic()
      cwd = os.path.normpath(f"/Workspace{cwd}")

      if dev_cli_snippet:
        cli_path = cli_exec_path
        exec(dev_cli_snippet, globals(), locals())

      env["DATABRICKS_BUNDLE_TMP"] = f"{tmp_base}/bundle-2-{hashlib.sha256(cwd.encode()).hexdigest()}"
      env["DATABRICKS_CLI_UPSTREAM"] = cli_upstream

      if enable_yaml_sync == "true":
        env["DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC"] = "1"
      else:
        env.pop("DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC", None)

      if enable_deployment_history == "true":
        env["DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY"] = "true"
      else:
        env.pop("DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY", None)

      env["DATABRICKS_BUNDLE_ENGINE"] = "direct"

      emit_event("setup_cli_environment", setup_cli_environment_start)

      if allow_python_execution == "true":
        install_dependencies_start = time.monotonic()
        venv_path = _cli_runner_get_venv_path_from_databricks_yml(cwd)
        venv_folder_path = os.path.normpath(f"{cwd}/{venv_path}")
        venv_bin_path = f"{venv_folder_path}/bin"

        os.makedirs(venv_bin_path, exist_ok=True)

        python3_wrapper_path = f"{venv_bin_path}/python3"
        if not os.path.exists(python3_wrapper_path):
          with open(python3_wrapper_path, "w") as f:
            f.write('#!/usr/bin/env bash\nexec python3 "$@"\n')
          os.chmod(python3_wrapper_path, 0o755)

        pyvenv_cfg_path = f"{venv_folder_path}/pyvenv.cfg"
        if not os.path.exists(pyvenv_cfg_path):
          with open(pyvenv_cfg_path, "w") as f:
            pass

        pyproject_path = f"{cwd}/pyproject.toml"
        uv_pip_install_args = ["uv", "pip", "install", "--quiet", "--python", python3_wrapper_path, "--link-mode=copy"]

        override_file = None
        if os.path.exists(pyproject_path):
          uv_pip_install_args.extend(["-r", pyproject_path, "--group", "dev"])
        else:
          uv_pip_install_args.append(f"databricks-bundles=={cli_version}")

        try:
          subprocess.run(uv_pip_install_args, check=True, cwd=cwd, stdout=subprocess.DEVNULL, env=env)
        finally:
          if override_file and os.path.exists(override_file):
            os.remove(override_file)
        emit_event("install_dependencies", install_dependencies_start)

        env.pop("DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION", None)
      else:
        env["DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION"] = "1"

      env.pop("DATABRICKS_CLUSTER_ID", None)

      with tempfile.NamedTemporaryFile(mode='w+', dir=tmpdirname, delete=True) as databrickscfg_file:
          cli_args = shlex.split(args)
          custom_profiles = None
          if use_custom_profiles == "true":
            custom_profiles = _cli_runner_get_custom_profiles(
              workspace_url,
              str(getattr(ctx, "workspaceId", None) or ""),
              str(getattr(ctx, "workspaceHostName", None) or ""),
            )
          if custom_profiles:
            databrickscfg_file.write(custom_profiles)
            env["DATABRICKS_BUNDLE_STRICT_PROFILE_MATCH"] = "1"
            cli_command = [cli_exec_path] + cli_args
          else:
            profile_name = "workspace_profile"
            databrickscfg_file.write(f"[workspace_profile]\nhost = {workspace_url}")
            cli_command = [cli_exec_path] + cli_args + ["-p", profile_name]
          databrickscfg_file.flush()
          env["DATABRICKS_CONFIG_FILE"] = databrickscfg_file.name

          tmp_file_path = tmp_file

          run_cli_start = time.monotonic()
          run_cli_phase = "run_cli_genie" if cli_upstream == "genie-code" else "run_cli"
          try:
            _cli_runner_run_cli_command_with_token(
              cli_command,
              cwd,
              tmp_file_path,
              include_stderr_in_error == "true",
              token,
              env,
              cli_upstream=cli_upstream,
              capture_stdout=capture_stdout == "true",
            )
          finally:
            emit_event(run_cli_phase, run_cli_start)


try:
  _databricks_cli_runner()
finally:
  del (
    _databricks_cli_runner,
    _cli_runner_get_venv_path_from_databricks_yml,
    _cli_runner_get_custom_profiles,
    _cli_runner_get_download_url,
    _cli_runner_run_cli_command,
    _cli_runner_run_cli_command_with_token,
  )
None


In [0]:
def _cli_runner_get_venv_path_from_databricks_yml(cwd):
  import yaml

  databricks_yml_path = f"{cwd}/databricks.yml"
  with open(databricks_yml_path, "r") as f:
    data = yaml.safe_load(f)
    return data.get("python", {}).get("venv_path", ".venv")


def _cli_runner_get_custom_profiles(workspace_url, workspace_id, custom_url_host):
  import re

  if not re.fullmatch(r"[0-9]+", workspace_id):
    return None
  profiles = "[__settings__]\ndefault_profile = workspace_classic_id\n"
  profiles += f"\n[workspace_classic]\nhost = {workspace_url}\n"
  profiles += f"\n[workspace_classic_id]\nhost = {workspace_url}\nworkspace_id = {workspace_id}\n"
  custom_url_host = custom_url_host.lower()
  if re.fullmatch(r"[a-z0-9-]+(\.[a-z0-9-]+)+", custom_url_host):
    profiles += f"\n[workspace_custom_host]\nhost = https://{custom_url_host}\nworkspace_id = {workspace_id}\n"
  return profiles


def _cli_runner_run_cli_command(cli_command, cwd, tmp_file_path, include_stderr, env, *, cli_upstream, capture_stdout=False):
  import os
  import subprocess

  stderr_arg = subprocess.PIPE if include_stderr else None
  if tmp_file_path or capture_stdout:
    if tmp_file_path:
      os.makedirs(os.path.dirname(tmp_file_path), exist_ok=True)
    result = subprocess.run(cli_command, cwd=cwd, stdout=subprocess.PIPE, stderr=stderr_arg, text=True, env=env)
    if tmp_file_path:
      with open(tmp_file_path, 'w') as output_file:
        output_file.write(result.stdout)
    print(result.stdout)
  else:
    result = subprocess.run(cli_command, cwd=cwd, stderr=stderr_arg, text=True, env=env)
  if result.returncode != 0:
    error_msg = f"Command {result.args[:3]} failed with exit code {result.returncode}"
    if include_stderr and result.stderr:
      error_msg += f":\n{result.stderr}"
    raise RuntimeError(error_msg)

  if cli_upstream == "genie-code":
    print(result.stderr)

    try:
      with open("/Workspace/.proc/self/status", "r") as status_file:
        status_file.read()
    except Exception as e:
      print(f"Warning: Failed to read /Workspace/.proc/self/status: {e}")


def _cli_runner_run_cli_command_with_token(cli_command, cwd, tmp_file_path, include_stderr, token, env, *, cli_upstream, capture_stdout=False):
  _cli_runner_run_cli_command(
    cli_command,
    cwd,
    tmp_file_path,
    include_stderr,
    {**env, "DATABRICKS_TOKEN": token},
    cli_upstream=cli_upstream,
    capture_stdout=capture_stdout,
  )


def _cli_runner_get_download_url(workspace_url, token, file_api_path, runner_download_error_marker):
  import requests
  import time

  api_create_download_url = f"{workspace_url}/api/2.0/fs/create-download-url"
  headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
  data = {"path": file_api_path}

  max_attempts = 3
  last_error = None
  for attempt in range(max_attempts):
    try:
      response = requests.post(api_create_download_url, headers=headers, json=data)
      if response.status_code == 200:
        return response.json()["url"]
      last_error = f"HTTP {response.status_code} {response.text}"
    except requests.RequestException as e:
      last_error = str(e)
    if attempt < max_attempts - 1:
      time.sleep(2 ** attempt)

  print(f"{runner_download_error_marker}: {file_api_path} {last_error}")
  raise RuntimeError(f"Failed to download {file_api_path}: {last_error}")


def _databricks_cli_runner():
  import hashlib
  import os
  import platform
  import requests
  import shlex
  import shutil
  import subprocess
  import tempfile
  import time
  import zipfile
  from dbruntime.databricks_repl_context import get_context
  from urllib.parse import urlparse

  cli_version = "1.24.0"
  cwd = "/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1"
  args = "bundle debug list-targets -o json"
  tmp_file = "/Workspace/Users/deepaksinghal@cyntexa.com/.bundle/claims_lakehouse/.output/list-targets-output-uqyebpih0zs.json"
  capture_stdout = "true"
  dev_cli_snippet = ""
  artifact_download_error_prefix = "DABsRunnerError: Failed to download CLI dependency"
  runner_download_error_marker = "DabsRunnerErrorFallback: Downloading error, using fallback URL"
  allow_python_execution = "false"
  enable_yaml_sync = "true"
  enable_deployment_history = "true"
  include_stderr_in_error = "false"
  cli_upstream = "dabs-in-workspace"
  use_custom_profiles = "false"

  if os.environ.get("IS_SERVERLESS", "").strip().lower() != "true":
    raise RuntimeError("Databricks CLI runner must run on serverless compute")

  setup_runner_environment_start = time.monotonic()
  ctx = get_context()
  token = ctx.apiToken
  workspace_url = f"https://{urlparse(ctx.apiUrl).netloc}"
  cpu_arch = "linux_arm64" if platform.machine() == "aarch64" else "linux_amd64"

  env = os.environ.copy()

  def emit_event(phase, start_time):
    print(f"\n__RUNNER_PHASE__:{phase}:{(time.monotonic() - start_time) * 1000:.0f}", flush=True)

  def get_download_url(file_api_path):
    return _cli_runner_get_download_url(
      workspace_url, token, file_api_path, runner_download_error_marker
    )

  def download_and_extract(file_url, dest_dir):
    response = requests.get(file_url)
    if response.status_code != 200:
      raise RuntimeError(f"HTTP {response.status_code} while downloading artifact: {response.text}")
    tmp_file = f"{dest_dir}.zip"

    with open(tmp_file, "wb") as zip_file:
        zip_file.write(response.content)
    with zipfile.ZipFile(tmp_file, 'r') as zip_ref:
        zip_ref.extractall(dest_dir)

  def download_artifact_if_not_exists(input_file_name, binary_name, dest_path, tmp_dir):
    if os.path.exists(dest_path):
      return

    try:
      os.makedirs(os.path.dirname(dest_path), exist_ok=True)

      path_prefix = f"/DatabricksInternal/Jobs/ArtifactRepository/public/databricks-cli-binaries/{cli_version}"
      download_url = get_download_url(f"{path_prefix}/{input_file_name}.zip")

      tmp_archive_dir = f"{tmp_dir}/{input_file_name}"
      download_and_extract(download_url, tmp_archive_dir)

      src_path = f"{tmp_archive_dir}/{binary_name}"
      if not os.path.exists(src_path):
        available = ", ".join(os.listdir(tmp_archive_dir)) if os.path.isdir(tmp_archive_dir) else "(missing archive directory)"
        raise RuntimeError(f"Expected binary '{binary_name}' not found in archive contents: {available}")

      shutil.move(src_path, dest_path)
      os.chmod(dest_path, 0o755)
    except Exception as e:
      error_msg = f"{artifact_download_error_prefix} {input_file_name}/{binary_name} CLI v{cli_version}: {str(e)}"
      raise RuntimeError(f"{error_msg}")

  tmp_base = os.path.expanduser("~/tmp")
  if not os.path.exists(tmp_base):
    os.makedirs(tmp_base)
  os.chmod(tmp_base, 0o700)

  with tempfile.TemporaryDirectory(dir=tmp_base) as tmpdirname:
      emit_event("setup_runner_environment", setup_runner_environment_start)

      download_artifacts_start = time.monotonic()
      cli_exec_path = f"{tmp_base}/databricks_{cli_version}_{cpu_arch}"
      download_artifact_if_not_exists(
        f"databricks_cli_{cpu_arch}",
        "databricks",
        cli_exec_path,
        tmpdirname
      )
      emit_event("download_artifacts", download_artifacts_start)

      setup_cli_environment_start = time.monotonic()
      cwd = os.path.normpath(f"/Workspace{cwd}")

      if dev_cli_snippet:
        cli_path = cli_exec_path
        exec(dev_cli_snippet, globals(), locals())

      env["DATABRICKS_BUNDLE_TMP"] = f"{tmp_base}/bundle-2-{hashlib.sha256(cwd.encode()).hexdigest()}"
      env["DATABRICKS_CLI_UPSTREAM"] = cli_upstream

      if enable_yaml_sync == "true":
        env["DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC"] = "1"
      else:
        env.pop("DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC", None)

      if enable_deployment_history == "true":
        env["DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY"] = "true"
      else:
        env.pop("DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY", None)

      env["DATABRICKS_BUNDLE_ENGINE"] = "direct"

      emit_event("setup_cli_environment", setup_cli_environment_start)

      if allow_python_execution == "true":
        install_dependencies_start = time.monotonic()
        venv_path = _cli_runner_get_venv_path_from_databricks_yml(cwd)
        venv_folder_path = os.path.normpath(f"{cwd}/{venv_path}")
        venv_bin_path = f"{venv_folder_path}/bin"

        os.makedirs(venv_bin_path, exist_ok=True)

        python3_wrapper_path = f"{venv_bin_path}/python3"
        if not os.path.exists(python3_wrapper_path):
          with open(python3_wrapper_path, "w") as f:
            f.write('#!/usr/bin/env bash\nexec python3 "$@"\n')
          os.chmod(python3_wrapper_path, 0o755)

        pyvenv_cfg_path = f"{venv_folder_path}/pyvenv.cfg"
        if not os.path.exists(pyvenv_cfg_path):
          with open(pyvenv_cfg_path, "w") as f:
            pass

        pyproject_path = f"{cwd}/pyproject.toml"
        uv_pip_install_args = ["uv", "pip", "install", "--quiet", "--python", python3_wrapper_path, "--link-mode=copy"]

        override_file = None
        if os.path.exists(pyproject_path):
          uv_pip_install_args.extend(["-r", pyproject_path, "--group", "dev"])
        else:
          uv_pip_install_args.append(f"databricks-bundles=={cli_version}")

        try:
          subprocess.run(uv_pip_install_args, check=True, cwd=cwd, stdout=subprocess.DEVNULL, env=env)
        finally:
          if override_file and os.path.exists(override_file):
            os.remove(override_file)
        emit_event("install_dependencies", install_dependencies_start)

        env.pop("DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION", None)
      else:
        env["DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION"] = "1"

      env.pop("DATABRICKS_CLUSTER_ID", None)

      with tempfile.NamedTemporaryFile(mode='w+', dir=tmpdirname, delete=True) as databrickscfg_file:
          cli_args = shlex.split(args)
          custom_profiles = None
          if use_custom_profiles == "true":
            custom_profiles = _cli_runner_get_custom_profiles(
              workspace_url,
              str(getattr(ctx, "workspaceId", None) or ""),
              str(getattr(ctx, "workspaceHostName", None) or ""),
            )
          if custom_profiles:
            databrickscfg_file.write(custom_profiles)
            env["DATABRICKS_BUNDLE_STRICT_PROFILE_MATCH"] = "1"
            cli_command = [cli_exec_path] + cli_args
          else:
            profile_name = "workspace_profile"
            databrickscfg_file.write(f"[workspace_profile]\nhost = {workspace_url}")
            cli_command = [cli_exec_path] + cli_args + ["-p", profile_name]
          databrickscfg_file.flush()
          env["DATABRICKS_CONFIG_FILE"] = databrickscfg_file.name

          tmp_file_path = tmp_file

          run_cli_start = time.monotonic()
          run_cli_phase = "run_cli_genie" if cli_upstream == "genie-code" else "run_cli"
          try:
            _cli_runner_run_cli_command_with_token(
              cli_command,
              cwd,
              tmp_file_path,
              include_stderr_in_error == "true",
              token,
              env,
              cli_upstream=cli_upstream,
              capture_stdout=capture_stdout == "true",
            )
          finally:
            emit_event(run_cli_phase, run_cli_start)


try:
  _databricks_cli_runner()
finally:
  del (
    _databricks_cli_runner,
    _cli_runner_get_venv_path_from_databricks_yml,
    _cli_runner_get_custom_profiles,
    _cli_runner_get_download_url,
    _cli_runner_run_cli_command,
    _cli_runner_run_cli_command_with_token,
  )
None


In [0]:
def _cli_runner_get_venv_path_from_databricks_yml(cwd):
  import yaml

  databricks_yml_path = f"{cwd}/databricks.yml"
  with open(databricks_yml_path, "r") as f:
    data = yaml.safe_load(f)
    return data.get("python", {}).get("venv_path", ".venv")


def _cli_runner_get_custom_profiles(workspace_url, workspace_id, custom_url_host):
  import re

  if not re.fullmatch(r"[0-9]+", workspace_id):
    return None
  profiles = "[__settings__]\ndefault_profile = workspace_classic_id\n"
  profiles += f"\n[workspace_classic]\nhost = {workspace_url}\n"
  profiles += f"\n[workspace_classic_id]\nhost = {workspace_url}\nworkspace_id = {workspace_id}\n"
  custom_url_host = custom_url_host.lower()
  if re.fullmatch(r"[a-z0-9-]+(\.[a-z0-9-]+)+", custom_url_host):
    profiles += f"\n[workspace_custom_host]\nhost = https://{custom_url_host}\nworkspace_id = {workspace_id}\n"
  return profiles


def _cli_runner_run_cli_command(cli_command, cwd, tmp_file_path, include_stderr, env, *, cli_upstream, capture_stdout=False):
  import os
  import subprocess

  stderr_arg = subprocess.PIPE if include_stderr else None
  if tmp_file_path or capture_stdout:
    if tmp_file_path:
      os.makedirs(os.path.dirname(tmp_file_path), exist_ok=True)
    result = subprocess.run(cli_command, cwd=cwd, stdout=subprocess.PIPE, stderr=stderr_arg, text=True, env=env)
    if tmp_file_path:
      with open(tmp_file_path, 'w') as output_file:
        output_file.write(result.stdout)
    print(result.stdout)
  else:
    result = subprocess.run(cli_command, cwd=cwd, stderr=stderr_arg, text=True, env=env)
  if result.returncode != 0:
    error_msg = f"Command {result.args[:3]} failed with exit code {result.returncode}"
    if include_stderr and result.stderr:
      error_msg += f":\n{result.stderr}"
    raise RuntimeError(error_msg)

  if cli_upstream == "genie-code":
    print(result.stderr)

    try:
      with open("/Workspace/.proc/self/status", "r") as status_file:
        status_file.read()
    except Exception as e:
      print(f"Warning: Failed to read /Workspace/.proc/self/status: {e}")


def _cli_runner_run_cli_command_with_token(cli_command, cwd, tmp_file_path, include_stderr, token, env, *, cli_upstream, capture_stdout=False):
  _cli_runner_run_cli_command(
    cli_command,
    cwd,
    tmp_file_path,
    include_stderr,
    {**env, "DATABRICKS_TOKEN": token},
    cli_upstream=cli_upstream,
    capture_stdout=capture_stdout,
  )


def _cli_runner_get_download_url(workspace_url, token, file_api_path, runner_download_error_marker):
  import requests
  import time

  api_create_download_url = f"{workspace_url}/api/2.0/fs/create-download-url"
  headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
  data = {"path": file_api_path}

  max_attempts = 3
  last_error = None
  for attempt in range(max_attempts):
    try:
      response = requests.post(api_create_download_url, headers=headers, json=data)
      if response.status_code == 200:
        return response.json()["url"]
      last_error = f"HTTP {response.status_code} {response.text}"
    except requests.RequestException as e:
      last_error = str(e)
    if attempt < max_attempts - 1:
      time.sleep(2 ** attempt)

  print(f"{runner_download_error_marker}: {file_api_path} {last_error}")
  raise RuntimeError(f"Failed to download {file_api_path}: {last_error}")


def _databricks_cli_runner():
  import hashlib
  import os
  import platform
  import requests
  import shlex
  import shutil
  import subprocess
  import tempfile
  import time
  import zipfile
  from dbruntime.databricks_repl_context import get_context
  from urllib.parse import urlparse

  cli_version = "1.24.0"
  cwd = "/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1"
  args = "bundle summary -o json --include-locations --target dev"
  tmp_file = "/Workspace/Users/deepaksinghal@cyntexa.com/.bundle/claims_lakehouse/.output/async-summary-output-33f1y5gffd.json"
  capture_stdout = "true"
  dev_cli_snippet = ""
  artifact_download_error_prefix = "DABsRunnerError: Failed to download CLI dependency"
  runner_download_error_marker = "DabsRunnerErrorFallback: Downloading error, using fallback URL"
  allow_python_execution = "false"
  enable_yaml_sync = "true"
  enable_deployment_history = "true"
  include_stderr_in_error = "false"
  cli_upstream = "dabs-in-workspace"
  use_custom_profiles = "false"

  if os.environ.get("IS_SERVERLESS", "").strip().lower() != "true":
    raise RuntimeError("Databricks CLI runner must run on serverless compute")

  setup_runner_environment_start = time.monotonic()
  ctx = get_context()
  token = ctx.apiToken
  workspace_url = f"https://{urlparse(ctx.apiUrl).netloc}"
  cpu_arch = "linux_arm64" if platform.machine() == "aarch64" else "linux_amd64"

  env = os.environ.copy()

  def emit_event(phase, start_time):
    print(f"\n__RUNNER_PHASE__:{phase}:{(time.monotonic() - start_time) * 1000:.0f}", flush=True)

  def get_download_url(file_api_path):
    return _cli_runner_get_download_url(
      workspace_url, token, file_api_path, runner_download_error_marker
    )

  def download_and_extract(file_url, dest_dir):
    response = requests.get(file_url)
    if response.status_code != 200:
      raise RuntimeError(f"HTTP {response.status_code} while downloading artifact: {response.text}")
    tmp_file = f"{dest_dir}.zip"

    with open(tmp_file, "wb") as zip_file:
        zip_file.write(response.content)
    with zipfile.ZipFile(tmp_file, 'r') as zip_ref:
        zip_ref.extractall(dest_dir)

  def download_artifact_if_not_exists(input_file_name, binary_name, dest_path, tmp_dir):
    if os.path.exists(dest_path):
      return

    try:
      os.makedirs(os.path.dirname(dest_path), exist_ok=True)

      path_prefix = f"/DatabricksInternal/Jobs/ArtifactRepository/public/databricks-cli-binaries/{cli_version}"
      download_url = get_download_url(f"{path_prefix}/{input_file_name}.zip")

      tmp_archive_dir = f"{tmp_dir}/{input_file_name}"
      download_and_extract(download_url, tmp_archive_dir)

      src_path = f"{tmp_archive_dir}/{binary_name}"
      if not os.path.exists(src_path):
        available = ", ".join(os.listdir(tmp_archive_dir)) if os.path.isdir(tmp_archive_dir) else "(missing archive directory)"
        raise RuntimeError(f"Expected binary '{binary_name}' not found in archive contents: {available}")

      shutil.move(src_path, dest_path)
      os.chmod(dest_path, 0o755)
    except Exception as e:
      error_msg = f"{artifact_download_error_prefix} {input_file_name}/{binary_name} CLI v{cli_version}: {str(e)}"
      raise RuntimeError(f"{error_msg}")

  tmp_base = os.path.expanduser("~/tmp")
  if not os.path.exists(tmp_base):
    os.makedirs(tmp_base)
  os.chmod(tmp_base, 0o700)

  with tempfile.TemporaryDirectory(dir=tmp_base) as tmpdirname:
      emit_event("setup_runner_environment", setup_runner_environment_start)

      download_artifacts_start = time.monotonic()
      cli_exec_path = f"{tmp_base}/databricks_{cli_version}_{cpu_arch}"
      download_artifact_if_not_exists(
        f"databricks_cli_{cpu_arch}",
        "databricks",
        cli_exec_path,
        tmpdirname
      )
      emit_event("download_artifacts", download_artifacts_start)

      setup_cli_environment_start = time.monotonic()
      cwd = os.path.normpath(f"/Workspace{cwd}")

      if dev_cli_snippet:
        cli_path = cli_exec_path
        exec(dev_cli_snippet, globals(), locals())

      env["DATABRICKS_BUNDLE_TMP"] = f"{tmp_base}/bundle-2-{hashlib.sha256(cwd.encode()).hexdigest()}"
      env["DATABRICKS_CLI_UPSTREAM"] = cli_upstream

      if enable_yaml_sync == "true":
        env["DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC"] = "1"
      else:
        env.pop("DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC", None)

      if enable_deployment_history == "true":
        env["DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY"] = "true"
      else:
        env.pop("DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY", None)

      env["DATABRICKS_BUNDLE_ENGINE"] = "direct"

      emit_event("setup_cli_environment", setup_cli_environment_start)

      if allow_python_execution == "true":
        install_dependencies_start = time.monotonic()
        venv_path = _cli_runner_get_venv_path_from_databricks_yml(cwd)
        venv_folder_path = os.path.normpath(f"{cwd}/{venv_path}")
        venv_bin_path = f"{venv_folder_path}/bin"

        os.makedirs(venv_bin_path, exist_ok=True)

        python3_wrapper_path = f"{venv_bin_path}/python3"
        if not os.path.exists(python3_wrapper_path):
          with open(python3_wrapper_path, "w") as f:
            f.write('#!/usr/bin/env bash\nexec python3 "$@"\n')
          os.chmod(python3_wrapper_path, 0o755)

        pyvenv_cfg_path = f"{venv_folder_path}/pyvenv.cfg"
        if not os.path.exists(pyvenv_cfg_path):
          with open(pyvenv_cfg_path, "w") as f:
            pass

        pyproject_path = f"{cwd}/pyproject.toml"
        uv_pip_install_args = ["uv", "pip", "install", "--quiet", "--python", python3_wrapper_path, "--link-mode=copy"]

        override_file = None
        if os.path.exists(pyproject_path):
          uv_pip_install_args.extend(["-r", pyproject_path, "--group", "dev"])
        else:
          uv_pip_install_args.append(f"databricks-bundles=={cli_version}")

        try:
          subprocess.run(uv_pip_install_args, check=True, cwd=cwd, stdout=subprocess.DEVNULL, env=env)
        finally:
          if override_file and os.path.exists(override_file):
            os.remove(override_file)
        emit_event("install_dependencies", install_dependencies_start)

        env.pop("DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION", None)
      else:
        env["DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION"] = "1"

      env.pop("DATABRICKS_CLUSTER_ID", None)

      with tempfile.NamedTemporaryFile(mode='w+', dir=tmpdirname, delete=True) as databrickscfg_file:
          cli_args = shlex.split(args)
          custom_profiles = None
          if use_custom_profiles == "true":
            custom_profiles = _cli_runner_get_custom_profiles(
              workspace_url,
              str(getattr(ctx, "workspaceId", None) or ""),
              str(getattr(ctx, "workspaceHostName", None) or ""),
            )
          if custom_profiles:
            databrickscfg_file.write(custom_profiles)
            env["DATABRICKS_BUNDLE_STRICT_PROFILE_MATCH"] = "1"
            cli_command = [cli_exec_path] + cli_args
          else:
            profile_name = "workspace_profile"
            databrickscfg_file.write(f"[workspace_profile]\nhost = {workspace_url}")
            cli_command = [cli_exec_path] + cli_args + ["-p", profile_name]
          databrickscfg_file.flush()
          env["DATABRICKS_CONFIG_FILE"] = databrickscfg_file.name

          tmp_file_path = tmp_file

          run_cli_start = time.monotonic()
          run_cli_phase = "run_cli_genie" if cli_upstream == "genie-code" else "run_cli"
          try:
            _cli_runner_run_cli_command_with_token(
              cli_command,
              cwd,
              tmp_file_path,
              include_stderr_in_error == "true",
              token,
              env,
              cli_upstream=cli_upstream,
              capture_stdout=capture_stdout == "true",
            )
          finally:
            emit_event(run_cli_phase, run_cli_start)


try:
  _databricks_cli_runner()
finally:
  del (
    _databricks_cli_runner,
    _cli_runner_get_venv_path_from_databricks_yml,
    _cli_runner_get_custom_profiles,
    _cli_runner_get_download_url,
    _cli_runner_run_cli_command,
    _cli_runner_run_cli_command_with_token,
  )
None


In [0]:
def _cli_runner_get_venv_path_from_databricks_yml(cwd):
  import yaml

  databricks_yml_path = f"{cwd}/databricks.yml"
  with open(databricks_yml_path, "r") as f:
    data = yaml.safe_load(f)
    return data.get("python", {}).get("venv_path", ".venv")


def _cli_runner_get_custom_profiles(workspace_url, workspace_id, custom_url_host):
  import re

  if not re.fullmatch(r"[0-9]+", workspace_id):
    return None
  profiles = "[__settings__]\ndefault_profile = workspace_classic_id\n"
  profiles += f"\n[workspace_classic]\nhost = {workspace_url}\n"
  profiles += f"\n[workspace_classic_id]\nhost = {workspace_url}\nworkspace_id = {workspace_id}\n"
  custom_url_host = custom_url_host.lower()
  if re.fullmatch(r"[a-z0-9-]+(\.[a-z0-9-]+)+", custom_url_host):
    profiles += f"\n[workspace_custom_host]\nhost = https://{custom_url_host}\nworkspace_id = {workspace_id}\n"
  return profiles


def _cli_runner_run_cli_command(cli_command, cwd, tmp_file_path, include_stderr, env, *, cli_upstream, capture_stdout=False):
  import os
  import subprocess

  stderr_arg = subprocess.PIPE if include_stderr else None
  if tmp_file_path or capture_stdout:
    if tmp_file_path:
      os.makedirs(os.path.dirname(tmp_file_path), exist_ok=True)
    result = subprocess.run(cli_command, cwd=cwd, stdout=subprocess.PIPE, stderr=stderr_arg, text=True, env=env)
    if tmp_file_path:
      with open(tmp_file_path, 'w') as output_file:
        output_file.write(result.stdout)
    print(result.stdout)
  else:
    result = subprocess.run(cli_command, cwd=cwd, stderr=stderr_arg, text=True, env=env)
  if result.returncode != 0:
    error_msg = f"Command {result.args[:3]} failed with exit code {result.returncode}"
    if include_stderr and result.stderr:
      error_msg += f":\n{result.stderr}"
    raise RuntimeError(error_msg)

  if cli_upstream == "genie-code":
    print(result.stderr)

    try:
      with open("/Workspace/.proc/self/status", "r") as status_file:
        status_file.read()
    except Exception as e:
      print(f"Warning: Failed to read /Workspace/.proc/self/status: {e}")


def _cli_runner_run_cli_command_with_token(cli_command, cwd, tmp_file_path, include_stderr, token, env, *, cli_upstream, capture_stdout=False):
  _cli_runner_run_cli_command(
    cli_command,
    cwd,
    tmp_file_path,
    include_stderr,
    {**env, "DATABRICKS_TOKEN": token},
    cli_upstream=cli_upstream,
    capture_stdout=capture_stdout,
  )


def _cli_runner_get_download_url(workspace_url, token, file_api_path, runner_download_error_marker):
  import requests
  import time

  api_create_download_url = f"{workspace_url}/api/2.0/fs/create-download-url"
  headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
  data = {"path": file_api_path}

  max_attempts = 3
  last_error = None
  for attempt in range(max_attempts):
    try:
      response = requests.post(api_create_download_url, headers=headers, json=data)
      if response.status_code == 200:
        return response.json()["url"]
      last_error = f"HTTP {response.status_code} {response.text}"
    except requests.RequestException as e:
      last_error = str(e)
    if attempt < max_attempts - 1:
      time.sleep(2 ** attempt)

  print(f"{runner_download_error_marker}: {file_api_path} {last_error}")
  raise RuntimeError(f"Failed to download {file_api_path}: {last_error}")


def _databricks_cli_runner():
  import hashlib
  import os
  import platform
  import requests
  import shlex
  import shutil
  import subprocess
  import tempfile
  import time
  import zipfile
  from dbruntime.databricks_repl_context import get_context
  from urllib.parse import urlparse

  cli_version = "1.24.0"
  cwd = "/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1"
  args = "bundle deploy --target dev"
  tmp_file = ""
  capture_stdout = "false"
  dev_cli_snippet = ""
  artifact_download_error_prefix = "DABsRunnerError: Failed to download CLI dependency"
  runner_download_error_marker = "DabsRunnerErrorFallback: Downloading error, using fallback URL"
  allow_python_execution = "false"
  enable_yaml_sync = "true"
  enable_deployment_history = "true"
  include_stderr_in_error = "false"
  cli_upstream = "dabs-in-workspace"
  use_custom_profiles = "false"

  if os.environ.get("IS_SERVERLESS", "").strip().lower() != "true":
    raise RuntimeError("Databricks CLI runner must run on serverless compute")

  setup_runner_environment_start = time.monotonic()
  ctx = get_context()
  token = ctx.apiToken
  workspace_url = f"https://{urlparse(ctx.apiUrl).netloc}"
  cpu_arch = "linux_arm64" if platform.machine() == "aarch64" else "linux_amd64"

  env = os.environ.copy()

  def emit_event(phase, start_time):
    print(f"\n__RUNNER_PHASE__:{phase}:{(time.monotonic() - start_time) * 1000:.0f}", flush=True)

  def get_download_url(file_api_path):
    return _cli_runner_get_download_url(
      workspace_url, token, file_api_path, runner_download_error_marker
    )

  def download_and_extract(file_url, dest_dir):
    response = requests.get(file_url)
    if response.status_code != 200:
      raise RuntimeError(f"HTTP {response.status_code} while downloading artifact: {response.text}")
    tmp_file = f"{dest_dir}.zip"

    with open(tmp_file, "wb") as zip_file:
        zip_file.write(response.content)
    with zipfile.ZipFile(tmp_file, 'r') as zip_ref:
        zip_ref.extractall(dest_dir)

  def download_artifact_if_not_exists(input_file_name, binary_name, dest_path, tmp_dir):
    if os.path.exists(dest_path):
      return

    try:
      os.makedirs(os.path.dirname(dest_path), exist_ok=True)

      path_prefix = f"/DatabricksInternal/Jobs/ArtifactRepository/public/databricks-cli-binaries/{cli_version}"
      download_url = get_download_url(f"{path_prefix}/{input_file_name}.zip")

      tmp_archive_dir = f"{tmp_dir}/{input_file_name}"
      download_and_extract(download_url, tmp_archive_dir)

      src_path = f"{tmp_archive_dir}/{binary_name}"
      if not os.path.exists(src_path):
        available = ", ".join(os.listdir(tmp_archive_dir)) if os.path.isdir(tmp_archive_dir) else "(missing archive directory)"
        raise RuntimeError(f"Expected binary '{binary_name}' not found in archive contents: {available}")

      shutil.move(src_path, dest_path)
      os.chmod(dest_path, 0o755)
    except Exception as e:
      error_msg = f"{artifact_download_error_prefix} {input_file_name}/{binary_name} CLI v{cli_version}: {str(e)}"
      raise RuntimeError(f"{error_msg}")

  tmp_base = os.path.expanduser("~/tmp")
  if not os.path.exists(tmp_base):
    os.makedirs(tmp_base)
  os.chmod(tmp_base, 0o700)

  with tempfile.TemporaryDirectory(dir=tmp_base) as tmpdirname:
      emit_event("setup_runner_environment", setup_runner_environment_start)

      download_artifacts_start = time.monotonic()
      cli_exec_path = f"{tmp_base}/databricks_{cli_version}_{cpu_arch}"
      download_artifact_if_not_exists(
        f"databricks_cli_{cpu_arch}",
        "databricks",
        cli_exec_path,
        tmpdirname
      )
      emit_event("download_artifacts", download_artifacts_start)

      setup_cli_environment_start = time.monotonic()
      cwd = os.path.normpath(f"/Workspace{cwd}")

      if dev_cli_snippet:
        cli_path = cli_exec_path
        exec(dev_cli_snippet, globals(), locals())

      env["DATABRICKS_BUNDLE_TMP"] = f"{tmp_base}/bundle-2-{hashlib.sha256(cwd.encode()).hexdigest()}"
      env["DATABRICKS_CLI_UPSTREAM"] = cli_upstream

      if enable_yaml_sync == "true":
        env["DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC"] = "1"
      else:
        env.pop("DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC", None)

      if enable_deployment_history == "true":
        env["DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY"] = "true"
      else:
        env.pop("DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY", None)

      env["DATABRICKS_BUNDLE_ENGINE"] = "direct"

      emit_event("setup_cli_environment", setup_cli_environment_start)

      if allow_python_execution == "true":
        install_dependencies_start = time.monotonic()
        venv_path = _cli_runner_get_venv_path_from_databricks_yml(cwd)
        venv_folder_path = os.path.normpath(f"{cwd}/{venv_path}")
        venv_bin_path = f"{venv_folder_path}/bin"

        os.makedirs(venv_bin_path, exist_ok=True)

        python3_wrapper_path = f"{venv_bin_path}/python3"
        if not os.path.exists(python3_wrapper_path):
          with open(python3_wrapper_path, "w") as f:
            f.write('#!/usr/bin/env bash\nexec python3 "$@"\n')
          os.chmod(python3_wrapper_path, 0o755)

        pyvenv_cfg_path = f"{venv_folder_path}/pyvenv.cfg"
        if not os.path.exists(pyvenv_cfg_path):
          with open(pyvenv_cfg_path, "w") as f:
            pass

        pyproject_path = f"{cwd}/pyproject.toml"
        uv_pip_install_args = ["uv", "pip", "install", "--quiet", "--python", python3_wrapper_path, "--link-mode=copy"]

        override_file = None
        if os.path.exists(pyproject_path):
          uv_pip_install_args.extend(["-r", pyproject_path, "--group", "dev"])
        else:
          uv_pip_install_args.append(f"databricks-bundles=={cli_version}")

        try:
          subprocess.run(uv_pip_install_args, check=True, cwd=cwd, stdout=subprocess.DEVNULL, env=env)
        finally:
          if override_file and os.path.exists(override_file):
            os.remove(override_file)
        emit_event("install_dependencies", install_dependencies_start)

        env.pop("DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION", None)
      else:
        env["DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION"] = "1"

      env.pop("DATABRICKS_CLUSTER_ID", None)

      with tempfile.NamedTemporaryFile(mode='w+', dir=tmpdirname, delete=True) as databrickscfg_file:
          cli_args = shlex.split(args)
          custom_profiles = None
          if use_custom_profiles == "true":
            custom_profiles = _cli_runner_get_custom_profiles(
              workspace_url,
              str(getattr(ctx, "workspaceId", None) or ""),
              str(getattr(ctx, "workspaceHostName", None) or ""),
            )
          if custom_profiles:
            databrickscfg_file.write(custom_profiles)
            env["DATABRICKS_BUNDLE_STRICT_PROFILE_MATCH"] = "1"
            cli_command = [cli_exec_path] + cli_args
          else:
            profile_name = "workspace_profile"
            databrickscfg_file.write(f"[workspace_profile]\nhost = {workspace_url}")
            cli_command = [cli_exec_path] + cli_args + ["-p", profile_name]
          databrickscfg_file.flush()
          env["DATABRICKS_CONFIG_FILE"] = databrickscfg_file.name

          tmp_file_path = tmp_file

          run_cli_start = time.monotonic()
          run_cli_phase = "run_cli_genie" if cli_upstream == "genie-code" else "run_cli"
          try:
            _cli_runner_run_cli_command_with_token(
              cli_command,
              cwd,
              tmp_file_path,
              include_stderr_in_error == "true",
              token,
              env,
              cli_upstream=cli_upstream,
              capture_stdout=capture_stdout == "true",
            )
          finally:
            emit_event(run_cli_phase, run_cli_start)


try:
  _databricks_cli_runner()
finally:
  del (
    _databricks_cli_runner,
    _cli_runner_get_venv_path_from_databricks_yml,
    _cli_runner_get_custom_profiles,
    _cli_runner_get_download_url,
    _cli_runner_run_cli_command,
    _cli_runner_run_cli_command_with_token,
  )
None


In [0]:
def _cli_runner_get_venv_path_from_databricks_yml(cwd):
  import yaml

  databricks_yml_path = f"{cwd}/databricks.yml"
  with open(databricks_yml_path, "r") as f:
    data = yaml.safe_load(f)
    return data.get("python", {}).get("venv_path", ".venv")


def _cli_runner_get_custom_profiles(workspace_url, workspace_id, custom_url_host):
  import re

  if not re.fullmatch(r"[0-9]+", workspace_id):
    return None
  profiles = "[__settings__]\ndefault_profile = workspace_classic_id\n"
  profiles += f"\n[workspace_classic]\nhost = {workspace_url}\n"
  profiles += f"\n[workspace_classic_id]\nhost = {workspace_url}\nworkspace_id = {workspace_id}\n"
  custom_url_host = custom_url_host.lower()
  if re.fullmatch(r"[a-z0-9-]+(\.[a-z0-9-]+)+", custom_url_host):
    profiles += f"\n[workspace_custom_host]\nhost = https://{custom_url_host}\nworkspace_id = {workspace_id}\n"
  return profiles


def _cli_runner_run_cli_command(cli_command, cwd, tmp_file_path, include_stderr, env, *, cli_upstream, capture_stdout=False):
  import os
  import subprocess

  stderr_arg = subprocess.PIPE if include_stderr else None
  if tmp_file_path or capture_stdout:
    if tmp_file_path:
      os.makedirs(os.path.dirname(tmp_file_path), exist_ok=True)
    result = subprocess.run(cli_command, cwd=cwd, stdout=subprocess.PIPE, stderr=stderr_arg, text=True, env=env)
    if tmp_file_path:
      with open(tmp_file_path, 'w') as output_file:
        output_file.write(result.stdout)
    print(result.stdout)
  else:
    result = subprocess.run(cli_command, cwd=cwd, stderr=stderr_arg, text=True, env=env)
  if result.returncode != 0:
    error_msg = f"Command {result.args[:3]} failed with exit code {result.returncode}"
    if include_stderr and result.stderr:
      error_msg += f":\n{result.stderr}"
    raise RuntimeError(error_msg)

  if cli_upstream == "genie-code":
    print(result.stderr)

    try:
      with open("/Workspace/.proc/self/status", "r") as status_file:
        status_file.read()
    except Exception as e:
      print(f"Warning: Failed to read /Workspace/.proc/self/status: {e}")


def _cli_runner_run_cli_command_with_token(cli_command, cwd, tmp_file_path, include_stderr, token, env, *, cli_upstream, capture_stdout=False):
  _cli_runner_run_cli_command(
    cli_command,
    cwd,
    tmp_file_path,
    include_stderr,
    {**env, "DATABRICKS_TOKEN": token},
    cli_upstream=cli_upstream,
    capture_stdout=capture_stdout,
  )


def _cli_runner_get_download_url(workspace_url, token, file_api_path, runner_download_error_marker):
  import requests
  import time

  api_create_download_url = f"{workspace_url}/api/2.0/fs/create-download-url"
  headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
  data = {"path": file_api_path}

  max_attempts = 3
  last_error = None
  for attempt in range(max_attempts):
    try:
      response = requests.post(api_create_download_url, headers=headers, json=data)
      if response.status_code == 200:
        return response.json()["url"]
      last_error = f"HTTP {response.status_code} {response.text}"
    except requests.RequestException as e:
      last_error = str(e)
    if attempt < max_attempts - 1:
      time.sleep(2 ** attempt)

  print(f"{runner_download_error_marker}: {file_api_path} {last_error}")
  raise RuntimeError(f"Failed to download {file_api_path}: {last_error}")


def _databricks_cli_runner():
  import hashlib
  import os
  import platform
  import requests
  import shlex
  import shutil
  import subprocess
  import tempfile
  import time
  import zipfile
  from dbruntime.databricks_repl_context import get_context
  from urllib.parse import urlparse

  cli_version = "1.24.0"
  cwd = "/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1"
  args = "bundle summary -o json --include-locations --target dev"
  tmp_file = "/Workspace/Users/deepaksinghal@cyntexa.com/.bundle/claims_lakehouse/.output/async-summary-output-8d0w8wmud28.json"
  capture_stdout = "true"
  dev_cli_snippet = ""
  artifact_download_error_prefix = "DABsRunnerError: Failed to download CLI dependency"
  runner_download_error_marker = "DabsRunnerErrorFallback: Downloading error, using fallback URL"
  allow_python_execution = "false"
  enable_yaml_sync = "true"
  enable_deployment_history = "true"
  include_stderr_in_error = "false"
  cli_upstream = "dabs-in-workspace"
  use_custom_profiles = "false"

  if os.environ.get("IS_SERVERLESS", "").strip().lower() != "true":
    raise RuntimeError("Databricks CLI runner must run on serverless compute")

  setup_runner_environment_start = time.monotonic()
  ctx = get_context()
  token = ctx.apiToken
  workspace_url = f"https://{urlparse(ctx.apiUrl).netloc}"
  cpu_arch = "linux_arm64" if platform.machine() == "aarch64" else "linux_amd64"

  env = os.environ.copy()

  def emit_event(phase, start_time):
    print(f"\n__RUNNER_PHASE__:{phase}:{(time.monotonic() - start_time) * 1000:.0f}", flush=True)

  def get_download_url(file_api_path):
    return _cli_runner_get_download_url(
      workspace_url, token, file_api_path, runner_download_error_marker
    )

  def download_and_extract(file_url, dest_dir):
    response = requests.get(file_url)
    if response.status_code != 200:
      raise RuntimeError(f"HTTP {response.status_code} while downloading artifact: {response.text}")
    tmp_file = f"{dest_dir}.zip"

    with open(tmp_file, "wb") as zip_file:
        zip_file.write(response.content)
    with zipfile.ZipFile(tmp_file, 'r') as zip_ref:
        zip_ref.extractall(dest_dir)

  def download_artifact_if_not_exists(input_file_name, binary_name, dest_path, tmp_dir):
    if os.path.exists(dest_path):
      return

    try:
      os.makedirs(os.path.dirname(dest_path), exist_ok=True)

      path_prefix = f"/DatabricksInternal/Jobs/ArtifactRepository/public/databricks-cli-binaries/{cli_version}"
      download_url = get_download_url(f"{path_prefix}/{input_file_name}.zip")

      tmp_archive_dir = f"{tmp_dir}/{input_file_name}"
      download_and_extract(download_url, tmp_archive_dir)

      src_path = f"{tmp_archive_dir}/{binary_name}"
      if not os.path.exists(src_path):
        available = ", ".join(os.listdir(tmp_archive_dir)) if os.path.isdir(tmp_archive_dir) else "(missing archive directory)"
        raise RuntimeError(f"Expected binary '{binary_name}' not found in archive contents: {available}")

      shutil.move(src_path, dest_path)
      os.chmod(dest_path, 0o755)
    except Exception as e:
      error_msg = f"{artifact_download_error_prefix} {input_file_name}/{binary_name} CLI v{cli_version}: {str(e)}"
      raise RuntimeError(f"{error_msg}")

  tmp_base = os.path.expanduser("~/tmp")
  if not os.path.exists(tmp_base):
    os.makedirs(tmp_base)
  os.chmod(tmp_base, 0o700)

  with tempfile.TemporaryDirectory(dir=tmp_base) as tmpdirname:
      emit_event("setup_runner_environment", setup_runner_environment_start)

      download_artifacts_start = time.monotonic()
      cli_exec_path = f"{tmp_base}/databricks_{cli_version}_{cpu_arch}"
      download_artifact_if_not_exists(
        f"databricks_cli_{cpu_arch}",
        "databricks",
        cli_exec_path,
        tmpdirname
      )
      emit_event("download_artifacts", download_artifacts_start)

      setup_cli_environment_start = time.monotonic()
      cwd = os.path.normpath(f"/Workspace{cwd}")

      if dev_cli_snippet:
        cli_path = cli_exec_path
        exec(dev_cli_snippet, globals(), locals())

      env["DATABRICKS_BUNDLE_TMP"] = f"{tmp_base}/bundle-2-{hashlib.sha256(cwd.encode()).hexdigest()}"
      env["DATABRICKS_CLI_UPSTREAM"] = cli_upstream

      if enable_yaml_sync == "true":
        env["DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC"] = "1"
      else:
        env.pop("DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC", None)

      if enable_deployment_history == "true":
        env["DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY"] = "true"
      else:
        env.pop("DATABRICKS_BUNDLE_DEPLOYMENT_HISTORY", None)

      env["DATABRICKS_BUNDLE_ENGINE"] = "direct"

      emit_event("setup_cli_environment", setup_cli_environment_start)

      if allow_python_execution == "true":
        install_dependencies_start = time.monotonic()
        venv_path = _cli_runner_get_venv_path_from_databricks_yml(cwd)
        venv_folder_path = os.path.normpath(f"{cwd}/{venv_path}")
        venv_bin_path = f"{venv_folder_path}/bin"

        os.makedirs(venv_bin_path, exist_ok=True)

        python3_wrapper_path = f"{venv_bin_path}/python3"
        if not os.path.exists(python3_wrapper_path):
          with open(python3_wrapper_path, "w") as f:
            f.write('#!/usr/bin/env bash\nexec python3 "$@"\n')
          os.chmod(python3_wrapper_path, 0o755)

        pyvenv_cfg_path = f"{venv_folder_path}/pyvenv.cfg"
        if not os.path.exists(pyvenv_cfg_path):
          with open(pyvenv_cfg_path, "w") as f:
            pass

        pyproject_path = f"{cwd}/pyproject.toml"
        uv_pip_install_args = ["uv", "pip", "install", "--quiet", "--python", python3_wrapper_path, "--link-mode=copy"]

        override_file = None
        if os.path.exists(pyproject_path):
          uv_pip_install_args.extend(["-r", pyproject_path, "--group", "dev"])
        else:
          uv_pip_install_args.append(f"databricks-bundles=={cli_version}")

        try:
          subprocess.run(uv_pip_install_args, check=True, cwd=cwd, stdout=subprocess.DEVNULL, env=env)
        finally:
          if override_file and os.path.exists(override_file):
            os.remove(override_file)
        emit_event("install_dependencies", install_dependencies_start)

        env.pop("DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION", None)
      else:
        env["DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION"] = "1"

      env.pop("DATABRICKS_CLUSTER_ID", None)

      with tempfile.NamedTemporaryFile(mode='w+', dir=tmpdirname, delete=True) as databrickscfg_file:
          cli_args = shlex.split(args)
          custom_profiles = None
          if use_custom_profiles == "true":
            custom_profiles = _cli_runner_get_custom_profiles(
              workspace_url,
              str(getattr(ctx, "workspaceId", None) or ""),
              str(getattr(ctx, "workspaceHostName", None) or ""),
            )
          if custom_profiles:
            databrickscfg_file.write(custom_profiles)
            env["DATABRICKS_BUNDLE_STRICT_PROFILE_MATCH"] = "1"
            cli_command = [cli_exec_path] + cli_args
          else:
            profile_name = "workspace_profile"
            databrickscfg_file.write(f"[workspace_profile]\nhost = {workspace_url}")
            cli_command = [cli_exec_path] + cli_args + ["-p", profile_name]
          databrickscfg_file.flush()
          env["DATABRICKS_CONFIG_FILE"] = databrickscfg_file.name

          tmp_file_path = tmp_file

          run_cli_start = time.monotonic()
          run_cli_phase = "run_cli_genie" if cli_upstream == "genie-code" else "run_cli"
          try:
            _cli_runner_run_cli_command_with_token(
              cli_command,
              cwd,
              tmp_file_path,
              include_stderr_in_error == "true",
              token,
              env,
              cli_upstream=cli_upstream,
              capture_stdout=capture_stdout == "true",
            )
          finally:
            emit_event(run_cli_phase, run_cli_start)


try:
  _databricks_cli_runner()
finally:
  del (
    _databricks_cli_runner,
    _cli_runner_get_venv_path_from_databricks_yml,
    _cli_runner_get_custom_profiles,
    _cli_runner_get_download_url,
    _cli_runner_run_cli_command,
    _cli_runner_run_cli_command_with_token,
  )
None
